<a href="https://colab.research.google.com/github/KeoniM/NFL_Data_Cleaning/blob/main/NFL_Plays_Week2_2023_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PURPOSE:**
- Accurately clean a week's worth of play data
  - Season 2023 -> Week 2

**NOTE:**
- What makes version 3 different than version 1 & 2 is the data being used. Although the core of the data is identical to the original, NFL.com has updated their formatting of how they display their data which has been scraped and used here. So minor adjustments will have to be made in creating the new version but I also see a beautiful opportunity to clean the older version here. Make the code more readible, organized and efficient.

**DATASET NOTES**
- The source data does not have every play recorded. This is very rare, but is worth noting. So far out of the 2752 plays available for Week 2, 2023, I have found 3 plays missing. (All extra point plays)

**STILL NEED TO WORK ON**
1. Fumble helper method
   - Fumbles resulting in touchdown
2. Laterals
   - Plays that have both a fumble and a lateral in it is going to be very tricky to clean.
3. Penalties

**MODIFICATIONS FOR LATER**
1. TRANSFORMING DATA
    - Defensive touchdowns
    - Plays that have the ball being handed to multiple players for a touchdown
      - Should I change the 'PlayOutcome' to this to have fractions? Depending
        on how many rows it took to cover the entire play?
        - (Touchdown (1/3), Touchdown (2/3), Touchdown (3/3))
          - Touchdown 3/3 being the row that actually had the ball in the
            endzone.

2. Add 2 separate yardage awarded features
    1. player yardage
       - How many yards the player gained on a play
    2. team net yardage
       - How many yards the team gained on a play

          - This might be helpful during fumbled plays.
            - If the defense recovers a fumble and takes it 80 yards behind LOS
              and they fumble and it is recovered by an offensive player and gains 50 yards (still 30 yards behind LOS) that player might be awarded a -30 yard gain but in reality they rushed for 50...

# MOUNTING AND IMPORTS

In [1]:
# Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Used to access personal google cloud services
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


In [3]:
# Imports

# Data manipulation
import pandas as pd

# Regular expressions
import re

# # Natural Language Toolkit (Used to find complete sentences)
# import nltk
# nltk.download('punkt')
# nltk.download('punkt_tab')
# from nltk.tokenize import sent_tokenize

import spacy
nlp = spacy.load("en_core_web_sm")

# Database access
from google.cloud import bigquery

# LOADING DATA (BigQuery)

In [4]:
# Client connect to bigquery project
client = bigquery.Client('nfl-data-430702')

## Season 2023 Week 2

In [5]:
# Grabbing all plays from 2023 Week 2 NFL Sesason
nfl_plays_week2_2023_query = """
                             SELECT *
                             FROM `nfl-data-430702.NFL_Plays_v3.NFL_Plays_Week2_2023`
                             """

# Running psuedo query, and returns the amount of bytes it will take to run query
dry_run_config = bigquery.QueryJobConfig(dry_run=True)
dry_run_query = client.query(nfl_plays_week2_2023_query, job_config=dry_run_config)
print("This query will process {} gigabytes.".format(dry_run_query.total_bytes_processed/10**9))

# Running query (Being mindful of the amount of data being grabbed)
# Will grab a maximum of a Gigabyte
safe_config = bigquery.QueryJobConfig(maximum_bytes_billed=10**9)
safe_config_query = client.query(nfl_plays_week2_2023_query, job_config=safe_config)

This query will process 0.000668109 gigabytes.


In [6]:
# Putting data attained from query into a dataframe
week2_2023_plays = safe_config_query.to_dataframe()

In [7]:
week2_2023_plays.head()

,Season,Week,GameSlot,Date,AwayTeam,HomeTeam,Quarter,DriveNumber,TeamWithPossession,IsScoringDrive,PlayNumberInDrive,IsScoringPlay,PlayOutcome,PlayStart,PlayTimeFormation,PlayDescription
0,2023,WEEK 2,Thursday Night Football,September 14th,MIN,PHI,1st Quarter,1,Philadelphia Eagles,1,1,0,Kickoff from MIN 35,None,Kickoff,— G.Joseph kicks 65 yards from MIN 35 to end z...
1,2023,WEEK 2,Thursday Night Football,September 14th,MIN,PHI,1st Quarter,1,Philadelphia Eagles,1,2,0,6 Yard Pass,1st & 10 at PHI 25,15:00 1st Shotgun,— J.Hurts pass short right to D.Smith to PHI 3...
2,2023,WEEK 2,Thursday Night Football,September 14th,MIN,PHI,1st Quarter,1,Philadelphia Eagles,1,3,0,-1 Yard Sack,2nd & 4 at PHI 31,14:27 1st Shotgun,— J.Hurts sacked at PHI 30 for -1 yards (sack ...
3,2023,WEEK 2,Thursday Night Football,September 14th,MIN,PHI,1st Quarter,1,Philadelphia Eagles,1,4,0,7 Yard Run,3rd & 5 at PHI 30,13:45 1st Shotgun,— J.Hurts scrambles right end pushed ob at PHI...
4,2023,WEEK 2,Thursday Night Football,September 14th,MIN,PHI,1st Quarter,1,Philadelphia Eagles,1,5,0,-1 Yard Pass,1st & 10 at PHI 37,13:10 1st Shotgun,— J.Hurts pass short left to D.Goedert to PHI ...


In [8]:
len(week2_2023_plays)

2752

# CATEGORIZE PLAYS
- The goal here is to parse out the different values for 'PlayOutcome'
  - Here is where I will separate different types of plays
    - ( pass / run / kickoff / etc..)

In [9]:
# All play outcomes from the game
# - From here we can categorize and clean plays accordingly
week2_2023_plays['PlayOutcome'].unique()

array(['Kickoff from MIN 35', '6 Yard Pass', '-1 Yard Sack', '7 Yard Run',
       '-1 Yard Pass', '54 Yard Pass', '1 Yard Pass', '1 Yard Run',
       '2 Yard Run', 'Field Goal', 'Kickoff from PHI 35', '15 Yard Pass',
       'Pass Incomplete', 'Punt', '-5 Yard Penalty', '3 Yard Run',
       'Fumble', '4 Yard Run', '12 Yard Run', '-7 Yard Sack',
       'Interception', '0 Yard Run', '5 Yard Pass', '-3 Yard Run',
       'Field Goal No Good', '5 Yard Penalty', '9 Yard Pass',
       '-2 Yard Run', '3 Yard Pass', '24 Yard Pass', '5 Yard Run',
       '7 Yard Pass', 'Touchdown', 'Extra Point', '6 Yard Run',
       '8 Yard Run', '11 Yard Pass', 'Timeout', '13 Yard Pass',
       '4 Yard Pass', '18 Yard Pass', '18 Yard Run', '0 Yard Pass',
       '-5 Yard Pass', '-10 Yard Penalty', '2 Yard Pass', '9 Yard Run',
       '11 Yard Run', '8 Yard Pass', '-2 Yard Sack', '-12 Yard Sack',
       '23 Yard Pass', '22 Yard Pass', '14 Yard Pass', '12 Yard Pass',
       '43 Yard Run', '10 Yard Pass', '16 Yard Pa

In [10]:
# NOTES:
# - Currently, I am eyeing all unique play outcomes to categorizing them.
#   - This type of approach is not flexable because a play outcome can
#     arise that has not been seen yet.
#     - There may be more play outcomes in the future when working on a full season,
#       let alone all seasons and future games.

# Play Types with complete cleaning methods (As far as this sample size goes)

# ~ OFFENSE ~
df_2023_pass_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Pass')]
df_2023_run_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Run')]
# ~ DEFENSE ~
df_2023_interception_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Interception')]
df_2023_sack_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Sack')]
# ~ SPECIAL TEAMS ~
df_2023_punt_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Punt')]
df_2023_kickoff_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Kickoff')]
# ~ SCORING ~
df_2023_touchdown_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Touchdown')]
df_2023_extrapoint_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Extra Point')]
df_2023_fieldgoal_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Field Goal')]
# df_2023_2pt_conversion_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('2PT Conversion')]
df_2023_2pt_conversion_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Conversion')]
# ~ OTHER ~
df_2023_fumble_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Fumble')]
df_2023_penalty_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Penalty')]
df_2023_turnover_on_downs_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Turnover on Downs')]
df_2023_timeout_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Timeout')]

## SANITY CHECK (All Plays Accounted for)
  - Once all plays have been categorized, will compare the sum of all plays in each category to the size of the original dataframe of plays.
    - Goal is to make sure the number of plays is the same.

In [11]:
# Categorized plays

plays_list = [df_2023_pass_week2,         # Offense
              df_2023_run_week2,
              df_2023_interception_week2, # Defense
              df_2023_sack_week2,
              df_2023_punt_week2,         # Special Teams
              df_2023_kickoff_week2,
              df_2023_touchdown_week2,    # Scoring
              df_2023_extrapoint_week2,
              df_2023_fieldgoal_week2,
              df_2023_2pt_conversion_week2,
              df_2023_fumble_week2,       # Other
              df_2023_penalty_week2,
              df_2023_turnover_on_downs_week2,
              df_2023_timeout_week2]

num_plays_categorized = 0

for plays in plays_list:
  num_plays_categorized = num_plays_categorized + len(plays)

num_plays_categorized == len(week2_2023_plays)

True

# PIPELINE
- ORDER
  1. Team Dictionary
    - Used to map team names with their acronyms
  2. Regular expressions
    - Used to find common patterns within raw data
  3. Transforming Data
    - So far, only label encoding
  4. Cleaning methods
    - Unique cleaning methods for each play type
  5. Main pipeline method
    - Control flow of cleaning methods

## 1. TEAM DICTIONARY

In [12]:
# KEY: Team name
# VALUE: Acronym of team

dict_teams = {
    'Cardinals': 'ARI', 'Falcons': 'ATL', 'Ravens': 'BAL', 'Bills': 'BUF', 'Panthers': 'CAR', 'Bears': 'CHI',
    'Bengals': 'CIN', 'Browns': 'CLE', 'Cowboys': 'DAL', 'Broncos': 'DEN', 'Lions': 'DET', 'Packers': 'GB',
    'Texans': 'HOU', 'Colts': 'IND', 'Jaguars': 'JAX', 'Chiefs': 'KC', 'Raiders': 'LV', 'Chargers': 'LAC',
    'Rams': 'LAR', 'Dolphins': 'MIA', 'Vikings': 'MIN', 'Patriots': 'NE', 'Saints': 'NO', 'Giants': 'NYG',
    'Jets': 'NYJ', 'Eagles': 'PHI', 'Steelers': 'PIT', '49ers': 'SF', 'Seahawks': 'SEA', 'Buccaneers': 'TB',
    'Titans': 'TEN', 'Commanders': 'WAS'
}

In [13]:
# KEY: Full Team name
# VALUE: Acronym of team

dict_teams_2 = {
    'Arizona Cardinals': 'ARI', 'Atlanta Falcons': 'ATL', 'Baltimore Ravens': 'BAL', 'Buffalo Bills': 'BUF', 'Carolina Panthers': 'CAR', 'Chicago Bears': 'CHI',
    'Cincinnati Bengals': 'CIN', 'Cleveland Browns': 'CLE', 'Dallas Cowboys': 'DAL', 'Denver Broncos': 'DEN', 'Detroit Lions': 'DET', 'Green Bay Packers': 'GB',
    'Houston Texans': 'HOU', 'Indianapolis Colts': 'IND', 'Jacksonville Jaguars': 'JAX', 'Kansas City Chiefs': 'KC', 'Las Vegas Raiders': 'LV', 'Los Angeles Chargers': 'LAC',
    'Los Angeles Rams': 'LAR', 'Miami Dolphins': 'MIA', 'Minnesota Vikings': 'MIN', 'New England Patriots': 'NE', 'New Orleans Saints': 'NO', 'New York Giants': 'NYG',
    'New York Jets': 'NYJ', 'Philadelphia Eagles': 'PHI', 'Pittsburgh Steelers': 'PIT', 'San Francisco 49ers': 'SF', 'Seattle Seahawks': 'SEA', 'Tampa Bay Buccaneers': 'TB',
    'Tennessee Titans': 'TEN', 'Washington Commanders': 'WAS'
}

In [14]:
# KEY: Acronym of team
# VALUE: Team name

dict_teams_3 = {
    'ARI': 'Arizona Cardinals', 'ATL': 'Atlanta Falcons', 'BAL': 'Baltimore Ravens', 'BUF': 'Buffalo Bills', 'CAR': 'Carolina Panthers', 'CHI': 'Chicago Bears',
    'CIN': 'Cincinnati Bengals', 'CLE': 'Cleveland Browns', 'DAL': 'Dallas Cowboys', 'DEN': 'Denver Broncos', 'DET': 'Detroit Lions', 'GB': 'Green Bay Packers',
    'HOU': 'Houston Texans', 'IND': 'Indianapolis Colts', 'JAX': 'Jacksonville Jaguars', 'KC': 'Kansas City Chiefs', 'LV': 'Las Vegas Raiders', 'LAC': 'Los Angeles Chargers',
    'LAR': 'Los Angeles Rams', 'MIA': 'Miami Dolphins', 'MIN': 'Minnesota Vikings', 'NE': 'New England Patriots', 'NO': 'New Orleans Saints', 'NYG': 'New York Giants',
    'NYJ': 'New York Jets', 'PHI': 'Philadelphia Eagles', 'PIT': 'Pittsburgh Steelers', 'SF': 'San Francisco 49ers', 'SEA': 'Seattle Seahawks', 'TB': 'Tampa Bay Buccaneers',
    'TEN': 'Tennessee Titans', 'WAS': 'Washington Commanders'
}

## 2. REGULAR EXPRESSIONS

In [15]:
####################################################
# REGULAR EXPRESSIONS USED TO LOCATE SPECIFIC DATA #
####################################################

###########
# GENERAL #
###########

# Players name (Grabs every variation come across so far)
# - I need this to be able to grab 'A.St. Brown' & 'C.Edwards-Helaire' & 'L.Van Ness'
name_pattern = r"(?:[A-Z][a-z]{0,4}\.)+(?:[- ]?[A-Z][a-z]+)+"

spotting_pattern = "(?:([A-Z]+) )?(-?[0-9]+)"

# Injuries (Returns the player(s) who go injuried during play)
injury_pattern = f"[A-Z]+-({name_pattern}) was injured during the play"

# Touchdowns
touchdown_pattern = f"for ([0-9]+) yards?, TOUCHDOWN"

################
# PLAY DETAILS #
################

# Positioning at the end of the play
standard_play_end_pattern = "(?:to|at) (?:([A-Z]+) )?([0-9]+) for (no gain|-?[0-9]+)(?: yards?)?"

###########
# OFFENSE #
###########

# Passer (Player passing, Player spiking, Player who got sacked)
passer_name_pattern = f"({name_pattern}) (?:pass|spiked|sacked)"

# Pass play (Returns intended receiver and the direction of the pass)
receiver_pattern = f"(short|deep) (left|right|middle) (?:to|intended for) ({name_pattern})"

# Rushing play (Player running ball)
rusher_pattern = f"({name_pattern})(?: scrambles)? (?:(left|right|up|kneels)) (?:(the middle|guard|tackle|end))?"

# 2 Point Conversion (Pass attempt)
tp_conversion_pass_pattern = f"({name_pattern}) pass to ({name_pattern})"

# 2 Point Conversion (Rush attempt)
tp_conversion_rush_pattern = f"({name_pattern}) rushes (left|right|up) (the middle|guard|tackle|end)"

###########
# FUMBLES #
###########

# Normal fumble
# FUMBLES (S.Ebukam) [S.Ebukam], RECOVERED by IND-K.Paye at HOU 15.
# fumble_pattern = rf"FUMBLES(?: \(({name_pattern})\))?(?: \[({name_pattern})\])?, (?:RECOVERED|recovered) by ([A-Z]+)-({name_pattern}) at ({spotting_pattern})"
fumble_pattern = rf"FUMBLES(?: \(({name_pattern})\))?(?: \[({name_pattern})\])?,(?: touched at ({spotting_pattern}),)? (?:RECOVERED|recovered) by ([A-Z]+)-({name_pattern}) at ({spotting_pattern})"

# Fumble out of bounds
# fumble_out_of_bounds_pattern = rf"FUMBLES(?: \(({name_pattern})\))?(?: \[({name_pattern})\])?, ball out of bounds at ({spotting_pattern})"
fumble_out_of_bounds_pattern = rf"FUMBLES(?: \(({name_pattern})\))?(?: \[({name_pattern})\])?,(?: touched at ({spotting_pattern}),)? ball out of bounds at ({spotting_pattern})"

# Own fumble recovery
own_fumble_recovery_pattern = rf"FUMBLES(?: \(({name_pattern})\)), and recovers at ({spotting_pattern})"

# Aborted fumbles

# Passer Aborted fumbles and recovers
aborted_and_recovers_pattern = rf"({name_pattern}) FUMBLES \(Aborted\) at ({spotting_pattern}), and recovers at ({spotting_pattern})"

# Aborted fumble and recovered by another player
# C.Stroud FUMBLES (Aborted) at HOU 18, recovered by HOU-N.Dell at HOU 16
# aborted_and_recovered_by_pattern = rf"({name_pattern}) FUMBLES \(Aborted\) at ({spotting_pattern}), recovered by ([A-Z]+)-({name_pattern}) at ({spotting_pattern})"
aborted_and_recovered_by_pattern = rf"({name_pattern}) FUMBLES \(Aborted\) at ({spotting_pattern}),(?: touched at ({spotting_pattern}),)? (?:RECOVERED|recovered) by ([A-Z]+)-({name_pattern}) at ({spotting_pattern})"

# Center Aborted fumble
center_aborted_pattern_1 = f"({name_pattern}) Aborted"
center_aborted_pattern_2 = f"({name_pattern}) FUMBLES at ({spotting_pattern}), touched at ({spotting_pattern}), (?:RECOVERED|recovered) by ([A-Z]+)-({name_pattern}) at ({spotting_pattern})"

# muffed catch during punt or kickoff return
muffed_catch_pattern = f"({name_pattern}) MUFFS catch, (?:RECOVERED|recovered) by ([A-Z]+)-({name_pattern}) at ({spotting_pattern})"

# Forced fumble player explicitly stated
forced_fumble_pattern = f"Fumble Forced by ([A-Z]+-?[0-9]*)-({name_pattern})"

###########
# DEFENSE #
###########

# Tackles

# solo / sack
solo_tackle_pattern = rf"\(({name_pattern})\)"

# shared
shared_tackle_pattern = rf"\(({name_pattern}), ({name_pattern})\)"

# shared
assisted_tackle_pattern = rf"\(({name_pattern}); ({name_pattern})\)"

# Pressure (Who applied pressure to passer)
# - I think it might be possible for multiple defenders to apply pressure to the passer.
defense_pressure_name_pattern = rf"\[({name_pattern})\]"

# Split sack (Players who equally received credit for sack)
split_sack_pattern = f"sack split by ({name_pattern}) and ({name_pattern})"

# Defense takeaway (takeaway for yardage)
# D.Hill pushed ob at 50 for 20 yards (J.Wills)
# J.Bates to ATL 49 for no gain (T.Marshall)
defensive_takeaway_run_pattern = f"({name_pattern}) (?:pushed ob at|ran ob at|to)(?: ([A-Z]+))? (-?[0-9]+) for (no gain|-?[0-9]+)(?: yards?)?" # yardage after fumble recovery & yardage after interception

defensive_takeaway_for_touchdown = f"({name_pattern}) for ([0-9]+) yards?, TOUCHDOWN"

# Interception (Player who intercepted pass)
# interception_name_pattern = rf"INTERCEPTED by ({name_pattern})(?:[ \t]*(?:\({name_pattern}\)|\[{name_pattern}\]))* at ((?:[A-Z]+ )?[0-9]+)"
interception_name_pattern = rf"INTERCEPTED by ({name_pattern})(?:[ \t]*(?:\({name_pattern}\)|\[{name_pattern}\]))* at ({spotting_pattern})"


#################
# SPECIAL TEAMS #
#################

# Punting play (Who was the punter, How many yards the ball went, Who was the Longsnapper)
# punting_pattern = f"({name_pattern}) punts (-?[0-9]+) yards? to(?: ((?:[A-Z]+ )?-?[0-9]+)| end zone), Center-({name_pattern})"
punting_pattern = f"({name_pattern}) punts (-?[0-9]+) yards? to(?: ({spotting_pattern})| end zone), Center-({name_pattern})"

# Punt return resulting in fair catch
punt_fair_catch_pattern = f", fair catch by ({name_pattern})"

# Punt or kickoff downed by
# downed by PHI-S.Brown
kick_downed_by_pattern = f"downed by [A-Z]+-({name_pattern})"

# Kickoff play (Who was the kicker, How many yards the ball was kicked )
kickoff_pattern = f"({name_pattern}) kicks(?: onside)? (-?[0-9]+) yards from ((?:[A-Z]+ )?[0-9]+) to ((?:[A-Z]+ )?-?[0-9]+|end zone)"

# Field goal (Good OR No Good)
field_goal_pattern = f"({name_pattern}) (-?[0-9]+) yard field goal is (?:GOOD|No Good),(?: ([A-Za-z]+(?: [A-Za-z]+)*),)? Center-({name_pattern}), Holder-({name_pattern})."

# Field goal (Blocked)
# — C.McLaughlin 40 yard field goal is BLOCKED (R.Green), Center-Z.Triner, Holder-J.Camarda, recovered by TB-J.Camarda at 50.
field_goal_blocked_pattern = rf"({name_pattern}) (-?[0-9]+) yard field goal is BLOCKED \(({name_pattern})\), Center-({name_pattern}), Holder-({name_pattern}), (?:RECOVERED|recovered) by ([A-Z]+)-({name_pattern}) at ((?:[A-Z]+ )?[0-9]+)"

# Extra Point (Good OR No Good)
extra_point_pattern = f"({name_pattern}) extra point is (?:GOOD|No Good),(?: ([A-Za-z]+(?: [A-Za-z]+)*),)? Center-({name_pattern}), Holder-({name_pattern})."

## 3. TRANSFORMING DATA

In [16]:
# PURPOSE:
# - Take value for 'PlayTimeFormation' and split into 3 separate features.
#   1. GameClock (Will come about when renaming 'PlayTimeFormation')
#   2. PlayInQuarter
#      - A 'Quarter' feature already exists. But that pertains more towards which drive belongs to which quarter.
#        - This feature will state which quarter the play took place in.
#   3. Formation

def playtimeformation_split(df_plays):

  df_plays_copy = df_plays.copy()

  # new_columns = ['Formation']
  new_columns = ['Formation', 'PlayInQuarter']

  df_plays_copy = df_plays_copy.rename(columns = {'PlayTimeFormation': 'GameClock'})

  df_plays = df_plays.reindex(columns=df_plays.columns.tolist() + new_columns)

  # Splitting original feauture 'PlayTimeFormation' (Now known as 'TimeLeftInQuarter')
  for idx, play in df_plays_copy['GameClock'].items():
    value_elements = play.split(' ')
    # Some plays (e.g. Kickoff) will only have the formation as a value
    if len(value_elements) <= 1:
      df_plays_copy.at[idx, 'Formation'] = value_elements[0]
      df_plays_copy.at[idx, 'GameClock'] = ""
    else:
      df_plays_copy.at[idx, 'GameClock'] = value_elements[0]
      df_plays_copy.at[idx, 'PlayInQuarter'] = value_elements[1]
      df_plays_copy.at[idx, 'Formation'] = " ".join(value_elements[2::])

  # # Transform values in 'Quarter' feature from string to integer (e.g. '1st Quarter' -> 1)
  # dict_replace_quarter = {'1st Quarter': 1, '2nd Quarter': 2, '3rd Quarter': 3, '4th Quarter': 4,
  #                         '1st': 1, '2nd': 2, '3rd': 3, '4th': 4}

  # All overtime quarters will be have the value 5 in their place
  # df_plays_copy['Quarter'] = df_plays_copy['Quarter'].map(dict_replace_quarter).fillna(5).astype(int)

  return df_plays_copy

# PURPOSE:
# - Take value for 'PlayStart' and split into 2 separate features.
#   1. DownAndDistance (Will come about when renaming 'PlayStart')
#   2. FieldPosition (Start of play)

def playstart_split(df_plays):

  df_plays_copy = df_plays.copy()

  new_columns = ['FieldPosition']

  df_plays_copy = df_plays_copy.rename(columns = {'PlayStart': 'DownAndDistance'})

  df_plays_copy = df_plays_copy.reindex(columns=df_plays_copy.columns.tolist() + new_columns)

  df_plays_copy['FieldPosition'] = df_plays_copy['FieldPosition'].astype(str)

  # Splitting original feature 'PlayStart' (Now known as 'DownAndDistance')
  for idx, play in df_plays_copy['DownAndDistance'].items():
    # Some plays to not have a down and distance or field position and contain 'nan' values here,
    # this is to catch those plays and keep going. (e.g. Kickoff / Extra Point / etc..)
    if pd.isna(play):
      continue
    else:
      value_elements = play.split(' at ')
      df_plays_copy.at[idx, 'DownAndDistance'] = value_elements[0]
      df_plays_copy.at[idx, 'FieldPosition'] = value_elements[1]

  return df_plays_copy

# PURPOSE:
# - Keep consistence with team names
#   - A team name will always be represented by their acronym

def consistent_team_names(df_plays):

  df_plays_copy = df_plays.copy()

  # df_plays_copy['AwayTeam'] = df_plays_copy['AwayTeam'].map(dict_teams)
  # df_plays_copy['HomeTeam'] = df_plays_copy['HomeTeam'].map(dict_teams)
  df_plays_copy['TeamWithPossession'] = df_plays_copy['TeamWithPossession'].map(dict_teams_2)

  return df_plays_copy

## 4. CLEANING METHODS

### HELPER CLEANING METHODS

#### SPLIT PLAY DESCRIPTION INTO SENTENCES

In [17]:
# PURPOSE:
# - Function will split the feature "PlayDescription" into
# its individual sentences and place them in a list.

# - I am using playdescription.split(". ") to separate
#   sentences within play description. The problem here
#   is that sometimes a player will have ". " within their
#   name, causing a sentence to split into 2 with the
#   divide being in the middle of the players name. To
#   overcome this, I will replace the ". " character
#   combination within player names with the string
#   "<DOT>" then split play description into separate
#   sentences and revert the player names back to normal
#   after the split.

def split_play_description(play_description):

  # Finding all player names that were mentioned in the play
  player_names = re.findall(name_pattern, play_description)

  # Creating map for player names that have ". " within their name
  # and mapping them to a safe replacement name for the time being.
  replacements = {}
  for name in player_names:
    if ". " in name:
      protected_name = name.replace(". ", "<DOT>")
      replacements[name] = protected_name

  # Replacing original player name with safe replacement name in
  # play description
  for original, protected in replacements.items():
    play_description = play_description.replace(original, protected)

  # Splitting play description by ". "
  play_split = play_description.split(". ")

  # Revert player names back to normal in play_split
  restored_names = [s.replace("<DOT>", ". ") for s in play_split]

  return restored_names

#### YARDAGE BETWEEN SPOTTINGS

In [18]:
# UPDATED YARDAGE BETWEEN SPOTTINGS METHOD.

# Within this dataset, a team scores by reaching the opposing team's endzone.
# - This does not alternate, they will never have to strive for their own
#   endzone to score.
#   - Their zone will always be 100-51 yards away from their target endzone
#   - Their opposing zone will always be 49-0 yards away from their target
#     endzone.

# GOAL:
# - For this method, I would like to calculate the distance between two field
#   positions (or spottings).

# PLAN:
# - INPUTS NEEDED:
#   1. Team with possession
#      - Need to be cautious when the ball is overturned
#        - (loss fumble, interception, punt return, kickoff return, etc..)
#          - Need to make sure that plays such as this have the feature
#            'TeamWithPossession' reflecting the team with the ball during that
#            time. During a single play, it is possible for both teams to
#            alternate possession of the ball.
#   2. start_spotting
#   3. end_spotting

# ALGORITHM:
# - IDEA:
#   - Both spottings (start and end) will be converted to numbers on a scale
#     from 0-100 depending on how far away the spotting is from the target
#     endzone. From there, I will subtract 'start_spotting' by 'end_spotting'
#     to receive yardage gained.
# 1. Compare team with possession to zone of spotting and convert spottings
#    - EXAMPLES
#      - TeamWithPossession = 'BUF'
#      - start_spotting = 'BUF 20'
#        - 'BUF 20' ->
#          - start_zone = 'BUF'
#          - start_yardage = 20
#        - TeamWithPossession == start_zone: (TRUE)
#          - converted_start_spotting = 100 - start_yardage (20)
#            -> converted_start_spotting = 80 (yards from target endzone)
#      - end_spotting = 'KC 35'
#        - 'KC 35' ->
#          - end_zone = 'KC'
#          - end_yardage = 35
#        - TeamWithPossession == start_zone: (FALSE)
#          - converted_end_spotting = end_yardage (35)
#            -> converted_start_spotting = 35 (yards from target endzone)

def yardage_between_spottings(team_with_possession, start_spotting, description_with_end_spotting):

  ##################
  # START SPOTTING #
  ##################

  # Would this grab a spotting at the 50?

  # Breaking down 'start_spotting' ('BUF 20' -> [('BUF', 20)])
  start_elements = re.findall(spotting_pattern, start_spotting)
  start_territory = start_elements[0][0]
  start_yardage = int(start_elements[0][1])

  # converting start_yardage to 0-100 scale
  if start_territory == team_with_possession:
    start_yardage = 100 - start_yardage

  ##########################################
  # END SPOTTING AND PSUEDO YARDAGE GAINED #
  ##########################################

  # touchdown
  touchdown = re.findall(touchdown_pattern, description_with_end_spotting)
  if touchdown:
    # print(start_yardage)
    return start_yardage

  # Breaking down 'end_spotting'
  end_spotting = re.findall(standard_play_end_pattern, description_with_end_spotting)
  if end_spotting:
    end_territory = end_spotting[0][0]
    end_yardage = int(end_spotting[0][1])

    # converting end_yardage to 0-100 scale
    if end_territory == team_with_possession:
      end_yardage = 100 - end_yardage

  # PLAY FAILED (e.g. pass incomplete)
  else:
    # print(0)
    return 0

  # print(start_yardage - end_yardage)
  return start_yardage - end_yardage

#### FUMBLES

In [19]:
# Need to go over all plays again.
# 1. double check correctness
# 2. only keep unique organizing/cleaning flows

# Design summary/aim:
# - GOAL: Represent a cleaned fumbled play within a dataframe that can result
#   in a return dataframe of 1 to many rows of data.
#   - Each row of the dataframe will represent a single action, also known as a
#     possession, that took place within this play.
#     - Each action within this play could require a range of data to accurately
#       represent that single action. All data that could potentially be used to
#       represent a single action will be collected and put into a list, this
#       list will be inside of another list, ultimately creating a collection
#       of all actions that took place within the play (2D list).
#       - Once the collection of actions has been obtained, each action within
#         the list will be cleaned, representing a single row within the return
#         dataframe which will replace the uncleaned original row within the
#         original dataframe.
#
#   ACTION LIST FORMAT:
#   [0  - fumble_spotting_from_previous_action,
#    1  - fumble_recovery_from_previous_action, (Or spotting of interception)
#    2  - team_on_offense,
#    3  - team_on_defense,
#    4  - current_team_with_possession,
#    5  - line_of_scrimmage,
#    6  - action_description,
#    7  - fumble_spotting_from_current_action, (Or spotting of end of possession?)
#    8  - fumble_description,
#    9  - player_who_forced_fumble,
#    10 - team_that_recovered_fumble_from_current_action,
#    11 - player_who_recovered_fumble_from_current_action,
#    12 - fumble_recovery_from_current_action,
#    13 - fumble out of bounds spotting]
#
#   DESGIN GOALS FOR FUTURE ADJUSTMENTS:
#   1. I WANT TO MAKE A METHOD NOW THAT IS EASILY ADJUSTABLE IN THE FUTURE.
#      - Odds are, I do not have every situation accounted for. I would like
#        to create a method where I can adjust for different situations easily.
#        I WOULD NEED TO:
#        1. Create a space where I can add more elements to take into account
#           (when organizing data from sentences)
#        2. A space where I can clean added adjustments
#           (when cleaning organized data)
#
#   DESIGN NOTES:
#   - I will need to go through all plays again and write through each unique
#     situation...
#   - There are going to be 2 stages for CLEANING each possession
#     1. Clean action description through play type
#        UNIVERSAL DATA NEEDED FROM DATA GATHERING:
#        - Spotting of the possession ending.
#          - Could be fumble spotting
#          - Could be player downed to complete play
#     2. Clean fumble actions that follow
#        DATA NEEDED FROM DATA GATHERING:
#        - SCENARIO: fumble recovery
#          - Player who forced the fumble (?) Might not be necessary
#          - team that recovered the fumble
#          - player that recovered the fumble
#          - fumble recovery spotting
#        - SCENARIO: ball out of bounds
#          - spotting of ball out of bounds
#        - SCENARIO: Aborted fumbles
#          - Need to grab player name who fumbled (passer/rusher/etc..)
#        - SCENARIO: 'and recovers'
#          - This is an indication that the next element within the list of
#            possessions will need to merge with the current element.
#
#   RULES FOR YARDAGE NOTES:
#   - If there is a fumble behind the LOS and recovered and thrown for
#     an incomplete pass, I think the intial player who fumbled receives
#     a 0 yard gain?
#   - If an action taken by the offense resulted behind LOS (negative yards)
#     and a following action by the offense ended beyond LOS (resulting in
#     positive yards), all actions up to that point that resulted in negative
#     yards now are awarded 0 yards.
#     - I am going to believe that it will be all consecutive actions by the
#       offense that are behind the line of scrimmage. This means that if at
#       some point the defense had possession and the offense recovered
#       possession, it would only be from when the offense recovered possession.
#       None of the offensive players that had possession before the defense
#       took over would be affected.
#   - Yardage situations
#     - ball out of bounds
#       - start_spotting (spotting of recovery, reception OR LOS)
#       - end_spotting (spotting of fumble)
#     - incomplete pass
#       - All consecutive previous offensive possessions that resulted in (-) yards
#         in play are awarded in 0 yards.
#     - complete pass
#       - start_spotting (LOS)
#       - end_spotting
#         1. fumble spotting
#            a. if def team recovers fumble
#            b. if off team recovers BEYOND fumble spotting
#         2. fumble recovery spotting
#            a. if off team recovers BEHIND fumble spotting
#         3. downed
#            a. if same receiver recovers own fumble
#               - 'and recovers' will be within fumble description.
#                 - This will indicate that an action description will
#                   have multiple lists within the list of actions.
#
#
#      - OWN RECOVERY THOUGHTS: (FIGURE OUT WHERER TO PUT THIS)
#        - I need to address the problem where a single player fumbles and recovers
#          multiple times.
#          - I know that I am going to merge each separate possession of the
#            same player, I just need to figure out how I would like to do it.
#            - Some features will have lists.
#              - action description ([1st possession, 2nd possession, etc..])
#              ~ fumble description(?) <- Would have to add another element
#                - might be a good idea so no loss of data.
#              ~ forced fumble player(s)
#              ~ player that recovered fumble
#            - fumble spotting
#              - Note to self: Only the last fumble spotting matters for yardage
#            - fumble recovery spotting
#              - Note to self: Only the last fumble recovery spotting matters for yardage
#            - I could put these all together within the second step that I have now?
#              TRIGGER:
#              - If an 'and recovers' appears in fumble description
#
#
#   WRITTEN OUT PSEUDO CODE:
#
#   ######################
#   # ORGANIZATION PHASE #
#   ######################
#   - Spliting play into possessions. Each possession will have their own list of
#     data to obtain an accurate account of that possession. Each possession within
#     the play will be collected in cronological order.
#
#   1. Split play description by sentences.
#      - Create a list, each element will be a sentence from the play
#        description.
#
#   2. Organize play into separate action descriptions
#
#      - Create variables that will fill through each loop
#        - previous_fumble_spotting = ''
#        - previous_fumble_recovery = ''
#        - team_on_offense = too
#        - team_on_defense = tod
#        - current_team_with_possession = TeamWithPossession
#        - line_of_scrimmage = LOS
#
#      - Create lists
#        1. Organized list of all actions within play
#           - Each action list (possession) will be put in here cronologically
#         2. action within play
#            - Each action before being placed into the organized list of all actions
#              will first go through this container. Once an action (possession) has
#              been finished, it will then move from this list to the list of all
#              actions and this list will reset for the next possession.
#
#      - Append variables for initial play
#        - [pfs, pfrs, too, tod, ctwp, 'LOS', '', '', '', '', '', '', '', '']
#
#      - Loop through each sentence within play
#        - Categorize each sentence into 1 of 5 categories
#
#        - NOTE: Maybe I should create a separate list on the side that has all
#                of these? So when we are creating the return df, we can put
#                them back in? These are probably important.
#        1. Additional data (reviews/rulings/etc..)
#           - IF (contains 'reported in as eligible')
#             - continue
#           - IF (contains 'The Replay Official reviewed')
#             - continue
#           - IF (contains 'The ruling on the field')
#             - continue
#           - IF (contains 'Penalty on')
#             - continue
#           - IF (contains 'was injured')
#             - continue
#           - IF (forced_fumble_pattern)
#             - IF (organized list of all actions within play[last_index][9] == '')
#               - organized list of all actions within play[last_index][9] = forced_fumble_pattern[0][1]
#             - ELSE:
#               - IF (organized list of all actions within play[last_index][9] is list)
#                 - organized list of all actions within play[last_index][9].append(forced_fumble_pattern[0][1])
#               - ELSE:
#                 - organized list of all actions within play[last_index][9] = [organized list of all actions within play[last_index][9], forced_fumble_pattern[0][1]]
#             - continue
#
#        2. Aborted fumble (Will probably have to be adjusted once sample size gets bigger)
#           - IF sentence contains 'Aborted'
#
#             (player aborts and recovers)
#             - IF sentence fits r.e. ([player] FUMBLES (Aborted) at [spotting], and recovers at [spotting])
#               - action description
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', '', '', '', '', '', '', '']
#               - fumble spotting
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', '', '', '', '', '', '']
#               - fumble description
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', '', '', '', '']
#               - fumble recovery spotting
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', '', '', 'frs', '']
#               - IF (this index == last index of play sentences list)
#                 - append [action within play] to [organized list of all actions within play]
#               - continue
#
#.              (player aborts and another player recovers)
#             - IF sentence fits r.e. ([player] FUMBLES (Aborted) at [spotting], recovered by [player] at [spotting])
#               - action description
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', '', '', '', '', '', '', '']
#               - fumble spotting
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', '', '', '', '', '', '']
#               - fumble description
#.                - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', '', '', '', '']
#               - fumble recovery team
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', '', '', '']
#               - player that recovered fumble
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', 'frp', '', '']
#               - fumble recovery spotting
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', 'frp', 'frs', '']
#
#               - append [action within play] to [organized list of all actions within play]
#               - IF (this index != last index of play sentences list)
#                 - action_within_play = ['fs', 'frs', 'too', 'tod', 'tfr', 'LOS', '', '', '', '', '', '', '', '']
#               - continue
#
#               (center aborted fumble (part 1))
#             - IF sentence fits r.e. (center_aborted_pattern_1)
#               - action description
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', '', '', '', '', '', '', '']
#               - continue
#
#        3. muffed catch
#           - IF sentence fits r.e. (muffed_catch_pattern)
#             - action description
#               - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', '', '', '', '', '', '', '']
#             - fumble spotting (Will be spotting of catch (LOS in this case))
#               - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', '', '', '', '', '', '']
#             - fumble description (same as action description)
#               - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', '', '', '', '']
#             - fumble recovery team
#               - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', '', '', '']
#             - player that recovered fumble
#               - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', 'frp', '', '']
#             - fumble recovery spotting
#               - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', 'frp', 'frs', '']
#             - IF (this index != last index of play sentences list)
#               - action_within_play = ['fs', 'frs', 'too', 'tod', 'tfr', 'LOS', '', '', '', '', '', '', '', '']
#             - append [action within play] to [organized list of all actions within play]
#             - continue
#
#        4. fumble description
#           - IF sentence contains 'FUMBLES'
#.            - IF ( fd element is NOT empty (index: 8) ): (For own recovery)
#               - append fd to previous fd ([fd1, fd2, etc..])
#                 - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad1', 'fs/eop', [fd1, fd2, etc..], '', '', '', '', '']
#             - ELSE:
#               - add fd
#                 - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', '', '', '', '']
#
#             - NOTE: There are many different types of fumbles. Each will have different sets of values
#                   to grab.
#
#.              (fumble out of bounds)
#             - IF sentence fits r.e. (fumble_out_of_bounds_pattern)
#               - IF ( ffp is NOT empty (index: 9) ): (For own recovery)
#                 - append ffp to previous ffp ([ffp1, ffp2, etc..])
#               - ELSE:
#                 - add player who forced the fumble (If there is a player who forced fumble)
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', '', '', '', '']
#               - add out of bounds spotting
#                 - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', '', '', '', 'oobs']
#
#               ('and recovers' own fumble recovery)
#             - IF sentence fits r.e. (own_fumble_recovery_pattern)
#               - IF ( fd is list ): (For own recovery after own recovery)
#                 - append ffp to previous ffp ([ffp1, ffp2, etc..])
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', [ffp1, ffp2, etc..], '', '', '', '']
#                 - append tfr to previous tfr ([tfr1, tfr2, etc..]) (same as team who fumbled)
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', [tfr1, tfr2, etc..], '', '', '']
#                 - append frp to previous frp ([frp1, frp2, etc..]) (same as who fumbled) <- May take this out...
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', [frp1, frp2, etc..], '', '']
#                 - append frs to previous frs ([frs1, frs2, etc..])
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', 'frp', [frs1, frs2, etc..], '']
#               - ELSE: (first own recovery)
#                 - add player who forced the fumble (If there is a player who forced fumble)
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', '', '', '', '']
#                 - add team that recovered the fumble (same as team who fumbled)
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', '', '', '']
#                 - add player that recovered the fumble (Need to figure out r.e. to accomplish this) <- May take this out...
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', 'frp', '', '']
#                 - add fumble recovery spotting
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', 'frp', 'frs', '']
#               - CONTINUE (Go straight to the next sentence)
#                 - crucial because when a player recovers their own fumble,
#                   this action description and the next action description
#                   will merge. This DOES NOT start a new row, but most other
#                   scenarios will.
#
#               (center aborted fumble (part 2))
#             - IF sentence fits r.e. (center_aborted_pattern_2)
#               - NOTE: May have to include where ball was touched in the future.
#               - own recovery check
#               - Check r.e. (center_aborted_pattern_1) on action description
#                 - IF (player aborted == fumble recovery player):
#                   - player that recovered fumble <- May take this out...
#                     - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', 'frp', '', '']
#                     - continue <- important for own recovery
#               - fumble spotting
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', '', '', '', '', '', '']
#               - fumble recovery team
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', '', '', '']
#               - player that recovered fumble
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', 'frp', '', '']
#               - fumble recovery spotting
#                 - ['', '', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', '', 'tfr', 'frp', 'frs', '']
#
#               (DEFAULT)
#             - IF sentence fits r.e. (center_aborted_pattern_2)
#               - IF ( fd is list ): (For own recovery)
#                 - append ffp to previous ffp ([ffp1, ffp2, etc..])
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', [ffp1, ffp2, etc..], '', '', '', '']
#                 - append tfr to previous tfr ([tfr1, tfr2, etc..])
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', [tfr1, tfr2, etc..], '', '', '']
#                 - append frp to previous frp ([frp1, frp2, etc..])
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', [frp1, frp2, etc..], '', '']
#                 - append frs to previous frs ([frs1, frs2, etc..])
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', 'frp', [frs1, frs2, etc..], '']
#               - ELSE:
#                 - add player who forced the fumble (If there is a player who forced fumble)
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', '', '', '', '']
#                 - add team that recovered the fumble
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', '', '', '']
#                 - add player that recovered the fumble
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', 'frp', '', '']
#                 - add current fumble recovery spotting
#                   - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop', 'fd', 'ffp', 'tfr', 'frp', 'frs', '']
#
#             - NOTE: Once a fumble description has been found, this is the sign of a complete possession resulting
#                     in a fumble. Only exceptions are possessions that are downed or own fumble recoveries.
#             - ONCE FUMBLE DESCRIPTION FOUND
#               1. append [action within play] to [organized list of all actions within play]
#               2. action_within_play = ['fs', 'frs', 'too', 'tod', 'tfr', 'LOS', '', '', '', '', '', '', '', '']
#                  - This completes the action description and starts a new
#
#        5. action description (If sentence does not have 'fumble' in it, it is an action description)
#           - IF ( ad element is NOT empty (index: 6) ): (For own recovery)
#             - append ad to previous ad ([ad1, ad2, etc..])
#               - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', ['ad1', 'ad2', ...], 'fs/eop', 'fd', '', '', '', '', '']
#             - append fs/eop to previous fs/eop ([fs/eop1, fs/eop2, etc..])
#                - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', ['fs/eop1', 'fs/eop2', etc..], '', '', '', '', '', '']
#           - ELSE:
#             - add ad
#               - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', '', '', '', '', '', '', '']
#           - add fs/eops/is (fumble spotting/end of play spotting/interception spotting)
#             - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', 'fs/eop1', '', '', '', '', '', '']
#
#           - IF sentence fits r.e. (interception_name_pattern)
#             - NOTE: If the action description is a pass that has been intercepted, the
#                     following action must have the fumble.
#             - remove fs/eops
#               - ['pfs', 'pfrs', 'too', 'tod', 'ctwp', 'LOS', 'ad', '', '', '', '', '', '', '']
#             - append [action within play] to [organized list of all actions within play]
#             - NOTE: interception spotting will be in place for FRS.
#             - action_within_play = ['', 'is', 'too', 'tod', 'tod', 'LOS', '', '', '', '', '', '', '', '']
#
#           - Final sentence has been organized
#           - IF (this index == last index of play sentences list)
#             - append [action within play] to [organized list of all actions within play]
#
#
#   ######################
#   #   CLEANING PHASE   #
#   ######################
#   - Go through each possession and accurately reflect the possession as a single row
#     within the return dataframe.
#
# - I have decided that I will condense the code once I have written all of it out
#   the long way. Meaning that for every unique situation, I will clean it without
#   looking for common ground with other plays.
#
#   3. Clean action descriptions
#
#      - create a return dataframe
#
#      - Loop through each element in [organized list of all actions within play]
#
#        - create a single row dataframe (a copy of the original)
#
#        - clean action description
#
#          - IF (action description is list):
#            - IF ad_list[0] fits r.e. (center_aborted_pattern_1) <- I will probably run into an issue where center_aborted_pattern_1 is the only
#              - action_description = ad_list[1]                     element in action description. Those plays may not have a play description.
#            - ELSE:
#              - action_description = ad_list[0]
#            - IF ad_list[0] fits r.e. (aborted_and_recovers)
#              - action_description = ad_list[1]
#            - ELSE:
#              - action_description = ad_list[0]
#          - ELSE:
#            - action_description = (index of action description in possession list)
#
#            (passing action description)
#          - IF ad fits r.e. (passer_name_pattern)
#            - add ad as single row dataframe 'playdescription'
#              (interception)
#            - IF ad fits r.e. (interception_name_pattern)
#              -  use 'clean_intercepted_plays' to clean single row dataframe <- Need to make sure 'clean_intercepted_plays' can do this.
#            - ELSE
#              - use 'clean_pass_plays' to clean single row dataframe
#
#            (run action description)
#          - IF ad fits r.e. (rusher_pattern) <- Will probably have to add another rushing r.e. for possessions that follow.
#            - add ad as single row dataframe 'playdescription'
#            - use 'clean_run_plays' to clean single row dataframe
#
#            (sacked action description) <- Do i not have a r.e. for sacks??
#          - IF ad fits r.e. (sack_pattern)
#            - add ad as single row dataframe 'playdescription'
#            - use 'clean_sacked_plays' to clean single row dataframe
#
#            (aborted and recovered by)
#          - IF ad fits r.e. (aborted_and_recovered_by_pattern)
#            - Adjust features to show that this is a run play
#              1. play type
#              2. who aborted
#              3. thats all I can think of for now
#
#            (aborted_and_recovers_pattern)
#          - IF ad fits r.e. (aborted_and_recovers_pattern)
#            - Adjust features to show that this is a run play
#              1. play type
#              2. who aborted
#              3. thats all I can think of for now
#
#            (muffed_catch_pattern)
#          - IF ad fits r.e. (muffed_catch_pattern)
#            - Adjust features to show that this is a return play
#              1. add player who returned ball
#              2. whatever else...
#
#          - IF (action description is list):
#            - add list to 'PlayDescription' of single row dataframe
#            - loop through all possession and record all players that received tackles,
#              put them in necessary features within single row dataframe.
#
#        - clean fumble description
#
#          - NOTE: as far as I can tell, the only fumble description that matters
#                  is the one the possession ended on.
#          - IF (fumble description is list):
#            - fumble_description = fd_list[last]
#          - ELSE:
#            - fumble_description = (index of fumble description in possession list)
#
#            (own recovery)
#          - IF fd fits r.e. (own_fumble_recovery_pattern)
#            - start_spotting
#            - IF (offense):
#              - IF (previous fumble recovery spotting):
#                - IF (PFRS < LOS):
#                  - start = LOS
#                - ELSE: (PRFS > LOS)
#                  - start = PFRS
#              - ELSE: (PFRS == '')
#                - start = LOS
#            - ELSE: (defense)
#              - start = PFRS
#            - end spotting
#            - IF (FS/EOPS is list):
#              - (FRS is list):
#                - IF (len(FS/EOPS > len(FRS)):
#                  - end = FS/EOPS[last index]
#                - ELSE:
#                  - end = FRS[last index]
#              - ELSE:
#                - end = FS/EOPS[last index]
#            - ELSE:
#              - IF (FRS < FS):
#                - end = FRS
#              - ELSE:
#                - end = (fumble spotting/end of play spotting)
#            - yardage = start -> end
#
#          - NOTE: Is there waste here?
#            - I'm not going to think of shortcuts right now.
#            - I'm almost positive that the only time a play can have an aborted fumble
#              is if the offense initially stumbles on the initial play and fumbles. I have
#              yet to see it anywhere else, so this is going to be my assumption.
#            (aborted_and_recovers_pattern)
#          - IF fd fits r.e. (aborted_and_recovers_pattern)
#            - start_spotting
#              - start = LOS
#            - end spotting
#              - IF (FS/EOPS is list):
#                - IF (FRS is list):
#                  - IF (len(FS/EOPS) > len(FRS)):
#                    - end = FS/EOPS[last index]
#                  - ELSE:
#                    - end = FRS[last index]
#                - ELSE:
#                  - end = FS/EOPS[last index]
#              - ELSE:
#                - IF (FRS < FS):
#                  - end = FRS
#                - ELSE:
#                  - end = (fumble spotting/end of play spotting)
#            - yardage = start -> end
#
#            (ball out of bounds)
#          - IF fd fits r.e. (fumble_out_of_bounds_pattern)
#            - start spotting (Where yardage begins for this possession)
#              - IF (Offense):
#                - IF (previous fumble recovery spotting):
#                  - IF (PFRS < LOS):
#                    - start = LOS
#                  - ELSE:
#                    - start = PFRS
#                - ELSE: (no previous fumble recovery spotting)
#                  - start = LOS
#              - ELSE: (defense)
#                - start = PFRS
#            - end spotting (Where yardage ends for this possession)
#              - IF (OOBS == ''): <- touchbacks. Ball went out of bounds in endzone.
#                - end = FS
#              - IF (OOBS < FS/EOPS):
#                - end = OOBS
#              - ELSE:
#                - end = FS
#            - yardage = start -> end
#
#            (center_aborted_pattern_2) <- Will probably have to adjust this in future.
#          - IF fd fits r.e. (center_aborted_pattern_2)
#            - start spotting
#              - IF (previous fumble recovery spotting):
#                - IF (PFRS < LOS):
#                  - start = LOS
#                - ELSE: (PFRS > LOS)
#                  - start = PFRS
#              - ELSE: (initial start of play)
#                - start = LOS
#            - end spotting:
#              - IF (fumble recovery):
#                - IF (same team fumble recovery):
#                  - IF (fumble recovery spotting < fumble spotting):
#                    - end = FRS
#                  - ELSE: (FRS > FS)
#                    - end = FS
#                - ELSE: (opposing team fumble recovery)
#                  - end = FS
#              - ELSE:
#                - end = EOPS
#            - yardage = start -> end
#
#            (fumble description is empty)
#          - IF (fumble description is empty):
#            - start spotting
#              - IF (offense):
#                - IF (PFRS < LOS):
#                  - start = LOS
#                - ELSE: (PFRS > LOS)
#                  - start = PFRS
#              - ELSE: (defense)
#                - start = PFRS
#            - end spotting
#              - end = (end of play spotting)
#              - IF (EOPS == ''):
#                - yardage = 0 (incomplete pass / interception)
#              - ELSE:
#                - yardage = start -> end
#
#            (DEFAULT)
#            (fumble by player 1 and recovery by player 2)
#            (aborted_and_recovered_by_pattern)
#            (muffed_catch_pattern)
#          - IF fd fits r.e. (fumble_pattern):
#          - IF fd fits r.e. (aborted_and_recovered_by_pattern):
#            - start spotting
#              - IF (offense):
#                - IF (previous fumble recovery spotting):
#                  - IF (PFRS < LOS):
#                    - start = LOS
#                  - ELSE: (PFRS > LOS)
#                    - start = PFRS
#                - ELSE: (initial start of play)
#                  - start = LOS
#              - ELSE: (defense)
#                - start = PFRS
#            - end spotting:
#              - IF (same team fumble recovery):
#                - IF (fumble recovery spotting < fumble spotting):
#                  - end = FRS
#                - ELSE: (FRS > FS)
#                  - end = FS
#              - ELSE: (opposing team fumble recovery)
#                - end = FS
#            - yardage = start -> end
#
#
#          - Check if this possession changes yardage for previous possessions
#            - NOTE: I have questions for this.
#              - What happens if an offensive player recovers in the backfield
#                and rushes 4 yards but its still behind the LOS? Does that
#                player recieve (-) yards (LOS -> EOPS)? Or do they receive 0?
#                - If they receive 0 yards, I would need to change this.
#          - IF (current team with possession == offense):
#            - IF (yardage >= 0):
#              - loop backwards in list:
#                - IF (team with possession == offense):
#                  - IF (yardage < 0):
#                    - yaradge for that possession = 0
#                  - ELSE:
#                    - break
#
#          - add new single row dataframe to return dataframe
#            - add yardage to single row dataframe
#            - add forced fumble player
#            - add player who applied pressure (still need to grab that in organization)
#            - add recovered by player
#            - add single row dataframe to return dataframe
#
#   4. return return dataframe
#
#   5. Replace old row with cleaned return dataframe
#      - I think for this part, I will have the methods that called this helper
#        method handle the replacement of the original row with this cleaned row.
#
#
#   ######################
#   #   UNIQUE EXAMPLES  #
#   ######################
#   - I want to have an example of all unique fumbles that I have come accros and
#     display how they have been cleaned here.
#
#
#   #######################
#   #   PASSING EXAMPLES  #
#   #######################
#   - All fumbles that started with a passing play type.
#
#   (UNIQUE FUMBLED PLAYS IN WEEK 2 SEASON 2023)
#
#   842
#   1. (passing play type -> fumble out of bounds)
#      (1) PLAY DESCRIPTION SPLIT:
#          — A.Richardson pass short left to M.Pittman to IND 37 for 12 yards (M.Stewart) [W.Anderson]
#          FUMBLES (M.Stewart), ball out of bounds at IND 42.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'IND', 'HOU', 'IND', 'IND 25', '— A.Richardson pass short left to M.Pittman to IND 37 for 12 yards (M.Stewart) [W.Anderson]', 'IND 37', 'FUMBLES (M.Stewart), ball out of bounds at IND 42.', 'M.Stewart', '', '', '', 'IND 42']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) passing play type
#          - (fd) fumble out of bounds
#            - offense
#              - no fumble recovery spotting
#                - start = LOS
#            - OOBS > FS
#              - end = FS
#            - yardage = LOS -> FS = 'IND 25' -> 'IND 37' = 12 yard pass
#
#   1135
#   2. (passing play type -> fumble and recovers -> run (still considered part of receiving yards))
#      (1) PLAY DESCRIPTION SPLIT:
#          — P.Mahomes pass short right to K.Toney to KC 22 for -1 yards (T.Herndon)
#          FUMBLES (T.Herndon), and recovers at KC 16
#          K.Toney to KC 12 for -4 yards (Dari.Williams, T.Herndon).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'KC', 'JAX', 'KC', 'KC 23', ['— P.Mahomes pass short right to K.Toney to KC 22 for -1 yards (T.Herndon)', 'K.Toney to KC 12 for -4 yards (Dari.Williams, T.Herndon).'], ['KC 22', 'KC 12'], 'FUMBLES (T.Herndon), and recovers at KC 16', 'T.Herndon', 'KC', '', 'KC 16', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - action description is list
#            - action_description = ad_list[0]
#              - (ad) passing play type
#          - (fd) and recovers
#            - offense
#              - no previous fumble spotting
#                - start = LOS
#            - FS/EOPS is list
#              - FRS is not a list
#                - end = FS/EOPS[last index]
#            - yardage = LOS -> EOPS = 'KC 23' -> 'KC 12' = -11 yard pass
#
#   1292
#   3. (passing play type -> default fumble)
#      (1) PLAY DESCRIPTION SPLIT:
#          — B.Mayfield pass short right to D.Wells to CHI 21 for no gain (J.Johnson)
#          FUMBLES (J.Johnson), recovered by TB-B.Mayfield at CHI 32.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'TB', 'CHI', 'TB', 'CHI 21', '— B.Mayfield pass short right to D.Wells to CHI 21 for no gain (J.Johnson)', 'CHI 21', 'FUMBLES (J.Johnson), recovered by TB-B.Mayfield at CHI 32.', 'J.Johnson', 'TB', 'B.Mayfield', 'CHI 32', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) passing play type
#          - (fd) default
#            - offense
#              - no fumble recovery spotting
#                - start = LOS
#            - same team fumble recovery
#              - FRS < FS
#                - end = FRS
#            - yardage = LOS -> FRS = 'CHI 21' -> 'CHI 32' = -11 yard pass
#
#   1909
#   4. (passing play type -> default fumble)
#      (1) PLAY DESCRIPTION SPLIT:
#          — D.Prescott pass short middle to C.Lamb to DAL 34 for 31 yards (A.Gardner; J.Sherwood)
#            FUMBLES (A.Gardner), recovered by DAL-T.Biadasz at DAL 36.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'DAL', 'NYJ', 'DAL', 'DAL 3', '— D.Prescott pass short middle to C.Lamb to DAL 34 for 31 yards (A.Gardner; J.Sherwood)', 'DAL 34', 'FUMBLES (A.Gardner), recovered by DAL-T.Biadasz at DAL 36.', 'A.Gardner', 'DAL', 'T.Biadasz', 'DAL 36', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) passing play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - same team fumble recovery
#              - (FRS > FS)
#                - end = FS
#            - yardage = LOS -> FS = 'DAL 3' -> 'DAL 34' = 31 yard pass
#
#   2167
#   5. UNIQUE PLAY (INTERCEPTION -> FUMBLE)
#      (1) PLAY DESCRIPTION SPLIT:
#          — R.Wilson pass short left intended for C.Sutton INTERCEPTED by E.Forbes at DEN 47
#          E.Forbes to DEN 44 for 3 yards
#          FUMBLES, ball out of bounds at DEN 44
#          The Replay Official reviewed the interception ruling, and the play was Upheld
#          The ruling on the field stands.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'DEN', 'WAS', 'DEN', 'DEN 36', '— R.Wilson pass short left intended for C.Sutton INTERCEPTED by E.Forbes at DEN 47', '', '', '', '', '', '', '']
#          i.1 ['', 'DEN 47', 'DEN', 'WAS', 'WAS', 'DEN 36', 'E.Forbes to DEN 44 for 3 yards', 'DEN 44', 'FUMBLES, ball out of bounds at DEN 44', '', '', '', '', 'DEN 44']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) passing play type
#            - pass intercepted
#          - (fd) empty
#            - offense
#              - doesn't matter..
#            - EOPS == ''
#              - yardage = 0
#          i.1
#          - (ad) rushing play type
#          - (fd) ball out of bounds
#            - defense
#              - start = previous fumble recovery spotting (spotting of interception)
#            - OOBS == FS/EOPS
#              - end = FS
#            - yardage = SOI -> FS = 'DEN 47' -> 'DEN 44' = 3 yard run
#
#   74
#   6. (pass play type -> fumble out of bounds -> touchback..?)
#      (1) PLAY DESCRIPTION SPLIT:
#          (Shotgun) K.Cousins pass deep left to J.Jefferson to PHI 1 for 30 yards (T.Edmunds)
#          FUMBLES (T.Edmunds), ball out of bounds in End Zone, Touchback.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'MIN', 'PHI', 'MIN', 'PHI 31', '(Shotgun) K.Cousins pass deep left to J.Jefferson to PHI 1 for 30 yards (T.Edmunds)', 'PHI 1', 'FUMBLES (T.Edmunds), ball out of bounds in End Zone, Touchback.', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) pass play type
#          - (fd) ball out of bounds
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - OOBS == ''
#              - end = FS
#            - yardage = LOS -> FS = 'PHI 31' -> 'PHI 1' = 30 yard pass
#
#   765
#   7. (pass play type -> opposing team fumble recovery -> run play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — J.Goff pass deep left to A.St. Brown to SEA 15 for 39 yards (T.Brown)
#          FUMBLES (T.Brown), RECOVERED by SEA-J.Love at SEA 6
#          J.Love pushed ob at SEA 9 for 3 yards (A.St. Brown; S.LaPorta).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'DET', 'SEA', 'DET', 'DET 46', '— J.Goff pass deep left to A.St. Brown to SEA 15 for 39 yards (T.Brown)', 'SEA 15', 'FUMBLES (T.Brown), RECOVERED by SEA-J.Love at SEA 6', 'T.Brown', 'SEA', 'J.Love', 'SEA 6', '']
#          i.1 ['SEA 15', 'SEA 6', 'DET', 'SEA', 'SEA', 'DET 46', 'J.Love pushed ob at SEA 9 for 3 yards (A.St. Brown; S.LaPorta).', 'SEA 9', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) pass play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#           - yardage = 'DET 46' -> 'SEA 15' = 39 yard pass
#          i.1
#          - (ad) run play type
#          - (fd) ''
#            - defense
#              - start = PFRS
#            - end = EOPS
#            - yardage = PFRS -> EOPS = 'SEA 6' -> 'SEA 9' = 3 yard run
#
#   1100
#   8. (passing play type -> opposing team fumble recovery -> penalty)
#      (1) PLAY DESCRIPTION SPLIT:
#          — P.Mahomes pass short right to Ju.Watson to JAX 47 for 14 yards (Dari.Williams)
#          FUMBLES (Dari.Williams), RECOVERED by JAX-F.Oluokun at JAX 48
#          PENALTY on KC-T.Kelce, Unsportsmanlike Conduct, 15 yards, enforced at JAX 48.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'KC', 'JAX', 'KC', 'KC 39', '— P.Mahomes pass short right to Ju.Watson to JAX 47 for 14 yards (Dari.Williams)', 'JAX 47', 'FUMBLES (Dari.Williams), RECOVERED by JAX-F.Oluokun at JAX 48', 'Dari.Williams', 'JAX', 'F.Oluokun', 'JAX 48', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) pass play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'KC 39' -> 'JAX 47' = 14 yard pass
#
#   2323
#   9. (pass play type -> opposing team fumble recovery)
#      (1) PLAY DESCRIPTION SPLIT:
#          — M.Jones pass short middle to D.Douglas to MIA 29 for 10 yards (B.Chubb)
#          FUMBLES (B.Chubb), RECOVERED by MIA-D.Elliott at MIA 27.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'NE', 'MIA', 'NE', 'MIA 39', '— M.Jones pass short middle to D.Douglas to MIA 29 for 10 yards (B.Chubb)', 'MIA 29', 'FUMBLES (B.Chubb), RECOVERED by MIA-D.Elliott at MIA 27.', 'B.Chubb', 'MIA', 'D.Elliott', 'MIA 27', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) passing play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'MIA 39' -> 'MIA 29' = 10 yard pass
#
#   2693
#   10. (pass play type -> opposing team fumble recovery -> run play type -> fumble out of bounds)
#      (1) PLAY DESCRIPTION SPLIT:
#          — K.Pickett pass short right to G.Olszewski to PIT 36 for -1 yards (D.Ward)
#          FUMBLES (D.Ward), RECOVERED by CLE-G.Delpit at PIT 37
#          G.Delpit to PIT 23 for 14 yards (P.Freiermuth)
#          FUMBLES (P.Freiermuth), touched at PIT 14, ball out of bounds at PIT 9.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'PIT', 'CLE', 'PIT', 'PIT 37', '— K.Pickett pass short right to G.Olszewski to PIT 36 for -1 yards (D.Ward)', 'PIT 36', 'FUMBLES (D.Ward), RECOVERED by CLE-G.Delpit at PIT 37', 'D.Ward', 'CLE', 'G.Delpit', 'PIT 37', '']
#          i.1 ['PIT 36', 'PIT 37', 'PIT', 'CLE', 'CLE', 'PIT 37', 'G.Delpit to PIT 23 for 14 yards (P.Freiermuth)', 'PIT 23', 'FUMBLES (P.Freiermuth), touched at PIT 14, ball out of bounds at PIT 9.', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) pass play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'PIT 37' -> 'PIT 36' = -1 yard pass
#          i.1
#          - (ad) run play type
#          - (fd) ball out of bounds
#            - defense
#              - start = PFRS
#            - OOBS > FS
#              - end = FS
#            - yardage = PFRS -> FS = 'PIT 37' -> 'PIT 23' = 14 yard run
#
#   2802
#   11. (pass play type -> opposing team fumble recovery -> run play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — D.Watson pass short middle to D.Njoku to PIT 46 for 13 yards (K.Neal, C.Holcomb)
#          FUMBLES (C.Holcomb), RECOVERED by PIT-D.Kazee at PIT 44
#          D.Kazee to PIT 44 for no gain (P.Strong).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'CLE', 'PIT', 'CLE', 'CLE 41', '— D.Watson pass short middle to D.Njoku to PIT 46 for 13 yards (K.Neal, C.Holcomb)', 'PIT 46', 'FUMBLES (C.Holcomb), RECOVERED by PIT-D.Kazee at PIT 44', 'C.Holcomb', 'PIT', 'D.Kazee', 'PIT 44', '']
#          i.1 ['PIT 46', 'PIT 44', 'CLE', 'PIT', 'PIT', 'CLE 41', 'D.Kazee to PIT 44 for no gain (P.Strong).', 'PIT 44', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) pass play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'CLE 41' -> 'PIT 46' = 13 yard pass
#          i.1
#          - (ad) run play type
#          - (fd) ''
#            - defense
#              - start = PFRS
#            - end = EOPS
#            - yardage = PFRS -> EOPS = 'PIT 44' -> 'PIT 44' = 0 yard run
#
#   #######################
#   #   RUSHING EXAMPLES  #
#   #######################
#   - All fumbles that started with a run play type.
#
#   963
#   1. (rushing play type -> default fumble -> passing play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — J.Patterson to HOU 36 for -5 yards
#          FUMBLES, recovered by HOU-C.Stroud at HOU 35
#          C.Stroud pass short right to R.Woods to HOU 49 for 8 yards (E.Speed).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'HOU', 'IND', 'HOU', 'HOU 41', '— J.Patterson to HOU 36 for -5 yards', 'HOU 36', 'FUMBLES, recovered by HOU-C.Stroud at HOU 35', '', 'HOU', 'C.Stroud', 'HOU 35', '']
#          i.1 ['HOU 36', 'HOU 35', 'HOU', 'IND', 'HOU', 'HOU 41', 'C.Stroud pass short right to R.Woods to HOU 49 for 8 yards (E.Speed).', 'HOU 49', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) rushing play type
#          - (fd) default
#            - offense
#              - no fumble recovery spotting
#                - start = LOS
#            - same team fumble recovery
#              - FRS < FS
#                - end = FRS
#            - yardage = LOS -> FRS = 'HOU 41' -> 'HOU 35' = -6 yard run
#          i.1
#          - (ad) passing play type
#          - (fd) ''
#            - offense
#              - PFRS < LOS
#                - start = LOS
#            - end = end of play spotting
#            - yardage = LOS -> EOPS = 'HOU 41' -> 'HOU 49' = 8 yard pass
#          - offense
#            - yardage > 0
#              - loop backwards
#                - i.0
#                  - TWP == offense
#                    - yardage < 0
#                      - yardage = 0 (i.0)
#
#   1037
#   2. (rushing play type -> default fumble -> passing play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — L.Fortner to JAX 22 for -9 yards
#          FUMBLES, recovered by JAX-T.Lawrence at JAX 19
#          T.Lawrence pass incomplete short left to Z.Jones.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'JAX', 'KC', 'JAX', 'JAX 31', '— L.Fortner to JAX 22 for -9 yards', 'JAX 22', 'FUMBLES, recovered by JAX-T.Lawrence at JAX 19', '', 'JAX', 'T.Lawrence', 'JAX 19', '']
#          i.1 ['JAX 22', 'JAX 19', 'JAX', 'KC', 'JAX', 'JAX 31', 'T.Lawrence pass incomplete short left to Z.Jones.', '', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) rushing play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - same team fumble recovery
#              - FRS < LOS
#                - end = FRS
#            - yardage = LOS -> FRS = 'JAX 31' -> 'JAX 19' = -12 yard run
#          i.1
#          - (ad) passing play type
#          - (fd) ''
#            - offense
#              - PFRS < LOS ('JAX 19' < 'JAX 31')
#                - start = LOS
#            - end = EOPS
#              - EOPS == ''
#                - yardage = 0
#          - offense
#            - yardage >= 0
#              - loop backwards
#                - i.0
#                  - TWP == offense ('JAX')
#                    - yardage < 0
#                      - yardage = 0 (i.0)
#
#                     GET TO THIS LATER ^^^
#                     - Worried about cleaning method. Currently, if a recovery possession
#                       results in 0 yards, I assume it was an incomplete pass or player
#                       rushed back to the LOS.
#                       - Is it possible for a player to recover a ball behind LOS and
#                         receive 0 yards rushing? Or would they receive LOS -> down, which
#                         when behind the LOS would result in (-) yards rushing.
#
#   30
#   3. (run play type -> opp team fumble recovery -> run play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — A.Mattison left tackle to MIN 37 for 2 yards (A.Maddox; J.Jobe)
#          FUMBLES (A.Maddox), RECOVERED by PHI-J.Evans at MIN 39
#          J.Evans to MIN 39 for no gain (A.Mattison).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'MIN', 'PHI', 'MIN', 'MIN 35', '— A.Mattison left tackle to MIN 37 for 2 yards (A.Maddox; J.Jobe)', 'MIN 37', 'FUMBLES (A.Maddox), RECOVERED by PHI-J.Evans at MIN 39', 'A.Maddox', 'PHI', 'J.Evans', 'MIN 39', '']
#          i.1 ['MIN 37', 'MIN 39', 'MIN', 'PHI', 'PHI', 'MIN 35', 'J.Evans to MIN 39 for no gain (A.Mattison).', 'MIN 39', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) run play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - opposite team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'MIN 35' -> 'MIN 37' = 2 yard run
#          i.1
#          - (ad) run play type
#          - (fd) ''
#            - defense
#              - start = PFRS
#            - end = EOPS
#            - yardage = PFRS -> EOPS = 'MIN 39' -> 'MIN 39' = 0 yard run
#
#   506
#   4. (run play type -> opposing team fumble recovery -> run play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — Z.White left tackle to BUF 15 for no gain (D.Jackson)
#          FUMBLES (D.Jackson), RECOVERED by BUF-T.Rapp at BUF 13
#          T.Rapp to BUF 19 for 6 yards (T.Tucker)
#          Penalty on LV-M.Mayer, Offensive Holding, declined.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'LV', 'BUF', 'LV', 'BUF 15', '— Z.White left tackle to BUF 15 for no gain (D.Jackson)', 'BUF 15', 'FUMBLES (D.Jackson), RECOVERED by BUF-T.Rapp at BUF 13', 'D.Jackson', 'BUF', 'T.Rapp', 'BUF 13', '']
#          i.1 ['BUF 15', 'BUF 13', 'LV', 'BUF', 'BUF', 'BUF 15', 'T.Rapp to BUF 19 for 6 yards (T.Tucker)', 'BUF 19', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) run play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = 'BUF 15' -> 'BUF 15' = 0 yard gain
#          i.1
#          - (ad) run play type
#          - (fd) ''
#            - defense
#              - start = PFRS
#            - end = EOPS
#            - yardage = PFRS -> EOPS = 'BUF 13' -> 'BUF 19' = 6 yard rush
#
#   767
#   5. (run play type -> opposing team fumble recovery)
#      (1) PLAY DESCRIPTION SPLIT:
#          — D.Montgomery up the middle to DET 22 for -3 yards (U.Nwosu)
#          FUMBLES (U.Nwosu), RECOVERED by SEA-Ja.Reed at DET 23.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'DET', 'SEA', 'DET', 'DET 25', '— D.Montgomery up the middle to DET 22 for -3 yards (U.Nwosu)', 'DET 22', 'FUMBLES (U.Nwosu), RECOVERED by SEA-Ja.Reed at DET 23.', 'U.Nwosu', 'SEA', 'Ja.Reed', 'DET 23', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) run play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = 'DET 25' -> 'DET 22' = -3 yard run
#
#   2042
#   6. (run play type -> opposing team fumble recovery -> run play type -> injury)
#      (1) PLAY DESCRIPTION SPLIT:
#          D.Cook left end to NYJ 37 for -2 yards (M.Parsons)
#          FUMBLES (M.Parsons), RECOVERED by DAL-M.Parsons at NYJ 37
#          M.Parsons to NYJ 37 for no gain (L.Tomlinson)
#          NYJ-D.Cook was injured during the play.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'NYJ', 'DAL', 'NYJ', 'NYJ 39', 'D.Cook left end to NYJ 37 for -2 yards (M.Parsons)', 'NYJ 37', 'FUMBLES (M.Parsons), RECOVERED by DAL-M.Parsons at NYJ 37', 'M.Parsons', 'DAL', 'M.Parsons', 'NYJ 37', '']
#          i.1 ['NYJ 37', 'NYJ 37', 'NYJ', 'DAL', 'DAL', 'NYJ 39', 'M.Parsons to NYJ 37 for no gain (L.Tomlinson)', 'NYJ 37', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) run play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'NYJ 39' -> 'NYJ 37' = -2 yards rushing
#          i.1
#          - (ad) run play type
#          - (fd) ''
#            - defense
#              - start = PFRS
#            - end = EOPS
#            - yardage = PFRS -> EOPS = 'NYJ 37' -> 'NYJ 37' = 0 yards rushing
#
#   2691
#   7. (run play type -> opposing team fumble recovery)
#      (1) PLAY DESCRIPTION SPLIT:
#          M.Dunn reported in as eligible
#          D.Watson left guard to PIT 42 for 1 yard (L.Ogunjobi, M.Adams)
#          FUMBLES (M.Adams), RECOVERED by PIT-L.Ogunjobi at PIT 42.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'CLE', 'PIT', 'CLE', 'PIT 43', 'D.Watson left guard to PIT 42 for 1 yard (L.Ogunjobi, M.Adams)', 'PIT 42', 'FUMBLES (M.Adams), RECOVERED by PIT-L.Ogunjobi at PIT 42.', 'M.Adams', 'PIT', 'L.Ogunjobi', 'PIT 42', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) run play type
#          - (fd) default
#            - offense
#              - no previous fumble recovery
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'PIT 43' -> 'PIT 42' = 1 yard rushing
#
#   ########################
#   #   ABORTED EXAMPLES   #
#   ########################
#   - All fumbles that started with a run play type.
#
#   936
#   1. (Aborted and recovered by -> rushing play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — C.Stroud FUMBLES (Aborted) at HOU 18, recovered by HOU-N.Dell at HOU 16
#          N.Dell to HOU 16 for no gain (Z.Franklin).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'HOU', 'IND', 'HOU', 'HOU 25', '— C.Stroud FUMBLES (Aborted) at HOU 18, recovered by HOU-N.Dell at HOU 16', 'HOU 18', '— C.Stroud FUMBLES (Aborted) at HOU 18, recovered by HOU-N.Dell at HOU 16', '', 'HOU', 'N.Dell', 'HOU 16', '']
#          i.1 ['HOU 18', 'HOU 16', 'HOU', 'IND', 'HOU', 'HOU 25', 'N.Dell to HOU 16 for no gain (Z.Franklin).', 'HOU 16', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) aborted and recovered by
#          - (fd) default (aborted and recovered by)
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - same team fumble recovery
#              - FRS < FS
#                - end = FRS
#            - yardage = LOS -> FRS = 'HOU 25' -> 'HOU 16' = -9 yard run
#          i.1
#          - (ad) rushing play type
#          - (fd) ''
#            - offense ('HOU' == 'HOU')
#              - PFRS < LOS
#                - start = LOS
#            - end = FS/EOPS
#            - yardage = LOS -> FS/EOPS = 'HOU 25' -> 'HOU 16' = -9 yard run
#
#          NOTE: Do both players recieve -9 yard run? Who gets this. This is going to be a problem...
#
#   1073
#   2. (Center aborted fumble)
#      unique because:
#      1. quarterback recovered ball..?
#      (1) PLAY DESCRIPTION SPLIT:
#          — P.Mahomes Aborted
#          C.Humphrey FUMBLES at KC 39, touched at KC 38, recovered by KC-P.Mahomes at KC 40
#          P.Mahomes to KC 44 for 4 yards (D.Lloyd).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'KC', 'JAX', 'KC', 'KC 43', ['— P.Mahomes Aborted', 'P.Mahomes to KC 44 for 4 yards (D.Lloyd).'], 'KC 44', 'C.Humphrey FUMBLES at KC 39, touched at KC 38, recovered by KC-P.Mahomes at KC 40', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) action description is list
#            - ad_list[0] is center_aborted_pattern_1
#              - (ad) rushing play type
#          - (fd) center_aborted_pattern_2
#            - no previous fumble recovery spotting
#              - start = LOS
#            - no fumble recovery (own recovery)
#              - end = FS/EOPS
#            - yardage = LOS -> EOPS = 'KC 43' -> 'KC 44' -> 1 yard run
#
#   1322
#   3. ((Aborted) and recovers)
#      (1) PLAY DESCRIPTION SPLIT:
#          — J.Fields FUMBLES (Aborted) at CHI 41, and recovers at CHI 38.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'CHI', 'TB', 'CHI', 'CHI 46', '— J.Fields FUMBLES (Aborted) at CHI 41, and recovers at CHI 38.', 'CHI 41', '— J.Fields FUMBLES (Aborted) at CHI 41, and recovers at CHI 38.', '', '', '', 'CHI 38', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) aborted_and_recovers_pattern
#          - (fd) aborted_and_recovers_pattern
#            - start = LOS
#            - FS/EOPS is not list
#              - FRS < FS
#                - end = FRS
#            - yardage = LOS -> FRS = 'CHI 46' -> 'CHI 38' -> -8 yard run
#
#   2388
#   4. ((Aborted) and recovers -> run play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — T.Tagovailoa FUMBLES (Aborted) at NE 35, and recovers at NE 37
#          T.Tagovailoa to NE 37 for no gain (M.Judon).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'MIA', 'NE', 'MIA', 'NE 35', ['— T.Tagovailoa FUMBLES (Aborted) at NE 35, and recovers at NE 37', 'T.Tagovailoa to NE 37 for no gain (M.Judon).'], ['NE 35', 'NE 37'], '— T.Tagovailoa FUMBLES (Aborted) at NE 35, and recovers at NE 37', '', '', '', 'NE 37', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - ad is list
#            - aborted and recovers
#              - ad = ad_list[1]
#                - (ad) run play type
#          - (fd) aborted and recovers
#            - start = LOS
#            - FS/EOPS is list
#              - FRS is not list
#                - end = FS/EOPS[last index]
#            - yardage = LOS -> FS/EOPS[last index] = 'NE 35' -> 'NE 37' = -2 yard run
#
#   1101
#   5. (aborted and recovered by -> opposing team fumble recovery -> run play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          (Shotgun) T.Lawrence FUMBLES (Aborted) at KC 43, touched at KC 42, RECOVERED by KC-L.Sneed at KC 43
#          L.Sneed to KC 43 for no gain (J.Agnew).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'JAX', 'KC', 'JAX', 'KC 37', '(Shotgun) T.Lawrence FUMBLES (Aborted) at KC 43, touched at KC 42, RECOVERED by KC-L.Sneed at KC 43', 'KC 43', '(Shotgun) T.Lawrence FUMBLES (Aborted) at KC 43, touched at KC 42, RECOVERED by KC-L.Sneed at KC 43', '', 'KC', 'L.Sneed', 'KC 43', '']
#          i.1 ['KC 43', 'KC 43', 'JAX', 'KC', 'KC', 'KC 37', 'L.Sneed to KC 43 for no gain (J.Agnew).', 'KC 43', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) aborted and recovery by
#          - (fd) default
#            - offense
#              - no previous fumble recovery pattern
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'KC 37' -> 'KC 43' -> -6 yard run
#          i.1
#          - (ad) run play type
#          - (fd) ''
#            - defense
#              - start = PFRS
#            - end = EOPS
#            - yardage = PFRS -> EOPS = 'KC 43' -> 'KC 43' = 0 yard run
#
#   #######################
#   #   SACKED EXAMPLES   #
#   #######################
#   - All fumbles that started with a sack.
#
#   1182
#   1. (sack -> fumble out of bounds)
#      (1) PLAY DESCRIPTION SPLIT:
#          (Shotgun) T.Lawrence sacked at KC 16 for -2 yards (sack split by F.Anudike-Uzomah and C.Jones)
#          FUMBLES (F.Anudike-Uzomah) [C.Jones], ball out of bounds at KC 16.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'JAX', 'KC', 'JAX', 'KC 14', '(Shotgun) T.Lawrence sacked at KC 16 for -2 yards (sack split by F.Anudike-Uzomah and C.Jones)', 'KC 16', 'FUMBLES (F.Anudike-Uzomah) [C.Jones], ball out of bounds at KC 16.', 'F.Anudike-Uzomah', '', '', '', 'KC 16']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) sacked
#          - (fd) fumble out of bounds
#            - offense
#              - not previous fumble recovery
#                - start = LOS
#            - OOBS == FS/EOPS
#              - end = FS
#            - yardage = LOS -> FS = 'KC 14' -> 'KC 16' = -2 yard sack
#
#   1260
#   2. (sack -> default fumble)
#      (1) PLAY DESCRIPTION SPLIT:
#          — J.Fields sacked at TB 37 for -10 yards (C.Gill)
#          FUMBLES (C.Gill) [C.Gill], recovered by CHI-L.Patrick at TB 34.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'CHI', 'TB', 'CHI', 'TB 27', '— J.Fields sacked at TB 37 for -10 yards (C.Gill)', 'TB 37', 'FUMBLES (C.Gill) [C.Gill], recovered by CHI-L.Patrick at TB 34.', 'C.Gill', 'CHI', 'L.Patrick', 'TB 34', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) sacked
#          - (fd) default
#            - offense
#              - no fumble recovery spotting
#                - start = LOS
#            - same team fumble recovery
#              - FRS > FS
#                - end = FS
#            - yardage = LOS -> FS = 'TB 27' -> 'TB 37' = -10 yards rushing
#
#   87
#   3. (sacked -> opposing team fumble recovery -> run play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — K.Cousins sacked at MIN 18 for -8 yards (J.Sweat)
#          FUMBLES (J.Sweat) [J.Sweat], RECOVERED by PHI-F.Cox at MIN 15
#          F.Cox to MIN 7 for 8 yards (E.Ingram).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'MIN', 'PHI', 'MIN', 'MIN 26', '— K.Cousins sacked at MIN 18 for -8 yards (J.Sweat)', 'MIN 18', 'FUMBLES (J.Sweat) [J.Sweat], RECOVERED by PHI-F.Cox at MIN 15', 'J.Sweat', 'PHI', 'F.Cox', 'MIN 15', '']
#          i.1 ['MIN 18', 'MIN 15', 'MIN', 'PHI', 'PHI', 'MIN 26', 'F.Cox to MIN 7 for 8 yards (E.Ingram).', 'MIN 7', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) sacked
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = 'MIN 26' -> 'MIN 18' = -8 yard run
#          i.1
#          - (ad) run play type
#          - (fd) ''
#            - defense
#              - start = PFRS
#            - end = EOPS
#            - yardage = 'MIN 15' -> 'MIN 7' = 8 yard run
#
#   879
#   4. (sacked -> default fumble)
#      (1) PLAY DESCRIPTION SPLIT:
#          (Shotgun) C.Stroud sacked at HOU 14 for -9 yards (S.Ebukam)
#          FUMBLES (S.Ebukam) [S.Ebukam], RECOVERED by IND-K.Paye at HOU 15
#          Fumble Forced by IND-54-D.Odeyingbo.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'HOU', 'IND', 'HOU', 'HOU 23', '(Shotgun) C.Stroud sacked at HOU 14 for -9 yards (S.Ebukam)', 'HOU 14', 'FUMBLES (S.Ebukam) [S.Ebukam], RECOVERED by IND-K.Paye at HOU 15', ['S.Ebukam', 'D.Odeyingbo'], 'IND', 'K.Paye', 'HOU 15', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) sacked
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = 'HOU 23' -> 'HOU 14' = -9 yard run
#
#   2165
#   5. (sacked -> opposing team fumble recovery -> run play type)
#      (1) PLAY DESCRIPTION SPLIT:
#          — R.Wilson sacked at WAS 46 for -1 yards (J.Davis)
#          FUMBLES (J.Davis) [J.Davis], RECOVERED by WAS-C.Barton at WAS 46
#          C.Barton to DEN 49 for 5 yards (B.Powers).
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'DEN', 'WAS', 'DEN', 'WAS 45', '— R.Wilson sacked at WAS 46 for -1 yards (J.Davis)', 'WAS 46', 'FUMBLES (J.Davis) [J.Davis], RECOVERED by WAS-C.Barton at WAS 46', 'J.Davis', 'WAS', 'C.Barton', 'WAS 46', '']
#          i.1 ['WAS 46', 'WAS 46', 'DEN', 'WAS', 'WAS', 'WAS 45', 'C.Barton to DEN 49 for 5 yards (B.Powers).', 'DEN 49', '', '', '', '', '', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) sacked
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'WAS 45' -> 'WAS 46' = -1 yard rushing
#          i.1
#          - (ad) run play type
#          - (fd) ''
#            - defense
#              - start = PFRS
#            - end = EOPS
#            - yardage = PRFS -> EOPS = 'WAS 46' -> 'DEN 49' = 5 yard rushing
#
#   2549
#   6. (sacked -> opposing team fumble recovery (touched))
#      (1) PLAY DESCRIPTION SPLIT:
#          — B.Young sacked at NO 23 for -5 yards (C.Granderson)
#          FUMBLES (C.Granderson) [C.Granderson], touched at NO 21, RECOVERED by NO-P.Adebo at NO 41.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'CAR', 'NO', 'CAR', 'NO 18', '— B.Young sacked at NO 23 for -5 yards (C.Granderson)', 'NO 23', 'FUMBLES (C.Granderson) [C.Granderson], touched at NO 21, RECOVERED by NO-P.Adebo at NO 41.', 'C.Granderson', 'NO', 'P.Adebo', 'NO 41', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) sacked
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'NO 18' -> 'NO 23' = -5 yard run
#
#   #############################
#   #   PUNT/KICKOFF EXAMPLES   #
#   #############################
#   - All fumbles that involve a kickoff or punt.
#
#   0
#   1. (punt return -> default fumble)
#      (1) PLAY DESCRIPTION SPLIT:
#          B.Covey to PHI 16 for 8 yards (T.Dye)
#          FUMBLES (T.Dye), recovered by PHI-K.Ringo at PHI 10.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'PHI', 'MIN', 'PHI', 'PHI 8', 'B.Covey to PHI 16 for 8 yards (T.Dye)', 'PHI 16', 'FUMBLES (T.Dye), recovered by PHI-K.Ringo at PHI 10.', 'T.Dye', 'PHI', 'K.Ringo', 'PHI 10', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) run play type (punt return)
#          - (fd) default
#            - offense
#              - no previous fumble recovery
#                - start = LOS <- Not LOS but spotting of catch...
#            - same team fumble recovery
#              - FRS < FS
#                - end = FRS
#            - yardage = LOS -> FRS = 'PHI 8' -> 'PHI 10' = 2 yard return
#
#   0
#   2. (punt return -> ball out of bounds)
#      (1) PLAY DESCRIPTION SPLIT:
#          D.Thompkins to TB 26 for 9 yards (N.Sewell; D.Cole)
#          FUMBLES (N.Sewell), ball out of bounds at TB 23
#          Penalty on TB-K.Britt, Defensive Offside, declined.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'TB', 'CHI', 'TB', 'TB 17', 'D.Thompkins to TB 26 for 9 yards (N.Sewell; D.Cole)', 'TB 26', 'FUMBLES (N.Sewell), ball out of bounds at TB 23', 'N.Sewell', '', '', '', 'TB 23']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) run play type (punt return)
#          - (fd) ball out of bounds
#            - offense
#              - no previous fumble recovery
#                - start = LOS <- Not LOS but spotting of catch
#            - OOBS < FS/EOPS
#              - end = OOBS
#            - yardage = LOS -> OOBS = 'TB 17' -> 'TB 23' = 5 yard return
#
#   0
#   3. (muffed catch -> default fumble)
#      (1) PLAY DESCRIPTION SPLIT:
#          R.James MUFFS catch, RECOVERED by JAX-T.Jones at KC 17.
#
#      (2) ORGANIZE SEPARATE ACTION DESCRIPTIONS:
#          i.0 ['', '', 'KC', 'JAX', 'KC', 'KC 16', 'R.James MUFFS catch, RECOVERED by JAX-T.Jones at KC 17.', 'KC 16', 'R.James MUFFS catch, RECOVERED by JAX-T.Jones at KC 17.', '', 'JAX', 'T.Jones', 'KC 17', '']
#
#      (3) CLEAN AND RETURN
#          i.0
#          - (ad) muffed catch
#          - (fd) default
#            - offense
#              - no previous fumble recovery spotting
#                - start = LOS
#            - opposing team fumble recovery
#              - end = FS
#            - yardage = LOS -> FS = 'KC 16' -> 'KC 16' = 0 yard return

In [20]:
# NEW VERSION

# This is written out code from the design found above.

# not ready yet.

def helper_clean_fumble_play(df_plays, play_index):

  print(play_index)

  df_play = df_plays.loc[play_index].copy()

  print(df_play['PlayOutcome'])

  ##########################################
  # 1. Split play description by sentences #
  ##########################################

  play_description = df_play['PlayDescription']

  ############
  # REVERSES #
  ############
  # In 'PlayDescription' all information before the "reversed" sentence is not needed.
  # - All information before is stored within 'ReverseDetails' and the remaining is cleaned.
  if play_description.find('REVERSED') != -1:
    play_elements = split_play_description(play_description)
    for i in play_elements:
      if i.find("REVERSED") != -1:
        df_play['ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
        play_description = ". ".join(play_elements[play_elements.index(i) + 1:])
        break

  play_description_split = split_play_description(play_description)

  play_description_length = "- 1. PLAY DESCRIPTION -"
  print("-" * len(play_description_length))
  print(play_description_length)
  print("-" * len(play_description_length))
  print(play_description)
  print()

  play_description_split_length = "- 2. PLAY DESCRIPTION SPLIT -"
  print("-" * len(play_description_split_length))
  print(play_description_split_length)
  print("-" * len(play_description_split_length))
  for i in play_description_split:
    print(i)
  print()

  ######################################################
  # 2. Organize play into separate action descriptions #
  ######################################################

  # Create variables that will fill through each loop
  previous_fumble_spotting = ''
  previous_fumble_recovery = ''
  team_on_offense = df_play['TeamWithPossession']
  team_on_defense = df_play['AwayTeam'] if team_on_offense == df_play['HomeTeam'] else df_play['HomeTeam']
  current_team_with_possession = team_on_offense
  line_of_scrimmage = df_play['FieldPosition']

  # Create lists
  cronological_list_of_possessions = []
  # Append variables for initial play
  current_possession = [previous_fumble_spotting,
                        previous_fumble_recovery,
                        team_on_offense,
                        team_on_defense,
                        current_team_with_possession,
                        line_of_scrimmage,
                        '', '', '', '', '', '', '', '']

  # Loop through each sentence within play
  for sentence in play_description_split:
    # - Each sentence will fall into 1 of 5 categories

    # 1. Additional data (reviews/rulings/etc..)
    if sentence.lower().find('reported in as eligible') != -1:
      continue
    if sentence.find('The Replay Official reviewed the interception ruling') != -1:
      if current_possession[7] == '':
        cronological_list_of_possessions.append(current_possession.copy())
      continue
    if sentence.find('The ruling on the field stands') != -1:
      if current_possession[7] == '':
        cronological_list_of_possessions.append(current_possession.copy())
      continue
    if sentence.lower().find('penalty on') != -1:
      if current_possession[7] == '':
        cronological_list_of_possessions.append(current_possession.copy())
      continue
    if sentence.lower().find('was injured') != -1:
      if current_possession[7] == '':
        cronological_list_of_possessions.append(current_possession.copy())
      continue
    explicit_ffp = re.findall(forced_fumble_pattern, sentence)
    if explicit_ffp:
      if cronological_list_of_possessions[-1][9] == '':
        cronological_list_of_possessions[-1][9] = explicit_ffp[0][1]
      else:
        if isinstance(cronological_list_of_possessions[-1][9], list):
          cronological_list_of_possessions[-1][9].append(explicit_ffp[0][1])
        else:
          cronological_list_of_possessions[-1][9] = [cronological_list_of_possessions[-1][9], explicit_ffp[0][1]]
      continue

    # 2. Aborted fumble (Will probably have to be adjusted once sample size gets bigger)
    if sentence.find('Aborted') != -1:

      # player aborts and recovers
      player_abort_recovers = re.findall(aborted_and_recovers_pattern, sentence)
      if player_abort_recovers:
        # - action description
        current_possession[6] = sentence
        # - fumble spotting
        current_possession[7] = player_abort_recovers[0][1]
        # - fumble description
        current_possession[8] = sentence
        # - fumble recovery spotting
        current_possession[12] = player_abort_recovers[0][4]
        # final sentence check
        if sentence == play_description_split[-1]:
          cronological_list_of_possessions.append(current_possession.copy())
        continue

      # player aborts and another player recovers
      player_abort_recovered_by = re.findall(aborted_and_recovered_by_pattern, sentence)
      if player_abort_recovered_by:
        # - action description
        current_possession[6] = sentence
        # - fumble spotting
        current_possession[7] = player_abort_recovered_by[0][1]
        # - fumble description
        current_possession[8] = sentence
        # - fumble recovery team
        current_possession[10] = player_abort_recovered_by[0][7]
        # - fumble recovery player
        current_possession[11] = player_abort_recovered_by[0][8]
        # - fumble recovery spotting
        current_possession[12] = player_abort_recovered_by[0][9]
        # 1. append 'current_possession' to list of actions
        cronological_list_of_possessions.append(current_possession.copy())
        # 2. final sentence check
        if sentence != play_description_split[-1]:
          current_possession_reset = [current_possession[7],
                                      current_possession[12],
                                      current_possession[2],
                                      current_possession[3],
                                      current_possession[10],
                                      current_possession[5],
                                      '', '', '', '', '', '', '', '']
          current_possession = current_possession_reset.copy()
        continue

      # center aborted fumble (part 1)
      center_aborted = re.findall(center_aborted_pattern_1, sentence)
      if center_aborted:
        # - action description
        current_possession[6] = sentence
        continue

    # 3. muffed catch
    muffed_catch = re.findall(muffed_catch_pattern, sentence)
    if muffed_catch:
      # - action description
      current_possession[6] = sentence
      # - fumble spotting
      current_possession[7] = current_possession[5]
      # - fumble description
      current_possession[8] = sentence
      # - fumble recovery team
      current_possession[10] = muffed_catch[0][1]
      # - fumble recovery player
      current_possession[11] = muffed_catch[0][2]
      # - fumble recovery spotting
      current_possession[12] = muffed_catch[0][3]
      if sentence != play_description_split[-1]:
        current_possession_reset = [current_possession[7],
                                    current_possession[12],
                                    current_possession[2],
                                    current_possession[3],
                                    current_possession[10],
                                    current_possession[5],
                                    '', '', '', '', '', '', '', '']
        current_possession = current_possession_reset.copy()
      cronological_list_of_possessions.append(current_possession.copy())
      continue

    # 4. fumble description
    if sentence.find('FUMBLES') != -1:
      # (continue on own fumble recovery)
      if current_possession[8] != '':
        # - fumble description
        if isinstance(current_possession[8], list):
          current_possession[8].append(sentence)
        else:
          current_possession[8] = [current_possession[8], sentence]
      else:
        current_possession[8] = sentence

      # fumble out of bounds
      out_of_bounds = re.findall(fumble_out_of_bounds_pattern, sentence)
      if out_of_bounds:
        # - forced fumble player
        if current_possession[9] != '':
          if isinstance(current_possession[9], list):
            current_possession[9].append(out_of_bounds[0][0])
          else:
            current_possession[9] = [current_possession[9], out_of_bounds[0][0]]
        else:
          current_possession[9] = out_of_bounds[0][0]
        # - out of bounds spotting
        current_possession[13] = out_of_bounds[0][5]
        cronological_list_of_possessions.append(current_possession.copy())
        continue

      # 2. 'and recovers' own fumble recovery
      own_fumble_recovery = re.findall(own_fumble_recovery_pattern, sentence)
      if own_fumble_recovery:
        # - forced fumble player
        # multiple
        if current_possession[9] != '':
          if isinstance(current_possession[9], list):
            current_possession[9].append(own_fumble_recovery[0][0])
          else:
            current_possession[9] = [current_possession[9], own_fumble_recovery[0][0]]
        else:
          # first
          current_possession[9] = own_fumble_recovery[0][0]
        # - team fumble recovery
        team_with_possession = current_possession[1] if current_possession[1] != '' else current_possession[4]
        if current_possession[10] != '':
          if isinstance(current_possession[10], list):
            current_possession[10].append(team_with_possession)
          else:
            current_possession[10] = [current_possession[10], team_with_possession]
        else:
          current_possession[10] = team_with_possession
        # - fumble recovery player (Possibly take this out for own recoveies?)
        # - fumble recovery spotting
        if current_possession[12] != '':
          if isinstance(current_possession[12], list):
            current_possession[12].append(own_fumble_recovery[0][1])
          else:
            current_possession[12] = [current_possession[12], own_fumble_recovery[0][1]]
        else:
          # first
          current_possession[12] = own_fumble_recovery[0][1]
        continue

      # 3. center aborted fumble (part 2)
      center_aborted_fumble = re.findall(center_aborted_pattern_2, sentence)
      if center_aborted_fumble:
        print(center_aborted_fumble)
        player_aborted = re.findall(center_aborted_pattern_1, current_possession[6])
        # check for own recovery
        if player_aborted:
          if player_aborted[0] == center_aborted_fumble[0][8]:
            # - fumble recovery player (Possibly take this out for own recoveies?)
            continue
        # - fumble spotting
        current_possession[7] = center_aborted_fumble[0][1]
        # - team fumble recovery
        current_possession[10] = center_aborted_fumble[0][7]
        # - fumble recovery player
        current_possession[11] = center_aborted_fumble[0][8]
        # - fumble recovery spotting
        current_possession[12] = center_aborted_fumble[0][9]

      # 4. Default (regular fumble)
      fumble = re.findall(fumble_pattern, sentence)
      if fumble:
        # - forced fumble player
        if current_possession[9] != '':
          if isinstance(current_possession[9], list):
            current_possession[9].append(fumble[0][0])
          else:
            current_possession[9] = [current_possession[9], fumble[0][0]]
        else:
          current_possession[9] = fumble[0][0]
        # - team fumble recovery
        if current_possession[10] != '':
          if isinstance(current_possession[10], list):
            current_possession[10].append(fumble[0][5])
          else:
            current_possession[10] = [current_possession[10], fumble[0][5]]
        else:
          current_possession[10] = fumble[0][5]
        # - fumble recovery player
        if current_possession[11] != '':
          if isinstance(current_possession[11], list):
            current_possession[11].append(fumble[0][6])
          else:
            current_possession[11] = [current_possession[11], fumble[0][6]]
        else:
          current_possession[11] = fumble[0][6]
        # - fumble recovery spotting
        if current_possession[12] != '':
          if isinstance(current_possession[12], list):
            current_possession[12].append(fumble[0][7])
          else:
            current_possession[12] = [current_possession[12], fumble[0][7]]
        else:
          current_possession[12] = fumble[0][7]

      # Fumble recovery description found
      # 1. append 'current_possession' to list of actions
      cronological_list_of_possessions.append(current_possession.copy())
      # 2. reset 'current_possession
      current_possession_reset = [current_possession[7],
                                  current_possession[12],
                                  current_possession[2],
                                  current_possession[3],
                                  current_possession[10],
                                  current_possession[5],
                                  '', '', '', '', '', '', '', '']
      current_possession = current_possession_reset.copy()
      continue

    # 5. Action Description (default)
    # - action description
    if current_possession[6] != '':
      if isinstance(current_possession[6], list):
        current_possession[6].append(sentence)
      else:
        current_possession[6] = [current_possession[6], sentence]
    else:
      current_possession[6] = sentence

    # Check for interception action description
    interception_description = re.findall(interception_name_pattern, sentence)
    if interception_description:
      # - append 'current_possession' to list of actions
      cronological_list_of_possessions.append(current_possession.copy())
      # - reset 'current_possession
      current_possession_reset = ['', # FS/EOPS not applicable during interception
                                  interception_description[0][1], # FRS <- spotting of interception
                                  current_possession[2],
                                  current_possession[3],
                                  current_possession[3], # TOD now has possession
                                  current_possession[5],
                                  '', '', '', '', '', '', '', '']
      current_possession = current_possession_reset.copy()
      continue

    # - fumble spotting / end of play spotting / interception spotting
    fumble_spotting = re.findall(spotting_pattern, sentence)
    if fumble_spotting:
      if current_possession[7] != '':
        if isinstance(current_possession[7], list):
          current_possession[7].append(" ".join(fumble_spotting[0]))
        else:
          current_possession[7] = [current_possession[7], " ".join(fumble_spotting[0])]
      else:
        current_possession[7] = " ".join(fumble_spotting[0])

    # final sentence check
    if sentence == play_description_split[-1]:
      # 1. append 'current_possession' to list of actions
      cronological_list_of_possessions.append(current_possession.copy())

  play_description_grouped_length = "- 3. PLAYDESCRIPTION GROUPED -"
  print("-" * len(play_description_grouped_length))
  print(play_description_grouped_length)
  print("-" * len(play_description_grouped_length))
  for j in cronological_list_of_possessions:
    print(j)
  print()
  print()

  ################################
  # 3. Clean action descriptions #
  ################################

  # - create a return dataframe
  df_cleaned_play = pd.Dataframe()

  # - Loop through each element in [organized list of all actions within play]
  for possession in cronological_list_of_possessions:

    # - create a single row dataframe (a copy of the original)
    df_possession = df_play.copy()

    # - clean action description
    action_description = possession[6]

    if isinstance(action_description, list):
      # check for center aborted OR aborted and recovers action_description
      center_aborted = re.findall(center_aborted_pattern_1, action_description[0])
      aborted_and_recovers = re.findall(aborted_and_recovers_pattern, action_description[0])
      if center_aborted or aborted_and_recovers:
        action_description = action_description[1]
      else:
        action_description = action_description[0]

    # (passing action description)
    if (re.findall(receiver_pattern, action_description)):
      df_possession['PlayDescription'] = action_description
      if (re.findall(interception_name_pattern, action_description)):
        df_possession = clean_intercepted_plays(df_possession)
      else:
        df_possession = clean_pass_plays(df_possession)




  return

SyntaxError: expected ':' (2113612905.py, line 387)

#### LATERALS

In [ ]:
# Lateral cleaning method

# NOTES AND THOUGHTS:
# - I think yardage gained here is similar to yardage gained during fumbles
#   - What I think the rules for yardage rewarded are:
#     1. Yardage awarded = Start_Spotting -> End_Spotting
#        - Start spotting could be:
#          1. line of scrimmage
#          2. From where the ball was received during a lateral
#        - End spotting could be:
#          1. Tackled
#          2. Spotting of player lateralling ball
#          3. Run out of bounds
#          4. touchdown
#          5. Fumble
#             - Hopefully this will be taken care of when using a
#               main cleaning method.
#               - Exactly like fumbles, this lateral cleaning method will
#                 organize the play into segments (each segment representing a
#                 separate player in possession of the ball within the play
#                 doing whatever. But it will always be from the start of when
#                 they get the ball to the end.)
#                 - Each segment will be cleaned using a main cleaning method,
#                   most will be cleaned using the 'clean_run_plays' method but
#                   it can be whatever.
#                   - Which now that I think about it, I have not thought of
#                     other main cleaning methods to use other than the
#                     'clean_run_plays' method. In most cases, after a player
#                     fumbles, the one who picks it up will run with the ball.
#                     But this is not always the case. They could pass, punt,
#                     throw an interception, anything.
#                     - It might be a good idea to come up with a
#                       'what_does_action_look_like' method. I could send in a
#                       play, or a segment of a play and it will clean the play
#                       based off of what it looks like and return the cleaned
#                       action in a single row dataframe. Or could even just
#                       return the cleaning method itself.
#                 - When all segments are cleaned, the idea is to bring them
#                   all together in a single dataframe and replace that
#                   single play row within the main dataframe of plays.
#          6. I am sure there are more

#             SIDENOTE:
#             - The whole point of identifying 'start_spotting' and
#               'end_spotting' is to place parameters on a single segment of
#               a play.
#               - [start of player possession,
#                  everything in between,
#                  end of player possession]

#     WHERE THE REAL CURVEBALL COMES:
#     - Sometimes, yardage calculated by one player can have an effect on
#       the next. I do not want to brute force this, I feel like it would
#       be chunky and inefficient.
#       - Below is how yardage gained from one can affect another

#     2. Does the ball cross the line of scrimmage?

#        a. The ball remains behind the line of scrimmage:
#          -  I THINK:
#             The initial player who laterals the ball
#             - will be awarded yardage from the
#               line of scrimmage (start_spotting)
#               ->
#               spotting of the lateral (end_spotting).
#          - I THINK:
#            The player who received the lateral (?)
#            - POSSIBLE OUTCOMES
#              1. Receive 0 yardage
#                 line of scrimmage (start_spotting)
#                 OR
#                 spotting of reception (stat_spotting) <- more likely Im guessing
#                 ->
#                 some end spotting behind LOS (end_spotting)
#                 =
#                 0.0
#              2. OR they receive yardage
#                 line of scrimmage (start_spotting)
#                 OR
#                 spotting of reception (stat_spotting) <- more likely Im guessing
#                 ->
#                 some end spotting behind LOS (end_spotting)
#                 =
#                 Distance between (start_spotting -> end_spotting)
#
#        b. The ball lateral behind the line of scrimmage THEN
#           ball crosses line of scrimmage:
#          -  I THINK:
#             The initial player who laterals the ball
#             - will be awarded yardage from the
#               line of scrimmage (start_spotting)
#               ->
#               spotting of the lateral (end_spotting)
#               (UNTIL) ball goes over line of scrimmage in same play
#               = receives 0.0 yards
#          - I THINK:
#            The player who received the lateral
#             - will be awarded yardage from the
#               line of scrimmage (start_spotting)
#               ->
#               some end spotting behind LOS (end_spotting)

#        c. The ball laterals beyond the line of scrimmage THEN
#           remains beyond line of scrimmage:
#          -  I THINK:
#             The initial player who laterals the ball
#             - will be awarded yardage from the
#               line of scrimmage (start_spotting)
#               ->
#               spotting of the lateral (end_spotting).
#          - I THINK:
#            The player who received the lateral
#             - will be awarded yardage from the
#               spotting of reception (stat_spotting) <- more likely Im guessing
#               ->
#               some end spotting behind LOS (end_spotting)

# Goal right now is to display the play and organize the play just like
# it is during the fumble cleaning method.

# CURRENT GOAL:
# 1. display play
# 2. display split play (play split up by sentence)
# 3. display organized grouping of plays
#    - each grouping will become a single row within
#      the return dataframe that will, as a whole,
#      represent the entire play

def helper_clean_lateral_play(df_plays, play_index):

  # print(play_index)

  df_play = df_plays.loc[play_index].copy()

  play_description = df_play['PlayDescription']

  ############
  # REVERSES #
  ############
  # In 'PlayDescription' all information before the "reversed" sentence is not needed.
  # - All information before is stored within 'ReverseDetails' and the remaining is cleaned.
  if play_description.find('REVERSED') != -1:
    play_elements = split_play_description(play_description)
    for i in play_elements:
      if i.find("REVERSED") != -1:
        df_play['ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
        play_description = ". ".join(play_elements[play_elements.index(i) + 1:])
        break

  # Splitting play by sentences
  play_description_split = split_play_description(play_description)

  # # 1. display play
  # play_description_length = "- 1. PLAY DESCRIPTION -"
  # print("-" * len(play_description_length))
  # print(play_description_length)
  # print("-" * len(play_description_length))
  # print(play_description)
  # print()

  # # 2. display split play (play split up by sentence)
  # play_description_split_length = "- 2. PLAY DESCRIPTION SPLIT -"
  # print("-" * len(play_description_split_length))
  # print(play_description_split_length)
  # print("-" * len(play_description_split_length))
  # for i in play_description_split:
  #   print(i)
  # print()

  # 3. display organized grouping of plays

  # ORGANIZING GROUPING OF PLAYS:
  # - REMINDER:
  #   - A list will hold all groupings of individual actions (that being each
  #     player that has possession of the ball from start to finish). I want
  #     each grouping, or element in the list, to have all information needed
  #     to create an individual single dataframe row that will represent the
  #     action individually. These individual actions will eventually come
  #     together as a single dataframe and will represent the play entirely.
  # - PLAN TO ACHIEVE COLLECTING ALL INFORMATION FOR AN INDIVIDUAL ACTION:
  #   - Each element of the list of actions should look something like this,
  #     [start_spotting, everything in between, play description with end_spotting]
  #     - (start_spotting)
  #       - line of scrimmage
  #       - reception of lateral
  #     - (everything in between)
  #       - ? I do not know if I have come across anything that is in between
  #         yet. But I am not ruling it out.
  #     - (play description with end_spotting)
  #       - This will have information on the play itself and will be cleaned
  #         using a main cleaning method.

  #       - Eventually when I merge this idea with fumbles, sometimes actions
  #         that follow will affect the 'end_spotting' of an action.
  #         (e.g. fumble behind LOS, a player on same team eventually crosses
  #               LOS, rewarded initial player who fumbles 0 yardage.)


  # - I am having trouble trying to figure out 'start_spotting' for certain
  #   plays, specifically ones that have to do with laterals or fumbles that
  #   occur behind the line of scrimmage.
  #   - If a player that receives a lateral behind the line of scrimmage and
  #     their run ends behind the line of scrimmage, does he gain -
  #     a. yardage from the reception of the lateral to the end of the carry
  #     b. yardage from the LOS to the end of the carry
  #     c. 0 yardage
  #   - If a player that receives a lateral behind the line of scrimmage and
  #     their run goes beyond the line of scrimmage, does he gain -
  #     a. yardage from the LOS to the end of the carry

  # THINKING ON PAGE:
  # - start_spotting for lateral behind LOS -> end carry beyond LOS
  #   - If the lateral spotting (start_spotting for the next player) is
  #     behind the LOS -> start_spotting will remain as the LOS.

  # - calculating yardage
  #   - Could I mess with the play description to calculate yardage?
  #     - In 'clean_run_plays' method, I can already input start_spotting
  #       and have the yardage calculated from there.
  #       - Would it be beneficial to manipulate the end_spotting in
  #         play descriptions so it calculates from start_spotting -> end_spotting?

  play_split = []

  start_spotting = df_play['FieldPosition']
  # print(start_spotting)

  # Cycling through every action (sentence) within play
  while (play_description_split):

    ###################
    # PRIMARY ACTIONS #
    ###################
    # - Primary actions are sentences that have details about a player rushing
    #   or passing the ball.

    # - Sentence that contains information of an attempt to gain yards
    # for play_pattern in [passer_name_pattern, standard_play_end_pattern]:
    for play_pattern in [standard_play_end_pattern]:
      standard_play = re.findall(play_pattern, play_description_split[0])
      if standard_play:
        # print(standard_play)
        play_split.append([start_spotting])
        play_split[len(play_split) - 1].append(play_description_split[0])
        start_spotting = " ".join(standard_play[0][:2:])
        break

    # - Sentence should be taken care of, onto the next sentence in the play
    play_description_split.pop(0)

  # play_description_grouped_length = "- 3. PLAYDESCRIPTION GROUPED -"
  # print("-" * len(play_description_grouped_length))
  # print(play_description_grouped_length)
  # print("-" * len(play_description_grouped_length))
  # for j in play_split:
  #   print(j)
  # print()


  return

### OFFENSIVE CLEANING METHODS

#### PASS PLAYS

In [ ]:
# PURPOSE:
# - Clean all passing play types
# INPUT PARAMETERS:
# df_plays    - dataframe - NFL plays
# index_start -  integer  - index in the dataframe of NFL plays where the method
#                           will start cleaning in ascending order.
# RETURN:
# df_plays - dataframe - the same input df_plays but with all passing play types cleaned

def clean_pass_plays(df_plays, index_start = None):

  # Adjusting df_plays to start cleaning at a specified index (index_start)
  if index_start != None:
    # Locating all passing type plays (starting from 'index_start')
    df_plays_adjusted = df_plays.loc[index_start:]
    df_pass_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Pass')]
  else:
    # Locating all passing type plays (From entire input dataframe)
    df_pass_plays = df_plays[df_plays['PlayOutcome'].str.contains('Pass')]

  for idx, play in df_pass_plays['PlayDescription'].items():

    # print(idx)
    # print(play)

    ################
    # PLAY DETAILS #
    ################

    # Play Type
    df_plays.loc[idx, 'PlayType'] = 'Pass'

    ############
    # REVERSES #
    ############

    # In 'PlayDescription' all information before the "reversed" sentence is not needed.
    # - All information before the "reversed" sentence is stored within "ReverseDetails"
    if play.find('REVERSED') != -1:
      play_elements = play.split(". ")
      for i in play_elements:
        if i.find('REVERSED') != -1:
          df_plays.at[idx, 'ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
          play = ". ".join(play_elements[play_elements.index(i) + 1:])
          break

    ############################
    # REPORTING IN AS ELIGIBLE #
    ############################

    # I do not think this contains any useful data so I am going to exclude it.
    if play.find('reported in as eligible') != -1:
      play_elements = play.split(". ")
      for i in play_elements:
        if i.find('reported in as eligible') != -1:
          play = ". ".join(play_elements[play_elements.index(i) + 1:])
          break

    ############
    # LATERALS #
    ############
    # - Yardage gained from a lateral.. what would this look like?
    #   - Would the lateral method completely clean that play?
    #     - I don't think so.
    #       - I am trying to figure out whether it would be better to check
    #         for laterals or fumbles first.
    #         - I want to say that it wont matter because fumble and lateral
    #           cleaning methods are using base cleaning methods for the bulk
    #           of the cleaning. They are just organizing the sentences to be
    #           cleaned by the base cleaning methods and calculating yardage..?
    #         - What happens if there is a fumble to a lateral? or a lateral
    #           to a fumble? Will the ordering affect this and clean a play
    #           such as this inaccurately?

    if play.lower().find("lateral") != -1:
      helper_clean_lateral_play(df_plays, idx)
      continue

    ###########
    # FUMBLES #
    ###########
    # - Yardage gained from a fumble.. what would this look like?
    #   - Would the fumble method completele clean that play?
    #     - I think so.
    if play.lower().find("fumble") != -1:
      helper_clean_fumble_play(df_plays, idx)
      continue

    ###########
    # OFFENSE #
    ###########

    # These may have to change in the future
    # - I do not think that the value with the 'end_spotting' will always
    #   be 'play'. I think that in the future, I will need to get more percise
    #   with this.
    #   - I think that the end spotting will actually always be in the description
    #     somewhere. I will have to locate it
    # - I do not think that 'start_spotting' will always be the field position.
    start_spotting = df_plays.loc[idx, 'FieldPosition']
    description_with_end_spotting = play
    df_plays.loc[idx, 'Yardage'] = yardage_between_spottings(df_plays.loc[idx, 'TeamWithPossession'], start_spotting, description_with_end_spotting)

    # I am not giving up on this option of receiving play yardage
    # VVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVV
    # - I think it is the fastest and is accurate for normal plays.

    # action_yardage = re.findall(standard_play_end_pattern, play)
    # if action_yardage:
    #   print(action_yardage)
    #   # End Spot
    #   df_plays.loc[idx, 'EndSpot'] = " ".join(action_yardage[0][:2])
    #   # Yardage
    #   if action_yardage[0][2] == 'no gain':
    #     df_plays.loc[idx, 'Yardage'] = 0
    #   else:
    #     df_plays.loc[idx, 'Yardage'] = action_yardage[0][2]
    # else:
    #   print("No action yardage")

    # Passer
    passer_name = re.findall(passer_name_pattern, play)
    if passer_name:
      # print(passer_name)
      df_plays.loc[idx, 'Passer'] = passer_name[0]

    # Receiver name and passing details
    receiver_name_and_passing_details = re.findall(receiver_pattern, play)
    if receiver_name_and_passing_details:
      # print(receiver_name_and_passing_details)
      df_plays.loc[idx, 'Direction'] = " ".join(receiver_name_and_passing_details[0][:2])
      df_plays.loc[idx, 'Receiver'] = receiver_name_and_passing_details[0][2]

    # Unique situation (offense spikes the ball)
    if play.find('spike') != -1:
      df_plays.loc[idx, 'Direction'] = 'spiked' # Direction?

    #############
    #  DEFENSE  #
    #############

    solo_tackle = re.findall(solo_tackle_pattern, play)
    if solo_tackle:
      if df_plays.loc[idx, 'PlayDescription'].find('pass incomplete') != -1:
        df_plays.loc[idx, 'PassDefendedBy'] = solo_tackle[0]
      else:
        df_plays.loc[idx, 'SoloTackle'] = solo_tackle[0]

    shared_tackle = re.findall(shared_tackle_pattern, play)
    if len(shared_tackle) > 0:
      if df_plays.loc[idx, 'PlayDescription'].find('pass incomplete') != -1:
        df_plays.at[idx, 'PassDefendedBy'] = shared_tackle[0]
      else:
        df_plays.at[idx, 'SharedTackle'] = shared_tackle[0]

    assisted_tackle = re.findall(assisted_tackle_pattern, play)
    if len(assisted_tackle) > 0:
      df_plays.at[idx, 'AssistedTackle'] = assisted_tackle[0][::]

    pressure_by = re.findall(defense_pressure_name_pattern, play)
    if len(pressure_by) > 0:
      df_plays.loc[idx, 'PressureBy'] = pressure_by[0]

    ##############
    #  INJURIES  #
    ##############

    injuries = re.findall(injury_pattern, play)
    if len(injuries) > 0:
      df_plays.at[idx, 'InjuredPlayers'] = injuries

    # print()

  if df_pass_plays.tail(1).index.tolist()[0] == idx:
    return df_plays

#### RUN PLAYS

In [ ]:
# PURPOSE:
# - Clean run play types
# INPUT PARAMETERS:
# df_plays    - dataframe - dataframe of plays
# index_start -  integer  - the starting index of the associated input dataframe
#                           to begin cleaning.
# RETURN:
# df_plays - dataframe - dataframe of plays that now has all useful run play
#                        data accessable and clean.

# NOTE:
# - This method will be used for all actions that involve running with the football.
#   (e.g. fumble recoveries for yardage, fumble recoveries for touchdown, laterals,
#         kickoff returns, punt returns, etc..)

# def clean_run_plays(df_plays, index_start = None):
def clean_run_plays(df_plays, start_spotting = None, index_start = None):

  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start:]
    df_run_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Run')]
  else:
    df_run_plays = df_plays[df_plays['PlayOutcome'].str.contains('Run')]

  # Iterating through every run play within 'df_run_plays'
  for idx, play in df_run_plays['PlayDescription'].items():

    # print(idx)
    # print(play)

    ################
    # Play details #
    ################

    # Play Type
    df_plays.loc[idx, 'PlayType'] = 'Run'

    ############
    # REVERSES #
    ############

    # In 'PlayDescription' all information before the "reversed" sentence is not needed.
    # - All information before is stored within 'ReverseDetails' and the remaining is cleaned.
    if play.find('REVERSED') != -1:
      play_elements = play.split(". ")
      for i in play_elements:
        if i.find("REVERSED") != -1:
          df_plays.at[idx, 'ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
          play = ". ".join(play_elements[play_elements.index(i) + 1:])
          break

    ############################
    # REPORTING IN AS ELIGIBLE #
    ############################

    # I do not think this contains any useful data so I am going to exclude it.
    if play.find('reported in as eligible') != -1:
      play_elements = play.split(". ")
      for i in play_elements:
        if i.find('reported in as eligible') != -1:
          play = ". ".join(play_elements[play_elements.index(i) + 1:])
          break

    ############
    # LATERALS #
    ############
    # - Yardage gained from a lateral.. what would this look like?
    #   - Would the lateral method completely clean that play?
    #     - I think so.

    ###########
    # FUMBLES #
    ###########
    # - Yardage gained from a fumble.. what would this look like?
    #   - Would the fumble method completele clean that play?
    #     - I think so.
    # if play.lower().find("fumble") != -1:
    if play.lower().find("fumble") != -1 or play.find("MUFFS") != -1:
      helper_clean_fumble_play(df_plays, idx)
      if df_run_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      # continue

    #############
    #  OFFENSE  #
    #############

    # Rusher
    rusher_patterns = [rusher_pattern, defensive_takeaway_run_pattern, defensive_takeaway_for_touchdown]
    # Loop through patterns and find the first match
    for pattern in rusher_patterns:
      rusher_name = re.findall(pattern, play)
      if rusher_name:
        # Regular run play
        if rusher_patterns == rusher_pattern:
          # Rusher
          df_plays.loc[idx, 'Rusher'] = rusher_name[0][0]
          # Direction
          df_plays.at[idx, 'Direction'] = " ".join(rusher_name[0][1::]).strip()
        # Defensive takeaway (interception or fumble)
        # - Because punt and kickoff returns follow the same format as
        #   interception or fumble returns for yardage, this will also grab
        #   the 'Returner' name for them.
        else:
          df_plays.loc[idx, 'Rusher'] = rusher_name[0][0]
        break

    # if not rusher_name:
    #   raise ValueError(f"rusher not found at {idx}, \"{play}\"")

    # These may have to change in the future
    # - I do not think that the value with the 'end_spotting' will always
    #   be 'play'. I think that in the future, I will need to get more percise
    #   with this.
    # - I do not think that 'start_spotting' will always be the field position.
    # start_spotting = df_plays.loc[idx, 'FieldPosition']

    if start_spotting == None:
      start_spotting = df_plays.loc[idx, 'FieldPosition']
    description_with_end_spotting = play
    # df_plays.loc[idx, 'Yardage'] = yardage_between_spottings(df_plays, idx, start_spotting, description_with_end_spotting)
    df_plays.loc[idx, 'Yardage'] = yardage_between_spottings(df_plays.loc[idx, 'TeamWithPossession'], start_spotting, description_with_end_spotting)
    # Need to reset for next play
    start_spotting = None

    # YARDAGE FOR HANDOFFS? #
    # - That's what was here in the older version. Need to keep an eye out.

    #############
    #  DEFENSE  #
    #############

    solo_tackle = re.findall(solo_tackle_pattern, play)
    if solo_tackle:
        df_plays.loc[idx, 'SoloTackle'] = solo_tackle[0]

    shared_tackle = re.findall(shared_tackle_pattern, play)
    if len(shared_tackle) > 0:
        df_plays.at[idx, 'SharedTackle'] = shared_tackle[0]

    assisted_tackle = re.findall(assisted_tackle_pattern, play)
    if len(assisted_tackle) > 0:
      df_plays.at[idx, 'AssistedTackle'] = assisted_tackle[0][::]

    ##############
    #  INJURIES  #
    ##############

    injuries = re.findall(injury_pattern, play)
    if len(injuries) > 0:
      df_plays.at[idx, 'InjuredPlayers'] = injuries

    #############
    #  PENALTY  #
    #############

    # print()

    # Return if the last play has been cleaned in 'df_run_plays'
    if df_run_plays.tail(1).index.tolist()[0] == idx:
      return df_plays

####2PT CONVERSIONS

In [ ]:
# I NEED A LARGER SAMPLE SIZE FOR MORE PLAYS
# - I need a sample size that has fumbled plays (if that's possible?)
# - I need a sample size that has interception (if that's possible?)
# - I need a sample size with injuries (as dark as that may sound)

def clean_2pt_conversion_plays(df_plays, index_start = None):

  # Cut 'df_plays' to begin from 'index_start' to the last '2pt conversion' play available in dataframe
  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start]
    df_2pt_conversion_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Conversion', case=False)]
  else:
    df_2pt_conversion_plays = df_plays[df_plays['PlayOutcome'].str.contains('Conversion', case=False)]

  # Iterating through every 2pt conversion play within 'df_2pt_conversion_plays'
  for idx, play in df_2pt_conversion_plays['PlayDescription'].items():

    # print(idx)
    # print(play)

    ############
    # REVERSES #
    ############

    # In 'PlayDescription' all information before the "reversed" sentence is not needed.
    # - All information before the "reversed" sentence is stored within "ReverseDetails"
    if play.find('REVERSED') != -1:
      play_elements = split_play_description(play)
      for i in play_elements:
        if i.find('REVERSED') != -1:
          df_plays.at[idx, 'ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
          play = ". ".join(play_elements[play_elements.index(i) + 1:])
          break

    ###################
    # PASSING ATTEMPT #
    ###################

    pass_2ptc = re.findall(tp_conversion_pass_pattern, play)
    if pass_2ptc:
      # print(pass_2ptc)
      df_plays.loc[idx, 'Passer'] = pass_2ptc[0][0]
      df_plays.loc[idx, 'Receiver'] = pass_2ptc[0][1]
      df_plays.loc[idx, 'PlayType'] = '2PT Conversion Pass'

    ###################
    # RUSHING ATTEMPT #
    ###################

    rush_2ptc = re.findall(tp_conversion_rush_pattern, play)
    if rush_2ptc:
      # print(rush_2ptc)
      df_plays.loc[idx, 'Rusher'] = rush_2ptc[0][0]
      # " ".join(rusher_name[0][1::]).strip()
      # df_plays.loc[idx, 'Direction'] = rush_2ptc[0][1]
      df_plays.loc[idx, 'Direction'] = " ".join(rush_2ptc[0][1::]).strip()
      df_plays.loc[idx, 'PlayType'] = '2PT Conversion Rush'

  return  df_plays

### DEFENSE CLEANING METHODS

#### INTERCEPTIONS

In [ ]:
# PURPOSE:
# - Clean intercepted plays
# INPUT PARAMETERS:
# df_plays    - dataframe - dataframe of plays
# index_start -  integer  - the starting index of the associated input dataframe
#                           to begin cleaning.
# RETURN:
# df_plays - dataframe - dataframe of plays that now has all useful intercepted play
#                        data accessible and clean.

# ROUGH DESGIN
# 1. Narrow dataframe using 'index_start'
#    - This is a recursive method, the narrowing will get smaller and
#      smaller until all 'intercepted' type plays have been cleaned.
# 2. Grab first 'intercepted' play from narrowed dataframe
# 3. Create 2 single row dataframes.
#    a. intended play
#    b. yardage after interception
# 4. Break down play into sentences and clean
#    - Depending on the sentence within the play, will determine which
#      single row dataframe it will go to.
# 5. Combine both dataframes of cleaned data into one dataframe
# 6. Replace old play row with new cleaned multi row
# 7. return clean_interceped_plays( x , y)
#    - x = updated df_plays
#    - y = index directly after the last clean added row

def clean_intercepted_plays(df_plays, index_start = None):

  # Will cut df_plays starting from index_start (narrowing our search space)
  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start:]
    df_intercepted_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Interception')]
  else:
    df_intercepted_plays = df_plays[df_plays['PlayOutcome'].str.contains('Interception')]

  # Exit case (If no more 'Interception' type plays are found)
  if df_intercepted_plays.empty:
    return df_plays

  # Retrieve the index and 'PlayDescription' of the first intercepted play in 'df_intercepted_plays'
  # - Process one play per iteration in the recursive method
  idx = df_intercepted_plays.index[0]
  play = df_plays['PlayDescription'].loc[idx]

  #############
  # VARIABLES #
  #############

  interception_spotting = None

  ############
  # REVERSES #
  ############

  # In 'PlayDescription' all information before the "reversed" sentence is not needed.
  # - All information before the "reversed" sentence is stored within "ReverseDetails"
  if play.find('REVERSED') != -1:
    play_elements = split_play_description(play)
    for i in play_elements:
      if i.find('REVERSED') != -1:
        df_plays.at[idx, 'ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
        play = ". ".join(play_elements[play_elements.index(i) + 1:])
        break

  ###########
  # FUMBLES #
  ###########
  # - I am worried about the types of interception fumbles that can happen that I have yet to see.
  #   - Such as a fumble by the QB then throws an interception
  # - Should I just send the entire play to the fumble helper method and clean it from there?
  #   - What to think about
  #     1. How would this type of play be split?
  #     2. Is this method 'clean_intercepted_plays', able to clean up to the interception
  #        alone? Or does it have to have the run after?
  #        - I think it does.
  if play.lower().find("fumble") != -1:
    helper_clean_fumble_play(df_plays, idx)
    # - Once 'helper_clean_fumble_play' returns a cleaned play I will have to
    #   replace the old uncleaned play with the new cleaned one.
    # - For now, I will just skip over this play.
    if df_intercepted_plays.tail(1).index.tolist()[0] == idx:
      return df_plays
    else:
      return clean_intercepted_plays(df_plays, idx+1)

  # Create 2 single row dataframes.
  # 1. intended play
  df_intended_play = df_plays.loc[idx].copy()
  df_intended_play = pd.DataFrame([df_intended_play], columns=df_plays.columns)
  df_intended_play.reset_index(drop=True, inplace=True)
  df_intended_play['PlayDescription'] = 'nan'
  # 2. yardage after interception
  df_yardage_after_interception = df_plays.loc[idx].copy()
  df_yardage_after_interception = pd.DataFrame([df_yardage_after_interception], columns=df_plays.columns)
  df_yardage_after_interception.reset_index(drop=True, inplace=True)
  df_yardage_after_interception['PlayDescription'] = 'nan'

  # break down play by sentences.
  play_elements = split_play_description(play)

  # Split play elements
  # 1. intended play
  #    - Grab all elements leading up to the sentence containing interception
  #      - Clean using 'clean_pass_plays' method
  # 2. actions after interception
  #    - Grab all elements after sentence containing interception
  #      - Clean using 'clean_run_plays' method
  #      - Clean using 'clean_touchdown_plays' method..?

  # Separating play into
  # 1. intended passing play
  # 2. remaining actions following interception
  for i in play_elements:
    if i.lower().find('intercepted') != -1:
      intended_play_playdescription = ". ".join(play_elements[:play_elements.index(i)+1])
      after_interception_playdescription = ". ".join(play_elements[play_elements.index(i)+1:])
      # print(idx)
      # print(intended_play_playdescription)
      # print(after_interception_playdescription)
      break

  #################
  # INTENDED PLAY #
  #################

  df_intended_play['PlayDescription'] = intended_play_playdescription
  df_intended_play['PlayOutcome'] = 'Pass'
  df_intended_play = clean_pass_plays(df_intended_play)
  df_intended_play['PlayOutcome'] =  df_plays['PlayOutcome'].loc[idx]

  # Intercepted by
  intercepted_by = re.findall(interception_name_pattern, intended_play_playdescription)
  if intercepted_by:
    df_intended_play['InterceptedBy'] = intercepted_by[0][0]
    interception_spotting = intercepted_by[0][1]
    # print(interception_spotting)
    # - During intercepted plays, The intended play portion of the play description is cleaned
    #   by the regular pass cleaning method. A defensive player awarded with a pass defend
    #   during an intercepted play is formatted the exact same as a player awarded a solo
    #   tackle during a completed pass play. I will leverage that here and move the player
    #   to the correct feature ('SoloTackle' -> 'PassDefendedBy')
    if df_intended_play['SoloTackle'].iloc[0] != 'nan':
      df_intended_play.at[0, 'PassDefendedBy'] = (intercepted_by[0][0], df_intended_play['SoloTackle'].iloc[0])
      df_intended_play['SoloTackle'] = 'nan'
    else:
      df_intended_play['PassDefendedBy'] = intercepted_by[0][0]





  #############################################################
  # YARDAGE AFTER INTERCEPTION / TOUCHDOWN AFTER INTERCEPTION #
  #############################################################
  # - I need this to be able to clean everything.
  #   - I need it to be able to clean regular interceptions for yardage (X)
  #   - I need it to be able to clean regular interceptions for yardage and then fumbled (X)
  #   - I need it to be able to clean interceptions resulting in multiple fumbles (X)
  #   - I need it to be able to clean interceptions for touchdowns (X)

  #   - I need it to be able to clean a fumbled interception that is recoverd for a touchdown
  #   - I need this to account for penalties

  # for action in [standard_play_end_pattern]:
  for action in [standard_play_end_pattern, touchdown_pattern]:
    yardage_after_interception = re.findall(action, after_interception_playdescription)
    if yardage_after_interception:
      df_yardage_after_interception['PlayDescription'] = after_interception_playdescription

      # Flipping team with possession when the play transitions from one team with possession to the other.
      if df_yardage_after_interception['TeamWithPossession'].iloc[0] == df_yardage_after_interception['HomeTeam'].iloc[0]:
        df_yardage_after_interception['TeamWithPossession'] = df_yardage_after_interception['AwayTeam'].iloc[0]
      else:
        df_yardage_after_interception['TeamWithPossession'] = df_yardage_after_interception['HomeTeam'].iloc[0]

      # - For yardage gained on this play, I would like to send this job to the
      #   cleaning method for run plays.
      #   - I will need to adjust 3 methods to accomplish this:
      #     1. this method
      #        - I need to add another regular expession
      #          "defensive_takeaway_run_pattern"
      #     2. run method
      #        1. I will need to add another parameter for 'start spotting'
      #           - I can grab the start spotting from the end of the sentence
      #             containing the intercepted information.
      #     3. yardage between spottings
      #        - I might have to adjust this method for touchdowns in the
      #          future. Right now I think it is capable of doing the trick,
      #          but not for touchdowns.

      # # Ideally I would like to send this off to another method.
      # if action == touchdown_after_takeaway_pattern:
      #   df_yardage_after_interception['IsScoringPlay'] = 1
      # # else:
      df_yardage_after_interception['PlayOutcome'] = 'Run'
      df_yardage_after_interception = clean_run_plays(df_yardage_after_interception, interception_spotting)
      df_yardage_after_interception['PlayOutcome'] =  df_plays['PlayOutcome'].loc[idx]

      # The 'clean_run_plays' method will change 'PlayType' so that is why I am
      # putting it down here.
      df_yardage_after_interception['PlayType'] = 'Run After Interception'
      break





  #############################
  # NEW REPLACEMENT DATAFRAME #
  #############################

  # combine both single row dataframes into one
  if df_yardage_after_interception['PlayDescription'].iloc[0] == 'nan':
    df_cleaned_replacement = df_intended_play
  else:
    df_cleaned_replacement = pd.concat([df_intended_play, df_yardage_after_interception], ignore_index=True)

  # Replace old row with new cleaned dataframe
  df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
  df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
  df_plays = pd.concat([df_before_row, df_cleaned_replacement, df_after_row], ignore_index=True)

  # print()
  # If this is the last play in the dataset
  if df_intercepted_plays.tail(1).index.tolist()[0] == idx:
    return df_plays
  else:
    return clean_intercepted_plays(df_plays, idx+len(df_cleaned_replacement))

#### SACKS

In [ ]:
# PURPOSE:
# - Clean sacked plays
# INPUT PARAMETERS:
# df_plays    - dataframe - dataframe of plays
# index_start -  integer  - the starting index of the associated input dataframe
#                           to begin cleaning.
# RETURN:
# df_plays - dataframe - dataframe of plays that now has all useful sacked play
#                        data accessible and clean.

def clean_sacked_plays(df_plays, index_start = None):

  # Will cut df_plays starting from index_start (narrowing our search space)
  if index_start != None:
    df_plays_adjusted = df_plays.iloc[index_start:]
    df_sacked_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Sack')]
  else:
    df_sacked_plays = df_plays[df_plays['PlayOutcome'].str.contains('Sack')]

  for idx, play in df_sacked_plays['PlayDescription'].items():
    # print(idx)
    # print(play)

    ############
    # REVERSES #
    ############

    # In 'PlayDescription' all information before the "reversed" sentence is not needed.
    # - All information before the "reversed" sentence is stored within "ReverseDetails"
    if play.find('REVERSED') != -1:
      play_elements = split_play_description(play)
      for i in play_elements:
        if i.find('REVERSED') != -1:
          df_plays.at[idx, 'ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
          play = ". ".join(play_elements[play_elements.index(i) + 1:])
          break

    ###########
    # FUMBLES #
    ###########
    # - Yardage gained from a fumble.. what would this look like?
    #   - Would the fumble method completele clean that play?
    #     - I think so.
    if play.lower().find("fumble") != -1:
      helper_clean_fumble_play(df_plays, idx)
      if df_sacked_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      # continue


    ###########
    # OFFENSE #
    ###########

    # Sacked Passer
    sacked_passer_name = re.findall(passer_name_pattern, play)
    if sacked_passer_name:
      df_plays.loc[idx, 'Passer'] = sacked_passer_name[0]

    # Sacked Yardage lost
    df_plays.loc[idx, 'Yardage'] = yardage_between_spottings(df_plays.loc[idx, 'TeamWithPossession'],
                                                             df_plays.loc[idx, 'FieldPosition'],
                                                             play)

    ###########
    # DEFENSE #
    ###########

    # Solo sack (One person sacked the passer)
    solo_sack = re.findall(solo_tackle_pattern, play)
    if solo_sack:
      df_plays.loc[idx, 'SackedBy'] = solo_sack[0]
      df_plays.loc[idx, 'SoloTackle'] = solo_sack[0]

    # Split sack (A sack was given to the passer by multiple defenders)
    split_sack = re.findall(split_sack_pattern, play)
    if split_sack:
      df_plays.at[idx, 'SackedBy'] = split_sack[0]
      df_plays.at[idx, 'AssistedTackle'] = split_sack[0]

    ##############
    #  INJURIES  #
    ##############

    #############
    #  PENALTY  #
    #############

    if df_sacked_plays.tail(1).index.tolist()[0] == idx:
      return df_plays

### SPECIAL TEAMS CLEANING METHODS

#### PUNTS

In [ ]:
# PURPOSE:
# - Clean all punt play types

# A punt playtype will be split into 2 or more rows
#   1. The Punt
#      - 'PlayType'
#         - Punt
#      - 'Punter'
#      - 'LongSnapper'
#   2. The Punt Return
#      - 'PlayType'
#         - Punt Return
#      - 'PlayOutcome'
#         - x yard punt return
#         - fair catch
#         - touchback
#         - out of bounds
#         - downed
#      - 'Returner'
#      - 'Receiver'
#      - 'Yardage'
#      - 'TackleBy1'
#      - 'TackleBy2'
#      - 'DownedBy'

def clean_punt_plays(df_plays, index_start = None):

  # Will cut df_plays starting from index_start (narrowing our search space)
  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start:]
    df_punt_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Punt')]
  else:
    df_punt_plays = df_plays[df_plays['PlayOutcome'].str.contains('Punt')]

  if df_punt_plays.empty:
    return df_plays

  # Retrieve the index and 'PlayDescription' of the first punt play in 'df_punt_plays'
  # - Process one play per iteration in the recursive method
  idx = df_punt_plays.index[0]
  play = df_plays['PlayDescription'].loc[idx]

  #############
  # VARIABLES #
  #############

  punt_catch_spotting = None

  ############
  # REVERSES #
  ############

  # In 'PlayDescription' all information before the "reversed" sentence is not needed.
  # - All information before is stored within 'ReverseDetails' and the remaining is cleaned.
  if play.find('REVERSED') != -1:
    play_elements = split_play_description(play)
    for i in play_elements:
      if i.find("REVERSED") != -1:
        # df_play['ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
        df_plays.at[idx, 'ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
        play = ". ".join(play_elements[play_elements.index(i) + 1:])
        break

  ###########
  # FUMBLES #
  ###########
  # - I have yet to see a fumble during a punt. I know that it is possible
  #   and will have to update this method when that time comes.
  # - Fumble returns will be taken care of using 'clean_run_plays'

  # Create 2 single row dataframes.
  # 1. The Punt
  # df_punt = df_play
  df_punt = df_plays.loc[idx].copy()
  df_punt = pd.DataFrame([df_punt], columns=df_plays.columns)
  df_punt.reset_index(drop=True, inplace=True)
  df_punt['PlayDescription'] = 'nan'
  # 2. The Punt Return
  # df_punt_return = df_play
  df_punt_return = df_plays.loc[idx].copy()
  df_punt_return = pd.DataFrame([df_punt_return], columns=df_plays.columns)
  df_punt_return.reset_index(drop=True, inplace=True)
  df_punt_return['PlayDescription'] = 'nan'

  # break down play by sentences.
  play_elements = split_play_description(play)

  # Split play elements
  # 1. punt
  #    - Grab all elements up to the sentence containing punt
  # 2. actions after punt
  #    - Grab all elements after sentence containing punt
  #      - Clean using 'clean_run_plays' method
  #      - Clean using 'clean_touchdown_plays' method..?

  # Separating play into
  # 1. punt play
  # 2. remaining actions following punt
  for i in play_elements:
    if i.lower().find('punts') != -1:
      punt_play_playdescription = ". ".join(play_elements[:play_elements.index(i)+1])
      punt_return_playdescription = ". ".join(play_elements[play_elements.index(i)+1:])
      # print(idx)
      # print(punt_play_playdescription)
      # print(punt_return_playdescription)
      # print()
      break

  ########
  # PUNT #
  ########

  # All data needed for first row in replacement dataframe
  df_punt['PlayDescription'] = punt_play_playdescription
  df_punt['PlayOutcome'] = 'Punt'
  punt = re.findall(punting_pattern, punt_play_playdescription)
  if punt:
    # print(punt)
    punt_catch_spotting = punt[0][2]
    df_punt['PlayType'] = 'Punt'
    df_punt['PlayDescription'] = i
    df_punt['Kicker'] = punt[0][0]
    df_punt['Yardage'] = int(punt[0][1])
    df_punt['LongSnapper'] = punt[0][5]
    # changing feature 'FieldPosition' in df_punt_return
    df_punt_return['FieldPosition'] = punt[0][2]
    # Touchback
    if i.find('Touchback') != -1:
      df_punt['PlayOutcome'] = 'Touchback'
    # Out of bounds
    if i.find('out of bounds') != -1:
      df_punt['PlayOutcome'] = 'out of bounds'
    # Downed by
    if i.find('downed by') != -1:
      df_punt['PlayOutcome'] = 'downed'
      downed_by = re.findall(kick_downed_by_pattern, i)
      df_punt['DownedBy'] = downed_by[0]
    # fair catch
    if i.find('fair catch') != -1:
      df_punt['PlayOutcome'] = 'fair catch'
      fair_catch_by = re.findall(punt_fair_catch_pattern, i)
      df_punt['Returner'] = fair_catch_by[0]

  ######################################
  # PUNT RETURN (Including touchdowns) #
  ######################################

  # All data needed for the second row within replacement dataframe
  # - Second row only needed when there is a punt return for yardage
  # - I think I am going to run into trouble if there is a fumble recovery for yardage
  punt_return_patterns = [standard_play_end_pattern, touchdown_pattern, muffed_catch_pattern]
  for return_pattern in punt_return_patterns:
    punt_return = re.findall(return_pattern, punt_return_playdescription)
    if punt_return:
      df_punt_return['PlayDescription'] = punt_return_playdescription
      df_punt_return['PlayOutcome'] = 'Run'

      # Change field position to point of catch..? <- is this necessary or redundant? What are the cons?
      # - The reason I was forced to change this is for fumbles during punt returns.
      #   - Is this the best possible option? Is it a smart one?
      # - If I do this here, I will have to be consistant and do this for all other turnover type plays
      #   such as kickoffs.

      df_punt_return['FieldPosition'] = punt_catch_spotting
      # Change team with possession on punt returns to the team that is returning the ball
      if df_punt['TeamWithPossession'].iloc[0] == df_punt['HomeTeam'].iloc[0]:
        df_punt_return.loc[0, 'TeamWithPossession'] = df_punt['AwayTeam'].iloc[0]
      else:
        df_punt_return.loc[0, 'TeamWithPossession'] = df_punt['HomeTeam'].iloc[0]
      df_punt_return = clean_run_plays(df_punt_return, punt_catch_spotting)
      df_punt_return['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]
      df_punt_return['PlayType'] = 'Punt Return'
      df_punt_return['Returner'] = df_punt_return['Rusher']
      df_punt_return['Rusher'] = 'nan'
      break

  #############
  #  PENALTY  #
  #############





  #############################
  # NEW REPLACEMENT DATAFRAME #
  #############################

  if df_punt_return['PlayDescription'].iloc[0] == 'nan':
    df_replacement_rows = df_punt
  elif df_punt['PlayDescription'].iloc[0] == 'nan': # Will happen during fumbled punt returns.
    df_replacement_rows = df_punt_return
  else:
    df_replacement_rows = pd.concat([df_punt, df_punt_return], ignore_index=True)

  df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
  df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
  df_plays = pd.concat([df_before_row, df_replacement_rows, df_after_row], ignore_index=True)

  if df_punt_plays.tail(1).index.tolist()[0] == idx:
    return df_plays
  else:
    return clean_punt_plays(df_plays, idx+len(df_replacement_rows))

#### KICKOFF

In [ ]:
# A kickoff playtype will be split into 1 or more rows

# I need to figure out an onside kick (recovered by kicking team)
# I need to figure out fumbled kickoff returns
# I need to figure out returns for a touchdown
# injuries?

# Method can mirror punts method.

def clean_kickoff_plays(df_plays, index_start = None):

  # Will cut df_plays starting from index_start (narrowing our search space)
  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start:]
    df_kickoff_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Kickoff', case=False)]
  else:
    df_kickoff_plays = df_plays[df_plays['PlayOutcome'].str.contains('Kickoff', case=False)]

  # exit case
  if df_kickoff_plays.empty:
    return df_plays

  # Retrieve the index and 'PlayDescription' of the first kickoff play in 'df_kickoff_plays'
  # - Process one play per iteration in the recursive method
  idx = df_kickoff_plays.index[0]
  play = df_plays['PlayDescription'].loc[idx]

  #############
  # VARIABLES #
  #############

  kickoff_catch_spotting = None

  ############
  # REVERSES #
  ############

  # In 'PlayDescription' all information before the "reversed" sentence is not needed.
  # - All information before is stored within 'ReverseDetails' and the remaining is cleaned.
  if play.find('REVERSED') != -1:
    play_elements = split_play_description(play)
    for i in play_elements:
      if i.find("REVERSED") != -1:
        # df_play['ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
        df_plays.at[idx, 'ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
        play = ". ".join(play_elements[play_elements.index(i) + 1:])
        break

  ###########
  # FUMBLES #
  ###########
  # - Will be taken care of using 'clean_run_plays'

  # Create 2 single row dataframes.
  # 1. The Kickoff
  df_kickoff = df_plays.loc[idx].copy()
  df_kickoff = pd.DataFrame([df_kickoff], columns=df_plays.columns)
  df_kickoff.reset_index(drop=True, inplace=True)
  df_kickoff['PlayDescription'] = 'nan'
  # 2. The Kickoff Return
  df_kickoff_return = df_plays.loc[idx].copy()
  df_kickoff_return = pd.DataFrame([df_kickoff_return], columns=df_plays.columns)
  df_kickoff_return.reset_index(drop=True, inplace=True)
  df_kickoff_return['PlayDescription'] = 'nan'

  # break down play by sentences.
  play_elements = split_play_description(play)

  # Split play elements
  # 1. kickoff
  #    - Grab all elements up to the sentence containing kickoff
  # 2. actions after kickoff
  #    - Grab all elements after sentence containing kickoff
  #      - Clean using 'clean_run_plays' method
  #      - Clean using 'clean_touchdown_plays' method..?

  # Separating play into
  # 1. punt play
  # 2. remaining actions following punt
  for i in play_elements:
    if i.lower().find('kicks') != -1:
      kickoff_play_playdescription = ". ".join(play_elements[:play_elements.index(i)+1])
      kickoff_return_playdescription = ". ".join(play_elements[play_elements.index(i)+1:])
      if kickoff_return_playdescription.find("(didn't try to advance)") != -1:
        kickoff_return_playdescription = kickoff_return_playdescription.replace("(didn't try to advance) ", "")
      # print(idx)
      # print(kickoff_play_playdescription)
      # if len(kickoff_return_playdescription) > 0:
      #   print(idx)
      #   print(kickoff_play_playdescription)
      #   print(kickoff_return_playdescription)
      #   print()
      break

  ###########
  # KICKOFF #
  ###########

  # All data needed for first row in replacement dataframe
  df_kickoff['PlayDescription'] = kickoff_play_playdescription
  df_kickoff['PlayOutcome'] = 'Kickoff'
  kickoff = re.findall(kickoff_pattern, kickoff_play_playdescription)
  if kickoff:
    # print(kickoff)
    kickoff_catch_spotting = kickoff[0][3]
    # Change team with possession on kickoff to the team that is kicking
    if df_kickoff['TeamWithPossession'].iloc[0] == df_kickoff['HomeTeam'].iloc[0]:
      df_kickoff_return.loc[0, 'TeamWithPossession'] = df_kickoff['AwayTeam'].iloc[0]
    else:
      df_kickoff_return.loc[0, 'TeamWithPossession'] = df_kickoff['HomeTeam'].iloc[0]
    df_kickoff['Kicker'] = kickoff[0][0]
    # df_kickoff['Yardage'] = int(kickoff[0][1]) <--- use helper method
    if kickoff_play_playdescription.find('Touchback') != -1:
      df_kickoff['PlayOutcome'] = 'Touchback'
    # I need to figure out what the difference will be when the kicking team recovers
    if kickoff_play_playdescription.find('onside') != -1:
      df_kickoff['PlayOutcome'] = 'onside'
      downed_by = re.findall(kick_downed_by_pattern, i)
      if downed_by:
        df_kickoff['DownedBy'] = downed_by[0]

  #########################################
  # KICKOFF RETURN (Including touchdowns) #
  #########################################



  # 1. might have to create new regular expression for kickoff return patterns
  # 2. might have to adjust yardage between spottings method to accomidate for
  #    kickoff yardage
  #    - Does yardage between spotting handle 'end zone' as an input..?



  # All data needed for the second row within replacement dataframe
  # - Second row only needed when there is a kickoff return for yardage
  kickoff_return_patterns = [standard_play_end_pattern]
  for return_pattern in kickoff_return_patterns:
    kickoff_return = re.findall(return_pattern, kickoff_return_playdescription)
    if kickoff_return:
      # print(kickoff_return)
      df_kickoff_return['PlayDescription'] = kickoff_return_playdescription
      df_kickoff_return['PlayOutcome'] = 'Run'
      # Change team with possession on kickoff returns to the team that is
      # returning the ball.
      if df_kickoff['TeamWithPossession'].iloc[0] == df_kickoff['HomeTeam'].iloc[0]:
        df_kickoff_return.loc[0, 'TeamWithPossession'] = df_kickoff['AwayTeam'].iloc[0]
      else:
        df_kickoff_return.loc[0, 'TeamWithPossession'] = df_kickoff['HomeTeam'].iloc[0]
      df_kickoff_return = clean_run_plays(df_kickoff_return, kickoff_catch_spotting)
      df_kickoff_return['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]
      df_kickoff_return['PlayType'] = 'Punt Return'
      df_kickoff_return['Returner'] = df_kickoff_return['Rusher']
      df_kickoff_return['Rusher'] = 'nan'
      break

  #############
  #  PENALTY  #
  #############





  #############################
  # NEW REPLACEMENT DATAFRAME #
  #############################

  if df_kickoff_return['PlayDescription'].iloc[0] == 'nan':
    df_replacement_rows = df_kickoff
  elif df_kickoff['PlayDescription'].iloc[0] == 'nan': # Will happen during fumbled punt returns.
    df_replacement_rows = df_kickoff_return
  else:
    df_replacement_rows = pd.concat([df_kickoff, df_kickoff_return], ignore_index=True)

  df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
  df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
  df_plays = pd.concat([df_before_row, df_replacement_rows, df_after_row], ignore_index=True)

  if df_kickoff_plays.tail(1).index.tolist()[0] == idx:
    return df_plays
  else:
    return clean_kickoff_plays(df_plays, idx+len(df_replacement_rows))

### SCORING CLEANING METHODS

#### TOUCHDOWNS

In [ ]:
def clean_touchdown_plays(df_plays, index_start=None):

  # Cut 'df_plays' to begin from 'index_start' to the last touchdown play available in dataframe
  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start:]
    df_touchdown_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Touchdown')]
  else:
    df_touchdown_plays = df_plays[df_plays['PlayOutcome'].str.contains('Touchdown')]

  # Iterating through every touchdown play within 'df_touchdown_plays'
  for idx, play in df_touchdown_plays['PlayDescription'].items():

    ##########################
    # INTERCEPTED TOUCHDOWNS #
    ##########################

    # Still need to clean intercepted play types

    if play.find("INTERCEPTED") != -1:

      # For intercepted plays returned for a touchdown, the raw data states that
      # the defensive team has possession of the ball for the entirety of the
      # drive. This inital block of code corrects that error.

      # Flipping all drives within play to have the offensive team as having possesion
      if df_plays['TeamWithPossession'].iloc[idx] == df_plays['HomeTeam'].iloc[idx]:
        df_plays.loc[idx, 'TeamWithPossession'] = df_plays.loc[idx, 'AwayTeam']
      else:
        df_plays.loc[idx, 'TeamWithPossession'] = df_plays.loc[idx, 'HomeTeam']

      # List of all unique features to drive
      matching_features = ['Season', 'Week', 'HomeTeam', 'Quarter', 'DriveNumber']

      # Dataframe of all rows with matching features within df_plays
      df_1 = df_plays.loc[
          (df_plays[matching_features].values == df_plays[matching_features].iloc[idx].values).all(axis=1)
          ]

      # Change all rows 'TeamWithPossession' feature to offensive team.
      for i in df_1.index:
        df_plays.loc[i, 'TeamWithPossession'] = df_plays.loc[idx, 'TeamWithPossession']

      # creating a copy of the incercepted touchdown play and cleaning the copy
      intercepted_touchdown_row = df_plays.loc[idx].copy()
      intercepted_touchdown_row['PlayOutcome'] = 'Interception'
      intercepted_touchdown_row['IsScoringPlay'] = 1 # This will only be the value for the team that threw the interception
      intercepted_touchdown_row = pd.DataFrame([intercepted_touchdown_row], columns=df_plays.columns)
      intercepted_touchdown_row.reset_index(drop=True, inplace=True)

      # REMINDER: This single play is separated into multiple actions (play will be represented with multiple rows)
      intercepted_touchdown_row = clean_intercepted_plays(intercepted_touchdown_row)
      intercepted_touchdown_row['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, intercepted_touchdown_row, df_after_row], ignore_index=True)

      # Recursion to update 'df_plays'
      if df_touchdown_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_touchdown_plays(df_plays, idx+len(intercepted_touchdown_row))

    #####################################
    # SACKED FUMBLE RECOVERY TOUCHDOWNS #
    #####################################

    ######################
    # PASSING TOUCHDOWNS #
    ######################

    # If a play has a passer throwing the ball, I am assuming it is a passing play
    passing_play = re.findall(passer_name_pattern, play)
    # - I need to adjust this in the future. I will take out:
    #   1. sack check
    #   2. interception check
    if len(passing_play) > 0 and play.find("sacked") == -1 and play.find("INTERCEPTED") == -1:
      # print(idx)
      # print(play)

      # creating a copy of the passing touchdown play row and cleaning the copy
      passing_touchdown_row = df_plays.loc[idx].copy()
      passing_touchdown_row['PlayType'] = 'Pass'
      passing_touchdown_row['PlayOutcome'] = 'Pass'
      passing_touchdown_row['IsScoringPlay'] = 1
      passing_touchdown_row = pd.DataFrame([passing_touchdown_row], columns=df_plays.columns)
      passing_touchdown_row = clean_pass_plays(passing_touchdown_row)
      passing_touchdown_row['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, passing_touchdown_row, df_after_row], ignore_index=True)

      # print()

      # Recursion to update 'df_plays'
      if df_touchdown_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_touchdown_plays(df_plays, idx+len(passing_touchdown_row))

    ######################
    # RUSHING TOUCHDOWNS #
    ######################

    rushing_play = re.findall(rusher_pattern, play)
    if rushing_play:

      # print(idx)
      # print(play)

      # creating a copy of the rushing touchdown play row and cleaning the copy
      rushing_touchdown_row = df_plays.loc[idx].copy()
      rushing_touchdown_row['PlayType'] = 'Run'
      rushing_touchdown_row['PlayOutcome'] = 'Run'
      rushing_touchdown_row['IsScoringPlay'] = 1
      rushing_touchdown_row = pd.DataFrame([rushing_touchdown_row], columns=df_plays.columns)
      rushing_touchdown_row = clean_run_plays(rushing_touchdown_row)
      rushing_touchdown_row['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, rushing_touchdown_row, df_after_row], ignore_index=True)

      # print()

      # Recursion to update 'df_plays'
      if df_touchdown_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_touchdown_plays(df_plays, idx+len(rushing_touchdown_row))

    ##########################
    # PUNT RETURN TOUCHDOWNS #
    ##########################

    punt_play = re.findall(punting_pattern, play)
    if punt_play:

      # print(idx)
      # print(play)

      # creating a copy of the punt touchdown play and cleaning the copy
      punt_touchdown_row = df_plays.loc[idx].copy()
      punt_touchdown_row['PlayOutcome'] = 'Punt'
      punt_touchdown_row['IsScoringPlay'] = 1 # This will only be the value for the team that punted the ball
      punt_touchdown_row = pd.DataFrame([punt_touchdown_row], columns=df_plays.columns)
      punt_touchdown_row.reset_index(drop=True, inplace=True)

      # REMINDER: This single play is separated into multiple actions (play will be represented with multiple rows)
      punt_touchdown_row = clean_punt_plays(punt_touchdown_row)
      punt_touchdown_row['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, punt_touchdown_row, df_after_row], ignore_index=True)

      # print()

      # Recursion to update 'df_plays'
      if df_touchdown_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_touchdown_plays(df_plays, idx+len(punt_touchdown_row))

    #################################
    # BLOCKED FIELD GOAL TOUCHDOWNS #
    #################################

  return df_plays

#### FIELD GOALS

In [ ]:
# I need an example of when a player returns the field goal for yardage
# I need a larger sample size for "Blocked" field goals
# I need to figure out what to do if someone fumbles a recovery
# I need to figure out what to do on a trick play (e.i. holder runs out with the ball)
# - INCOMPLETE. NEED LARGER SAMPLE SIZE

def clean_field_goal_plays(df_plays, index_start = None):

  # Adjusting df_plays to start cleaning at a specified index (index_start)
  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start:]
    # Locating all field goal plays within dataframe
    df_field_goal_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Field Goal')]
  else:
    # Locating all field goal plays within dataframe
    df_field_goal_plays = df_plays[df_plays['PlayOutcome'].str.contains('Field Goal')]

  for idx, play in df_field_goal_plays['PlayDescription'].items():

    # print(idx)
    # print(play)

    ###################
    # EXTRA PLAY DATA #
    ###################

    #########################
    # FIELD GOAL SITUATIONS #
    #########################

    field_goal = re.findall(field_goal_pattern, play)
    if field_goal:
      # print(field_goal)
      df_plays.loc[idx, 'PlayType'] = 'Field Goal'
      df_plays.loc[idx, 'Kicker'] = field_goal[0][0]
      df_plays.loc[idx, 'Yardage'] = int(field_goal[0][1])
      # Element will only fill if the field goal was no good
      if field_goal[0][2] != '':
        df_plays.loc[idx, 'Direction'] = field_goal[0][2]
      df_plays.loc[idx, 'LongSnapper'] = field_goal[0][3]
      df_plays.loc[idx, 'Holder'] = field_goal[0][4]

    ######################
    # FIELD GOAL BLOCKED #
    ######################

    field_goal_blocked = re.findall(field_goal_blocked_pattern, play)
    if field_goal_blocked:
      # print(idx)
      # print(play)
      # print(field_goal_blocked)

      # Because of the potential recovery for yardage after a blocked field
      # goal, I need 2 dataframes:
      # 1. intended field goal attempt (containing blocked details)
      df_blocked_fg = df_plays.loc[idx].copy()
      df_blocked_fg = pd.DataFrame([df_blocked_fg], columns=df_plays.columns)
      df_blocked_fg.reset_index(drop=True, inplace=True)
      df_blocked_fg['PlayDescription'] = 'nan'
      # 2. yardage after recovery
      df_yardage_after_recovery = df_plays.loc[idx].copy()
      df_yardage_after_recovery = pd.DataFrame([df_yardage_after_recovery], columns=df_plays.columns)
      df_yardage_after_recovery.reset_index(drop=True, inplace=True)
      df_yardage_after_recovery['PlayDescription'] = 'nan'

      # I need to separate the play description into 2 groups:
      # 1. All actions involving the field goal and field goal block
      #    - for df_blocked_fg
      # 2. All actions following the field goal block recovery for yardage
      #    - for df_yardage_after_recovery
      play_elements = split_play_description(play)
      for i in play_elements:
        if i.lower().find('blocked') != -1:
          blocked_fg_playdescription = ". ".join(play_elements[:play_elements.index(i)+1])
          yardage_after_recovery_playdescription = ". ".join(play_elements[play_elements.index(i)+1:])
          # print(blocked_fg_playdescription)
          # print(yardage_after_recovery_playdescription)

      #####################
      # BOCKED FIELD GOAL #
      #####################

      df_blocked_fg['PlayDescription'] = blocked_fg_playdescription
      df_blocked_fg['PlayType'] = 'Field Goal'
      blocked_fg_data = re.findall(field_goal_blocked_pattern, blocked_fg_playdescription)
      df_blocked_fg['Kicker'] = blocked_fg_data[0][0]
      df_blocked_fg['Yardage'] = int(blocked_fg_data[0][1])
      df_blocked_fg['BlockedBy'] = blocked_fg_data[0][2]
      df_blocked_fg['LongSnapper'] = blocked_fg_data[0][3]
      df_blocked_fg['Holder'] = blocked_fg_data[0][4]

      ###################################
      # FIELD GOAL RECOVERY FOR YARDAGE #
      ###################################

      # If there was a recovery for yardage
      if re.findall(standard_play_end_pattern, yardage_after_recovery_playdescription):
        df_yardage_after_recovery['PlayDescription'] = yardage_after_recovery_playdescription
        df_yardage_after_recovery['PlayOutcome'] = 'Run'
        # blocked_fg_data[0][5] - team of the player that recovered the ball
        df_yardage_after_recovery['TeamWithPossession'] = blocked_fg_data[0][5]
        # blocked_fg_data[0][7] - recovery spotting of player who recovered ball
        df_yardage_after_recovery = clean_run_plays(df_yardage_after_recovery, blocked_fg_data[0][7])
        df_yardage_after_recovery['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]
        df_yardage_after_recovery['PlayType'] = 'Field Goal Return'

      #################
      # NEW DATAFRAME #
      #################
      # - If there was a recovery for yardage, there will be multiple rows
      #   within the dataframe of plays that represent the blocked field goal
      #   along with actions that followed.

      if df_yardage_after_recovery['PlayDescription'].iloc[0] == 'nan':
        df_replacement_rows = df_blocked_fg
      else:
        df_replacement_rows = pd.concat([df_blocked_fg, df_yardage_after_recovery], ignore_index=True)

      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, df_replacement_rows, df_after_row], ignore_index=True)

      if df_field_goal_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_field_goal_plays(df_plays, idx+len(df_replacement_rows))

      # print()

  return df_plays

#### EXTRA POINTS

In [ ]:
def clean_extra_point_plays(df_plays, index_start = None):

  # Adjusting df_plays to start cleaning at a specified index (index_start)
  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start:]
    # Locating all field goal plays within dataframe
    df_field_goal_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Extra Point')]
  else:
    # Locating all field goal plays within dataframe
    df_field_goal_plays = df_plays[df_plays['PlayOutcome'].str.contains('Extra Point')]

  for idx, play in df_field_goal_plays['PlayDescription'].items():

    # print(idx)
    # print(play)

    ###################
    # EXTRA PLAY DATA #
    ###################


    ##########################
    # EXTRA POINT SITUATIONS #
    ##########################

    extra_point = re.findall(extra_point_pattern, play)
    if extra_point:
      # print(extra_point)
      df_plays.loc[idx, 'PlayType'] = 'Extra Point'
      df_plays.loc[idx, 'Kicker'] = extra_point[0][0]
      if extra_point[0][1] != '':
        df_plays.loc[idx, 'Direction'] = extra_point[0][1]
      df_plays.loc[idx, 'LongSnapper'] = extra_point[0][2]
      df_plays.loc[idx, 'Holder'] = extra_point[0][2]

    # print()

  return df_plays

### OTHER CLEANING METHODS

#### FUMBLES

In [ ]:
# What about punt returns?

def clean_fumble_plays(df_plays, index_start=None):

  # Cut 'df_plays' to begin from 'index_start' to the last penalty play available in dataframe
  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start:]
    df_fumble_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('fumble', case=False)]
  else:
    df_fumble_plays = df_plays[df_plays['PlayOutcome'].str.contains('fumble', case=False)]

  for idx, play in df_fumble_plays['PlayDescription'].items():

    ############
    # REVERSES #
    ############

    # In 'PlayDescription' all information before the "reversed" sentence is not needed.
    # - All information before is stored within 'ReverseDetails' and the remaining is cleaned.
    if play.find('REVERSED') != -1:
      play_elements = split_play_description(play)
      for i in play_elements:
        if i.find("REVERSED") != -1:
          df_plays.at[idx, 'ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
          play = ". ".join(play_elements[play_elements.index(i) + 1:])
          break

    ############################
    # REPORTING IN AS ELIGIBLE #
    ############################

    # I do not think this contains any useful data so I am going to exclude it.
    if play.find('reported in as eligible') != -1:
      play_elements = split_play_description(play)
      for i in play_elements:
        if i.find('reported in as eligible') != -1:
          play = ". ".join(play_elements[play_elements.index(i) + 1:])
          break

    initial_action = split_play_description(play)[0]

    # print(idx)
    # print(play)
    # print(initial_action)

    ##################
    # PASSING FUMBLE #
    ##################

    fumble_pass = re.findall(receiver_pattern, initial_action)
    if fumble_pass:

      # creating a copy of the passing fumbled play row and cleaning the copy
      passing_fumble_row = df_plays.loc[idx].copy()
      passing_fumble_row['PlayOutcome'] = 'Pass'
      passing_fumble_row = pd.DataFrame([passing_fumble_row], columns=df_plays.columns)
      passing_fumble_row = clean_pass_plays(passing_fumble_row)

      # Record whether the pass was complete or incomplete.
      if play.find('pass incomplete') != -1:
        passing_fumble_row['PlayOutcome'] = f"{df_plays['PlayOutcome'].loc[idx]} (I)"
      else:
        passing_fumble_row['PlayOutcome'] = f"{df_plays['PlayOutcome'].loc[idx]} (C)"

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, passing_fumble_row, df_after_row], ignore_index=True)

      # Recursion to update 'df_plays'
      if df_fumble_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_fumble_plays(df_plays, idx+len(passing_fumble_row))

    ##################
    # RUSHING FUMBLE #
    ##################

    fumble_rush = re.findall(rusher_pattern, initial_action)
    fumble_aborted = initial_action.find('Aborted')
    if fumble_rush or fumble_aborted != -1:

      # creating a copy of the rushing fumbled play row and cleaning the copy
      rushing_fumble_row = df_plays.loc[idx].copy()
      rushing_fumble_row['PlayOutcome'] = 'Run'
      rushing_fumble_row = pd.DataFrame([rushing_fumble_row], columns=df_plays.columns)
      rushing_fumble_row = clean_run_plays(rushing_fumble_row)
      rushing_fumble_row['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, rushing_fumble_row, df_after_row], ignore_index=True)

      # Recursion to update 'df_plays'
      if df_fumble_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_fumble_plays(df_plays, idx+len(rushing_fumble_row))

    #################
    # SACKED FUMBLE #
    #################

    if initial_action.find('sacked') != -1:

      # creating a copy of the sacked fumble play row and cleaning the copy
      sacked_fumble_row = df_plays.loc[idx].copy()
      sacked_fumble_row['PlayOutcome'] = 'Sack'
      sacked_fumble_row = pd.DataFrame([sacked_fumble_row], columns=df_plays.columns)
      sacked_fumble_row = clean_sacked_plays(sacked_fumble_row)
      sacked_fumble_row['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, sacked_fumble_row, df_after_row], ignore_index=True)

      # Recursion to update 'df_plays'
      if df_fumble_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_fumble_plays(df_plays, idx+len(sacked_fumble_row))

    ##################
    # KICKOFF FUMBLE #
    ##################

    kickoff_fumble = re.findall(kickoff_pattern, initial_action)
    if kickoff_fumble:

      # creating a copy of the passing fumbled play row and cleaning the copy
      kickoff_fumble_row = df_plays.loc[idx].copy()
      kickoff_fumble_row['PlayOutcome'] = 'kickoff'
      kickoff_fumble_row = pd.DataFrame([kickoff_fumble_row], columns=df_plays.columns)
      kickoff_fumble_row = clean_kickoff_plays(kickoff_fumble_row)
      kickoff_fumble_row['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, kickoff_fumble_row, df_after_row], ignore_index=True)

      # Recursion to update 'df_plays'
      if df_fumble_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_fumble_plays(df_plays, idx+len(kickoff_fumble_row))

    ###############
    # PUNT FUMBLE #
    ###############
    # - Need to figure out what to do for "muffed" catches

    punt_fumble = re.findall(punting_pattern, initial_action)
    if punt_fumble:

      # creating a copy of the fumbled play row and cleaning the copy
      punt_fumble_row = df_plays.loc[idx].copy()
      punt_fumble_row['PlayOutcome'] = 'Punt'
      punt_fumble_row = pd.DataFrame([punt_fumble_row], columns=df_plays.columns)
      punt_fumble_row = clean_punt_plays(punt_fumble_row)
      punt_fumble_row['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, punt_fumble_row, df_after_row], ignore_index=True)

      # Recursion to update 'df_plays'
      if df_fumble_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_fumble_plays(df_plays, idx+len(punt_fumble_row))

    # print()

  return df_plays

#### PENALTIES

In [ ]:
# Will work on after fumbles.

#### TURNOVER ON DOWNS

In [ ]:
# Looks like either a pass / run / sack play

def clean_turnover_on_downs_plays(df_plays, index_start=None):

  # Cut 'df_plays' to begin from 'index_start' to the last penalty play available in dataframe
  if index_start != None:
    df_plays_adjusted = df_plays.loc[index_start:]
    df_turnover_on_downs_plays = df_plays_adjusted[df_plays_adjusted['PlayOutcome'].str.contains('Turnover on Downs', case=False)]
  else:
    df_turnover_on_downs_plays = df_plays[df_plays['PlayOutcome'].str.contains('Turnover on Downs', case=False)]

  # Iterating through every penalty play within 'df_turnover_on_downs_plays'
  for idx, play in df_turnover_on_downs_plays['PlayDescription'].items():

    ############
    # REVERSES #
    ############

    # In 'PlayDescription' all information before the "reversed" sentence is not needed.
    # - All information before is stored within 'ReverseDetails' and the remaining is cleaned.
    if play.find('REVERSED') != -1:
      play_elements = split_play_description(play)
      for i in play_elements:
        if i.find("REVERSED") != -1:
          df_plays.at[idx, 'ReverseDetails'] = play_elements[:play_elements.index(i) + 1]
          play = ". ".join(play_elements[play_elements.index(i) + 1:])
          break

    ##############################
    # TURNOVER ON DOWNS (SACKED) #
    ##############################

    if play.find("sacked") != -1:

      sacked_turnover_on_downs = df_plays.loc[idx].copy()
      sacked_turnover_on_downs['PlayOutcome'] = 'Sack'
      sacked_turnover_on_downs = pd.DataFrame([sacked_turnover_on_downs], columns=df_plays.columns)
      sacked_turnover_on_downs.reset_index(drop=True, inplace=True)
      sacked_turnover_on_downs = clean_sacked_plays(sacked_turnover_on_downs)
      sacked_turnover_on_downs['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, sacked_turnover_on_downs, df_after_row], ignore_index=True)

      # Recursion to update 'df_plays'
      if df_turnover_on_downs_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_turnover_on_downs_plays(df_plays, idx+len(sacked_turnover_on_downs))

    ############################
    # TURNOVER ON DOWNS (PASS) #
    ############################

    passing_play = re.findall(passer_name_pattern, play)
    if passing_play:

      passing_turnover_on_downs = df_plays.loc[idx].copy()
      passing_turnover_on_downs['PlayOutcome'] = 'Pass'
      passing_turnover_on_downs = pd.DataFrame([passing_turnover_on_downs], columns=df_plays.columns)
      passing_turnover_on_downs = clean_pass_plays(passing_turnover_on_downs)

      # Record whether the pass was complete or incomplete.
      if play.find('pass incomplete') != -1:
        passing_turnover_on_downs['PlayOutcome'] = f"{df_plays['PlayOutcome'].loc[idx]} (I)"
      else:
        passing_turnover_on_downs['PlayOutcome'] = f"{df_plays['PlayOutcome'].loc[idx]} (C)"

      # Replacing old row with cleaned row
      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, passing_turnover_on_downs, df_after_row], ignore_index=True)

      # Recursion to update 'df_plays'
      if df_turnover_on_downs_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_turnover_on_downs_plays(df_plays, idx+len(passing_turnover_on_downs))

    ############################
    # TURNOVER ON DOWNS (RUSH) #
    ############################

    rushing_play = re.findall(rusher_pattern, play)
    if len(rushing_play) > 0:

      rushing_turnover_on_downs = df_plays.loc[idx].copy()
      rushing_turnover_on_downs['PlayOutcome'] = 'Run'
      rushing_turnover_on_downs = pd.DataFrame([rushing_turnover_on_downs], columns=df_plays.columns)
      rushing_turnover_on_downs = clean_run_plays(rushing_turnover_on_downs)
      rushing_turnover_on_downs['PlayOutcome'] = df_plays['PlayOutcome'].loc[idx]

      df_before_row = df_plays.iloc[:df_plays.index.tolist().index(idx)]
      df_after_row = df_plays.iloc[df_plays.index.tolist().index(idx)+1:]
      df_plays = pd.concat([df_before_row, rushing_turnover_on_downs, df_after_row], ignore_index=True)

      if df_turnover_on_downs_plays.tail(1).index.tolist()[0] == idx:
        return df_plays
      else:
        return clean_turnover_on_downs_plays(df_plays, idx+len(rushing_turnover_on_downs))

## 5. PIPELINE MAIN METHOD

In [ ]:
# PURPOSE:
# - Accept a dataframe of nfl plays (formatted by NFL_Scrapers) and
#   return a cleaned dataframe of those plays.
# INPUT PARAMETERS:
# df_all_plays         - dataframe - all plays in raw form from NFL_Scraper that user
#                                    would like to clean.
# OUTPUT:
# df_all_plays_cleaned - dataframe - all plays from 'df_all_plays' cleaned and data
#                                    dispersed into individual new features.

# CURRENT DESIGN PLAN:
# 1. Use uniquely designed methods for each play type to clean within dataframe
#    - (e.g. pass, run, touchdown, punt, sack, ... )
# 2. Repeat until all plays within dataframe have been cleaned.
#   NOTE:
#   - It is important to fully clean a play type before moving to the next
#      because sometimes cleaning could involve adding a new row to the dataframe,
#      causing a reset to the dataframes indexing.
#      - If we were to separate all play types from the beginning, the indexes
#        could shift around causing, for example, an index that might originally
#        point to a run play to now instead point at a pass play.

def clean_dataframe_of_plays(df_all_plays):

  # Return Dataframe
  df_all_plays_cleaned = df_all_plays.copy()

  ################################
  # RAW DATA COLUMN DESCRIPTIONS #
  ################################

  # Season             - Year of the season
  # Week               - Game week of the season (e.g. 'Week 1')
  # Day                - Day of the week (e.g. 'MON')
  # Date               - Month and day of the game formatted MM/DD (e.g. '09/07')
  # AwayTeam           - Visiting team of the game
  # HomeTeam           - Home team of the game
  # Quarter            - Quarter the drive is in.
  #                      - Complete drives are together in a single quarter.
  #                        - Drives that go between quarters will have all plays
  #                          within the latter drive.
  # DriveNumber        - Drive number of the quarter that the play is in
  # TeamWithPossession - Team that started with the ball at the beginning of the play.
  # IsScoringDrive     - Does the drive that the focused play in result in a score?
  # PlayNumberInDrive  - Play count in the drive
  # IsScoringPlay      - Did the play result in a score?
  # PlayOutcome        - Ultimate result of the play (e.g. '13 Yard Pass')
  # PlayStart          - The down and where the play started on the field (e.g. '2nd & 9 at DET 21')
  # PlayTimeFormation  - Time left in the quarter / quarter / play formation
  # PlayDescription    - The raw description given of the focused play, entailing everything
  #                      that happened within it.

  #############################################################
  # TRANSFORMING FEATURE VALUES (PREPPING DATA TO BE CLEANED) #
  #############################################################
  df_all_plays_cleaned = playtimeformation_split(df_all_plays_cleaned)
  df_all_plays_cleaned = playstart_split(df_all_plays_cleaned)
  df_all_plays_cleaned = consistent_team_names(df_all_plays_cleaned)

  ######################################
  # NEW ADDITIONAL COLUMN DESCRIPTIONS #
  ######################################

  # ~ General features ~
  # TimeOnTheClock     - NOT HERE ANYMORE.

  # ~ Offensive features ~
  # EndSpot            - Where the end of the play has been spotted
  #                      - This can also be where the end of the action within a play has been spotted.
  # PlayType           - The type of play (e.g. pass/run)
  # Formation          - Play formation
  # PlayInQuarter      - Quarter the play took place in
  # Passer             - Player that threw the ball (mostly the quarterback)
  # Rusher             - Player that ran the ball (mostly the runningback)
  # Receiver           - Player on the same team as the passer that caught the ball
  # Direction          - Where the ball is going during the play
  # Yardage            - Yards gained during the play
  #                      - (Should specify that yardage does not include extra yardage gained from penalties)
  #                      - (Player awarded yardage)
  #                      - (also includes how far kicks have gone during kickoffs and punts)

  # ~ Defensive features ~
  # SoloTackle         - Player awarded a solo tackle from a play
  # AssistedTackle     - Player awarded an assisted tackle from a play
  # SharedTackle       - Player awarded a shared tackle from a play
  # PassDefendedBy     - Defender that defended the passing play
  # PressureBy         - Defender that applied pressure to the passer
  # InterceptedBy      - Defender that intercepted the passing play
  # SackedBy           - Player awarded a sack from a play. (Could be solo or split)
  # ForcedFumbledBy    - Player awarded a forced fumble from a play

  # ~ Unique features (uncommon) ~
  # WhoFumbled         - Player who last held the ball during a fumble.
  # FumbleRecoveredBy  - Player who recovered the fumbled ball
  # FumbleDetails      - A list that has what happened after the fumble
  #                      - [forced fumble by, recovered by, yards gained, tackled by]
  # ReverseDetails     - A list having plays leading up to play reversal
  # InjuredPlayers     - Players that were injured during the play
  # AcceptedPenalty    - Penalty on the field that was accepted
  # DeclinedPenalty    - Penalty on the field that was declined

  # ~ Special teams features ~
  # Kicker             - Player who kicked the ball during a kickoff / punt / extra point / field goal
  # LongSnapper        - Player who snapped the ball during a punt / extra point / field goal
  # Returner           - Player who returned the ball during a kickoff / punt
  # DownedBy           - ? ? ? I forget
  # Holder             - Player who held ball for extra point / field goal
  # BlockedBy          - Player who blocked a punt / extra point / field goal

  new_columns = ["EndSpot",
                 "PlayType", "Passer", "Rusher", "Receiver", "Direction", "Yardage",
                 "SoloTackle", "AssistedTackle", "SharedTackle", 'PassDefendedBy', "PressureBy", "InterceptedBy", "SackedBy", "ForcedFumbleBy",
                 "WhoFumbled", "FumbleRecoveredBy", "FumbleDetails", "ReverseDetails", "InjuredPlayers", "AcceptedPenalty", "DeclinedPenalty",
                 "Kicker", "LongSnapper", "Returner", "DownedBy", "Holder", "BlockedBy"]

  string_columns = ["EndSpot",
                    "PlayType", "Passer", "Rusher", "Receiver", "Direction",
                    "SoloTackle", "AssistedTackle", "SharedTackle", 'PassDefendedBy', "PressureBy", "InterceptedBy", "SackedBy", "ForcedFumbleBy",
                    "WhoFumbled", "FumbleRecoveredBy", "FumbleDetails", "ReverseDetails", "InjuredPlayers", "AcceptedPenalty", "DeclinedPenalty",
                    "Kicker", "LongSnapper", "Returner", "DownedBy", "Holder", "BlockedBy"]

  int_columns = ["Yardage"]

  ########################################
  # RETURN DATAFRAME WITH ADDED FEATURES #
  ########################################

  df_all_plays_cleaned = df_all_plays_cleaned.reindex(columns=df_all_plays_cleaned.columns.tolist() + new_columns)
  df_all_plays_cleaned[string_columns] = df_all_plays_cleaned[string_columns].astype(str)
  df_all_plays_cleaned[int_columns] = df_all_plays_cleaned[int_columns].astype(float)

  ########################################
  # GETTING PLAY CATEGORIES AND CLEANING #
  ########################################
  # TOUCHDOWNS MUST BE CLEANED FIRST
  # - Any touchdown resulting from a change in possession (e.g. Interception for Touchdown)
  #   raw data states that the team on defense had possession the entire drive.
  #   - So all plays leading up to the touchdown state that the defense has possession.
  df_all_plays_cleaned = clean_touchdown_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_pass_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_run_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_2pt_conversion_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_intercepted_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_sacked_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_punt_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_kickoff_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_field_goal_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_extra_point_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_turnover_on_downs_plays(df_all_plays_cleaned)
  df_all_plays_cleaned = clean_fumble_plays(df_all_plays_cleaned)



  return df_all_plays_cleaned

# TESTING

In [ ]:
df_week2_plays_cleaned = clean_dataframe_of_plays(week2_2023_plays)

In [ ]:
df_week2_plays_cleaned.shape

In [ ]:
df_week2_plays_cleaned

# PLAYTYPE OBSERVATIONS

In [ ]:
# Modifying plays to match cleaned plays transformed features
# ( e.g. Quarter(original) = '1st Quarter
#        Quarter(transform) = 1 )
# - This is needed in order to match plays from the original dataframe
#   to the cleaned dataframe.
df_week2_plays_modified = week2_2023_plays.copy()
df_week2_plays_modified = playtimeformation_split(df_week2_plays_modified)
df_week2_plays_modified = playstart_split(df_week2_plays_modified)
df_week2_plays_modified = consistent_team_names(df_week2_plays_modified)

## HELPER METHOD

In [ ]:
# PURPOSE:
# - A tool that can be used to compare original plays and their cleaned versions

# I would like to return a map that has:
# KEY: index of original unclean play
# VALUE: index(es) of cleaned play

def unclean_to_clean_play_matches(df_unclean_plays, df_clean_plays):

  my_map = {}

  # This list of features is unique to each play
  # - Both the unclean and cleaned versions of the plays have these same features, therefore
  #   they will be used to match unclean plays in 'df_unclean_plays' to clean plays in 'df_clean_plays'
  matching_features = ['Season', 'Week', 'Date', 'AwayTeam', 'HomeTeam', 'Quarter', 'DriveNumber', 'PlayNumberInDrive']

  # Iterate through each row of the unclean plays dataframe
  for u_row in df_unclean_plays.itertuples(index=True):
    u_features = [getattr(u_row, col) for col in matching_features]

    matching_indexes = []
    matches_found = False

    # Iterate through each row of the dataframe of cleaned plays
    # - The starting index will be the index of the unclean play within the main original dataframe of plays
    #   - The matching cleaned pair will either be at the exact same location or higher
    for c_row in df_clean_plays[u_row.Index::].itertuples(index=True):
      c_features = [getattr(c_row, col) for col in matching_features]

      # If a match is found, check for consective rows of matches because some uncleaned plays needed to be cleaned using multiple rows
      # - Once a row that does not match follows one that does, will break the loop because the one play match has been found.
      if u_features == c_features:
        matching_indexes.append(c_row.Index)
        matches_found = True
      elif matches_found:
        my_map[u_row.Index] = matching_indexes
        break

  return my_map

## OFFENSIVE PLAYS

### PASSING PLAYS

In [ ]:
# All passing plays
df_unclean_pass_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Pass'))]

map_unclean_clean_pass_plays = unclean_to_clean_play_matches(df_unclean_pass_plays, df_week2_plays_cleaned)

len(map_unclean_clean_pass_plays.keys())

# # All passing plays to 'A.St. Brown'
# # - I need to figure out how to separate each sentence of the play description. Currently I am splitting them
# #   by finding this set of characters ". ", This will not work all the time and might actually cause error because
# #   some players have names that have ". " in them and this will cause the splitting to be at their name.
# df_unclean_pass_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Pass')) &
#                                                     (df_week2_plays_modified['PlayDescription'].str.contains('A.St. Brown', case=False))]

# map_unclean_clean_pass_plays = unclean_to_clean_play_matches(df_unclean_pass_plays, df_week2_plays_cleaned)

# len(map_unclean_clean_pass_plays.keys())

In [ ]:
# Every unclean passing play and their associated cleaned play breakdown

for i in map_unclean_clean_pass_plays.keys():
  print(f"({i}, {map_unclean_clean_pass_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  # play_split = play.split(". ")
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

### RUN PLAYS

In [ ]:
# All rushing plays
df_unclean_run_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Run'))]

map_unclean_clean_run_plays = unclean_to_clean_play_matches(df_unclean_run_plays, df_week2_plays_cleaned)

len(map_unclean_clean_run_plays.keys())

In [ ]:
# Every unclean run play and their associated cleaned play breakdown

for i in map_unclean_clean_run_plays.keys():
  print(f"({i}, {map_unclean_clean_run_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = play.split(". ")
  for j in play_split:
    print(j)
  print()

### 2PT CONVERSION

In [ ]:
df_2023_2pt_conversion_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Conversion')]

# All 2PT conversion attempts
df_unclean_conversion_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Conversion'))]

map_unclean_clean_conversion_plays = unclean_to_clean_play_matches(df_unclean_conversion_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean conversion play and their associated cleaned play breakdown

for i in map_unclean_clean_conversion_plays.keys():
  print(f"({i}, {map_unclean_clean_conversion_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = play.split(". ")
  for j in play_split:
    print(j)
  print()

## DEFENSIVE PLAYS

### INTERCEPTION

In [ ]:
df_2023_interception_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Interception')]

# All interception attempts
df_unclean_interception_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Interception'))]

map_unclean_clean_interception_plays = unclean_to_clean_play_matches(df_unclean_interception_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean interception play and their associated cleaned play breakdown

for i in map_unclean_clean_interception_plays.keys():
  print(f"({i}, {map_unclean_clean_interception_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = play.split(". ")
  for j in play_split:
    print(j)
  print()

### SACK

In [ ]:
df_2023_sack_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Sack')]

# All sacks
df_unclean_sack_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Sack'))]

map_unclean_clean_sack_plays = unclean_to_clean_play_matches(df_unclean_sack_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean sacked play and their associated cleaned play breakdown

for i in map_unclean_clean_sack_plays.keys():
  print(f"({i}, {map_unclean_clean_sack_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = play.split(". ")
  for j in play_split:
    print(j)
  print()

## SPECIAL TEAMS PLAYS

### PUNTS

In [ ]:
df_2023_punt_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Punt')]

# All punts
df_unclean_punt_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Punt')) &
                                                    (df_week2_plays_modified['PlayDescription'].str.contains('FUMBLE'))]

map_unclean_clean_punt_plays = unclean_to_clean_play_matches(df_unclean_punt_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean punt play and their associated cleaned play breakdown

for i in map_unclean_clean_punt_plays.keys():
  print(f"({i}, {map_unclean_clean_punt_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

### KICKOFFS

In [ ]:
df_2023_kickoff_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Kickoff')]

# All punts
df_unclean_kickoff_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Kickoff')) &
                                                       (df_week2_plays_modified['PlayDescription'].str.contains('FUMBLES'))]

map_unclean_clean_kickoff_plays = unclean_to_clean_play_matches(df_unclean_kickoff_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean kickoff play and their associated cleaned play breakdown

for i in map_unclean_clean_kickoff_plays.keys():
  print(f"({i}, {map_unclean_clean_kickoff_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

## SCORING PLAYS

### TOUCHDOWNS

In [ ]:
df_2023_touchdown_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Touchdown')]

# All touchdowns
df_unclean_touchdown_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Touchdown'))]

map_unclean_clean_touchdown_plays = unclean_to_clean_play_matches(df_unclean_touchdown_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean touchdown play and their associated cleaned play breakdown

for i in map_unclean_clean_touchdown_plays.keys():
  print(f"({i}, {map_unclean_clean_touchdown_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

### FIELD GOALS

In [ ]:
df_2023_field_goal_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Field Goal')]

# All field goals
df_unclean_field_goal_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Field Goal'))]

# # All blocked field goals
# df_unclean_field_goal_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Field Goal')) &
#                                                           (df_week2_plays_modified['PlayDescription'].str.contains('BLOCKED'))]

map_unclean_clean_field_goal_plays = unclean_to_clean_play_matches(df_unclean_field_goal_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean field goal play and their associated cleaned play breakdown

for i in map_unclean_clean_field_goal_plays.keys():
  print(f"({i}, {map_unclean_clean_field_goal_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

### EXTRA POINTS

In [ ]:
df_2023_extra_point_week2 = week2_2023_plays[week2_2023_plays['PlayOutcome'].str.contains('Extra Point')]

# All field goals
df_unclean_extra_point_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Extra Point'))]

map_unclean_clean_extra_point_plays = unclean_to_clean_play_matches(df_unclean_extra_point_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean field goal play and their associated cleaned play breakdown

for i in map_unclean_clean_extra_point_plays.keys():
  print(f"({i}, {map_unclean_clean_extra_point_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

## OTHER

### LATERALS

In [ ]:
# All lateral plays
df_unclean_lateral_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayDescription'].str.contains('lateral', case=False))]

map_unclean_clean_lateral_plays = unclean_to_clean_play_matches(df_unclean_lateral_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean lateral and their associated cleaned play breakdown

for i in map_unclean_clean_lateral_plays.keys():
  print(f"({i}, {map_unclean_clean_lateral_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

### FUMBLES

#### FUMBLE OUTCOMES

In [ ]:
# All fumble plays
df_unclean_fumble_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('fumble', case=False))]

map_unclean_clean_fumble_plays = unclean_to_clean_play_matches(df_unclean_fumble_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean fumble and their associated cleaned play breakdown

for i in map_unclean_clean_fumble_plays.keys():
  print(f"({i}, {map_unclean_clean_fumble_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

#### ABORTED FUMBLES

In [ ]:
# All aborted fumble plays
df_unclean_aborted_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayDescription'].str.contains('aborted', case=False))]

map_unclean_clean_aborted_plays = unclean_to_clean_play_matches(df_unclean_aborted_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean fumble and their associated cleaned play breakdown

for i in map_unclean_clean_aborted_plays.keys():
  print(f"({i}, {map_unclean_clean_aborted_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

### TURNOVER ON DOWNS

In [ ]:
# All turnover on downs plays
df_unclean_turnover_on_downs_plays = df_week2_plays_modified.loc[(df_week2_plays_modified['PlayOutcome'].str.contains('Turnover on Downs'))]

map_unclean_clean_turnover_on_downs_plays = unclean_to_clean_play_matches(df_unclean_turnover_on_downs_plays, df_week2_plays_cleaned)

In [ ]:
# Every unclean field goal play and their associated cleaned play breakdown

for i in map_unclean_clean_turnover_on_downs_plays.keys():
  print(f"({i}, {map_unclean_clean_turnover_on_downs_plays.get(i)})")
  play = df_week2_plays_modified['PlayDescription'].iloc[i]
  play_split = split_play_description(play)
  for j in play_split:
    print(j)
  print()

## INDEX SEARCHING

In [ ]:
# df_week2_plays_cleaned.iloc[962]

# Intercepted touchdown (team with possesion is wrong)
df_week2_plays_cleaned.iloc[1089]

# df_week2_plays_cleaned.iloc[2292]

# TEAM STAT TABLES

## HELPER METHODS

In [ ]:
# Get rid of duplicate rows (for play's that have multiple rows)

# Why wouldn't I want duplicate rows again..?
# - For plays that consume multiple rows that result in a touchdown.
#   - If we did not remove duplicates, mutiple touchdowns would be counted from
#     the "PlayOutcome" feature.

def no_duplicates(df_with_duplicates, index_start=None):

  # exit case
  # - The last element has been grabbed
  if df_with_duplicates.tail(1).index[0] == index_start:
    return df_with_duplicates

  if index_start == None:
    index_start = df_with_duplicates.index[0]

  first_element = df_with_duplicates.loc[index_start]

  second_element_index = df_with_duplicates.index[df_with_duplicates.index.tolist().index(index_start)+1]
  second_element = df_with_duplicates.loc[second_element_index]

  # Features that will decipher whether the two rows are apart of the same play
  matching_features = ['Season', 'Week', 'Date', 'AwayTeam', 'HomeTeam', 'Quarter', 'DriveNumber', 'PlayNumberInDrive']

  # - Check to see if 1st and 2nd elements are match
  if first_element[matching_features].equals(second_element[matching_features]):

    # 1. remove 1st element
    df_with_duplicates = df_with_duplicates.drop(index_start, inplace=False)

    # 2. run method starting search from 1st element
    return no_duplicates(df_with_duplicates, second_element_index)

  else:

    return no_duplicates(df_with_duplicates, second_element_index)

GOAL:
- Reflect the Score Table provided when a game has finished.

COLUMNS:
- Index - Team Abreviations (Away Team / Home Team)
- Quarters - Each quarter played within game
- Total - Total points scored by each team

In [ ]:
# Display the scoring table for a specified game

def score_table(away_team, home_team, df_cleaned_plays):
  df_all_plays_in_game = df_cleaned_plays.loc[(df_cleaned_plays['HomeTeam'] == home_team) &
                                              (df_cleaned_plays['AwayTeam'] == away_team)]

  df_all_plays_in_game = no_duplicates(df_all_plays_in_game)

  teams = [[away_team], [home_team]]

  for i in range(len(teams)):
    total_score = 0
    for quarter in df_all_plays_in_game['Quarter'].unique():
      quarter_score = 0

      # touchdowns (Needs work)
      df_touchdowns_in_quarter = df_all_plays_in_game.loc[(df_all_plays_in_game['PlayOutcome'].str.contains('touchdown', case=False)) &
                                                          (df_all_plays_in_game['TeamWithPossession'].str.contains(teams[i][0])) &
                                                          (df_all_plays_in_game['Quarter'] == quarter)]
      quarter_score += df_touchdowns_in_quarter.shape[0] * 6

      # PAT
      df_extra_points_in_quarter = df_all_plays_in_game.loc[(df_all_plays_in_game['PlayOutcome'].str.contains('Extra Point', case=False)) &
                                                            ~(df_all_plays_in_game['PlayOutcome'].str.contains('No Good', case=False)) &
                                                              (df_all_plays_in_game['Quarter'] == quarter) &
                                                              (df_all_plays_in_game['TeamWithPossession'] == teams[i][0])]
      quarter_score += df_extra_points_in_quarter.shape[0] * 1

      # field goals
      df_field_goals_in_quarter = df_all_plays_in_game.loc[(df_all_plays_in_game['PlayOutcome'].str.contains('Field Goal', case = False)) &
                                                           ~(df_all_plays_in_game['PlayOutcome'].str.contains('No Good', case=False)) &
                                                            (df_all_plays_in_game['Quarter'] == quarter) &
                                                            (df_all_plays_in_game['TeamWithPossession'] == teams[i][0])]
      quarter_score += df_field_goals_in_quarter.shape[0] * 3

      # 2 Pt Conversions
      df_two_pt = df_all_plays_in_game.loc[(df_all_plays_in_game['PlayOutcome'].str.contains('2 Point Conversion', case=False)) &
                                           (df_all_plays_in_game['Quarter'] == quarter) &
                                           (df_all_plays_in_game['TeamWithPossession'] == teams[i][0])]
      quarter_score += df_two_pt.shape[0] * 2

      teams[i].append(quarter_score)
      total_score += quarter_score

    teams[i].append(total_score)
    teams[i].pop(0)

  scoring_columns = df_all_plays_in_game['Quarter'].unique().tolist()

  scoring_columns.append("Total")

  return pd.DataFrame(teams, columns = scoring_columns, index=[away_team, home_team])

In [ ]:
# Display passing stats for a specified game

def passing_table(away_team, home_team, df_cleaned_plays, home_or_away):

  # User decides which passing table they want (home or away)
  if home_or_away == 0:
    team = home_team
  elif home_or_away == 1:
    team = away_team
  else:
    return "home_or_away parameter should be a 0 or 1. (home = 0, away = 1)"

  # All plays within game
  df_all_plays_in_game = df_cleaned_plays.loc[(df_cleaned_plays['HomeTeam'] == home_team) &
                                              (df_cleaned_plays['AwayTeam'] == away_team)]

  # list of passers in game
  list_passers = df_all_plays_in_game['Passer'].unique().tolist()
  if 'nan' in list_passers:
    list_passers.pop(list_passers.index('nan'))

  # key: passer
  # value: team
  dict_passers_to_team = {}
  for passer in list_passers:
    dict_passers_to_team[passer] = df_all_plays_in_game['TeamWithPossession'].loc[df_all_plays_in_game['Passer'] == passer].value_counts().index[0]

  # Filtering 'dict_passers_to_team'
  # key: passer(s) from desired team
  # value: team
  dict_passers_to_team = {k:v for k,v in dict_passers_to_team.items() if v == team}

  # Setup Dataframe for Passer summary
  df_passer_data = pd.DataFrame(columns=["CP/ATT", "YDS", "TD", "INT"], index = list(dict_passers_to_team.keys()))

  # Grabbing data from each passer in game
  for passer in df_passer_data.index:

    passing_attempts = df_all_plays_in_game.loc[(df_all_plays_in_game['Passer'] == passer) &
                                                (df_all_plays_in_game['PlayType'] == "Pass")]

    passing_completions = passing_attempts.loc[(passing_attempts['PlayOutcome'].str.contains('yard pass', case=False)) |
                                               (passing_attempts['PlayOutcome'].str.contains('touchdown', case=False) & passing_attempts['InterceptedBy'].str.contains('nan')) |
                                               (passing_attempts['PlayOutcome'].str.contains(rf"Turnover on Downs \(C\)", case=False)) |
                                               (passing_attempts['PlayOutcome'].str.contains("Pass for No Gain", case=False)) |
                                               (passing_attempts['PlayOutcome'].str.contains(rf"Fumble \(C\)", case=False))]
# defense_pressure_name_pattern = rf"\[({name_pattern})\]"
    df_passer_data.loc[passer, 'CP/ATT'] = f"{passing_completions.shape[0]}/{passing_attempts.shape[0]}"

    df_passer_data.loc[passer, 'YDS'] = int(passing_completions['Yardage'].sum())

    total_touchdowns = passing_completions.loc[(passing_completions['PlayOutcome'].str.contains('touchdown', case=False) & passing_attempts['InterceptedBy'].str.contains('nan'))]

    df_passer_data.loc[passer, 'TD'] = total_touchdowns.shape[0]

    total_interceptions = passing_attempts.loc[passing_attempts['PlayDescription'].str.contains('intercepted', case=False)]

    df_passer_data.loc[passer, 'INT'] = total_interceptions.shape[0]

  return df_passer_data

In [ ]:
# Display receiving stats for a specified game

def receiving_table(away_team, home_team, df_cleaned_plays, home_or_away):

  # All plays within game
  df_all_plays_in_game = df_cleaned_plays.loc[(df_cleaned_plays['HomeTeam'] == home_team) &
                                              (df_cleaned_plays['AwayTeam'] == away_team)]

  # User decides which receiving table they want (home or away)
  if home_or_away == 0:
    team = home_team
  elif home_or_away == 1:
    team = away_team
  else:
    return "home_or_away parameter should be a 0 or 1. (home = 0, away = 1)"

  print(f"Receiving Table: {team}")
  print()

  df_receiving_plays = df_all_plays_in_game.loc[(df_all_plays_in_game['PlayType'].str.contains('pass', case=False)) &
                                                (df_all_plays_in_game['TeamWithPossession'] == team) &
                                                (~df_all_plays_in_game['PlayOutcome'].str.contains('2PT Conversion', case=False))]

  receivers = df_receiving_plays['Receiver'].unique().tolist()
  if 'nan' in receivers:
    receivers.pop(receivers.index('nan'))

  df_receiver_data = pd.DataFrame(columns=["REC", "YDS", "TD", "TGTS"], index = receivers)

  for receiver in df_receiver_data.index:
    receiver_plays = df_receiving_plays.loc[df_receiving_plays['Receiver'] == receiver]

    df_receiver_data.loc[receiver, 'REC'] = receiver_plays.loc[(
        (receiver_plays['PlayOutcome'].str.contains('yard pass', case=False)) |
        (receiver_plays['PlayOutcome'].str.contains(f'touchdown', case=False) & receiver_plays['InterceptedBy'].str.contains('nan')) |
        (receiver_plays['PlayOutcome'].str.contains(rf"Turnover On Downs \(C\)", case=False)) |
        (receiver_plays['PlayOutcome'].str.contains("Pass for No Gain", case=False)) |
        (receiver_plays['PlayOutcome'].str.contains(rf"Fumble \(C\)", case=False))) &
                                                               ~ (receiver_plays['PlayType'].str.contains("lateral after pass", case=False))
                                                               ].shape[0]

    df_receiver_data.loc[receiver, 'YDS'] = int(receiver_plays['Yardage'].sum())
    df_receiver_data.loc[receiver, 'TD'] = receiver_plays.loc[(receiver_plays['PlayOutcome'].str.contains(f'touchdown', case=False)) &
                                                              (receiver_plays['InterceptedBy'].str.contains('nan'))].shape[0]
    df_receiver_data.loc[receiver, 'TGTS'] = receiver_plays.shape[0] - receiver_plays.loc[receiver_plays['PlayType'] == 'lateral after pass'].shape[0]

  return df_receiver_data.sort_values(by="YDS", ascending=False)

## WEEK SCHEDULE (WEEK 2, 2023)


In [ ]:
# Season 2023 Week 1 schedule

df_2023_week2_schedule = df_week2_plays_cleaned[['HomeTeam', 'AwayTeam', 'Season', 'Date', 'GameSlot']].drop_duplicates().sort_values(by='Date').reset_index(drop=True)

df_2023_week2_schedule

## SCORING TABLE

In [ ]:
game_num = 15

away_team = df_2023_week2_schedule['AwayTeam'].iloc[game_num]
home_team = df_2023_week2_schedule['HomeTeam'].iloc[game_num]

score_table(away_team, home_team, df_week2_plays_cleaned)

PASSING TABLE

In [ ]:
away_team = df_2023_week2_schedule['AwayTeam'].iloc[game_num]
home_team = df_2023_week2_schedule['HomeTeam'].iloc[game_num]

passing_table(away_team, home_team, df_week2_plays_cleaned, 0)

In [ ]:
receiving_table(away_team, home_team, df_week2_plays_cleaned, 1)

## MICROSCOPE
- Area to check a specific set of plays

In [ ]:
# Area for observing a specific set of plays

df_game_plays = df_week2_plays_cleaned.loc[df_week2_plays_cleaned['HomeTeam'] == 'PIT']

df_specific_plays = df_game_plays[(df_game_plays['PlayDescription'].str.contains('pass', case=False)) &
                                  (df_game_plays['Passer'].str.contains('D.Watson', case=False))]

df_specific_plays

In [ ]:
for idx, play in df_specific_plays['PlayDescription'].items():
  print(idx)
  print(play)
  print()

In [ ]:
df_specific_plays = df_game_plays[(df_game_plays['PlayDescription'].str.contains('E.Moore', case=False))]

df_specific_plays

In [ ]:
for idx, play in df_specific_plays['PlayDescription'].items():
  print(idx)
  print(play)
  print()